# Stateless agent – injection experiments

This notebook contains controlled injection experiments used to evaluate the stateless decision-theoretic SCADA agent.

The experiments include:
- evidence manipulation (injection)
- baseline vs injected comparisons
- analysis of posterior distributions and decisions

This notebook is not part of the core agent implementation.

In [1]:
# %% 0.0 [SETUP] IMPORTS & GLOBAL CONFIGURATION (REV16-style, stram og NA-safe)

from __future__ import annotations

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from dataclasses import dataclass
from typing import Dict, List, Literal, Tuple, Optional
from pathlib import Path

# -----------------------------
# Global random seed
# -----------------------------
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

# -----------------------------
# Project root detection (robust)
# -----------------------------
ENV_PROJECT_DIR = os.getenv("ADA511_PROJECT_DIR", "").strip()
PROJECT_ROOT = Path(ENV_PROJECT_DIR).resolve() if ENV_PROJECT_DIR else Path(".").resolve()

DATA_DIR    = PROJECT_ROOT / "data"
MN_DIR      = PROJECT_ROOT / "MN"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

for d in [DATA_DIR, MN_DIR, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Project-wide configuration
# -----------------------------
YEARS = [2018, 2019, 2020, 2021]
FUNKSJONSTABELL_PATH = DATA_DIR / "Funksjonstabell.xlsx"
assert FUNKSJONSTABELL_PATH.exists(), f"Mangler: {FUNKSJONSTABELL_PATH}"

# -----------------------------
# Load funksjonstabell (FAST kolonnenamn)
# -----------------------------
# NB: dtype="string" => bevarer NA som <NA> (ikkje 'nan'-tekst)
ft = pd.read_excel(FUNKSJONSTABELL_PATH, dtype="string")
ft.columns = [str(c).strip() for c in ft.columns]

REQUIRED = [
    "ScopeGroup",
    "fault_loc_key",
    "Component ID",
    "Data Type",
    "Measured / Status Signals",
]
missing = [c for c in REQUIRED if c not in ft.columns]
assert not missing, f"Funksjonstabell manglar kolonner: {missing}"

# Kanoniske felt (utan å endre hensikt)
ft["component_id"] = ft["Component ID"].astype("string").str.strip()
ft["fault_loc_key"] = ft["fault_loc_key"].astype("string").str.strip()
ft["data_type"] = ft["Data Type"].astype("string").str.strip()
ft["measured_status_signals"] = ft["Measured / Status Signals"].astype("string").str.strip()

# Kritisk hygiene: fjern tom/NA + stopp 'nan'-streng
ft["fault_loc_key"] = ft["fault_loc_key"].replace({"nan": pd.NA, "NaN": pd.NA, "": pd.NA})
ft["component_id"]  = ft["component_id"].replace({"nan": pd.NA, "NaN": pd.NA, "": pd.NA})

# Utility-kolonner
utility_cols = [c for c in ft.columns if str(c).endswith("_Utility")]
assert utility_cols, "Fant ingen *_Utility-kolonner i Funksjonstabell."

# Sanity (dette er debug-ankeret for rev16→rev18-feilen)
print("Project root      :", PROJECT_ROOT)
print("Funksjonstabell   :", FUNKSJONSTABELL_PATH)
print("ft rows/cols      :", ft.shape)
print("fault_loc_key NA  :", int(ft["fault_loc_key"].isna().sum()))
print("fault_loc_key=='nan':", int((ft["fault_loc_key"].astype("string") == "nan").sum()))
print("utility cols      :", len(utility_cols))


Project root      : C:\Python\ADA511_prosjekt
Funksjonstabell   : C:\Python\ADA511_prosjekt\data\Funksjonstabell.xlsx
ft rows/cols      : (86, 30)
fault_loc_key NA  : 6
fault_loc_key=='nan': 0
utility cols      : 5


In [2]:
# 0.1 [KODESJEKK] VALIDATION: files + folder structure

assert DATA_DIR.exists(), f"Mangler DATA_DIR: {DATA_DIR}"
assert MN_DIR.exists(), f"Mangler MN_DIR: {MN_DIR}"

assert FUNKSJONSTABELL_PATH.exists(), (
    f"Fant ikke Funksjonstabell.xlsx på forventet plass:\n{FUNKSJONSTABELL_PATH}\n"
    f"Tips: Sjekk at du kjører fra prosjektroten, eller sett ADA511_PROJECT_DIR."
)

# Sjekk at måledata-mappestrukturen finnes for alle år
missing_year_folders = []
for y in YEARS:
    y_dir = MN_DIR / f"{y}_measurements"
    if not y_dir.exists():
        missing_year_folders.append(str(y_dir))

if missing_year_folders:
    print("NB: Mangler én eller flere year-folders for målinger:")
    for p in missing_year_folders:
        print(" -", p)
else:
    print("OK: Fant måle-årsmapper for alle YEARS.")

print("OK: Validation ferdig.")



OK: Fant måle-årsmapper for alle YEARS.
OK: Validation ferdig.


In [3]:
# 2.5 [SETUP]


funksjonstabell_df = pd.read_excel(FUNKSJONSTABELL_PATH)




In [4]:
# 2.6 [SETUP] Build HS ladder-correct topology from Funksjonstabell

#   - main: Upstream/Downstream ONLY for (line/breaker) that are not earthing switches
#   - branch: BranchTo ONLY for (line/breaker) that are not earthing switches
#   - attach: ParentComponent preferred; fallback MeasuresAt; for earthing, allow BranchTo to host
#   - serves: COM LinkedTo (metadata only, not used for reachability)

import pandas as pd
import networkx as nx
import re

assert "funksjonstabell_df" in globals(), "Mangler funksjonstabell_df (kjør 2.5)"

df = funksjonstabell_df.copy()
df.columns = df.columns.astype(str).str.strip()

# ---------------- helpers ----------------
def _clean(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    return s if s else None

def _split_csv(x):
    x = _clean(x)
    if not x:
        return []
    return [s.strip() for s in x.split(",") if s.strip()]

def _is_earthing_switch(row) -> bool:
    # Robust: look in Component Name and Type for earthing/jordkniv
    nm = (_clean(row.get("Component Name")) or "").lower()
    tp = (_clean(row.get("Type")) or "").lower()
    return ("earthing" in nm) or ("jordkniv" in nm) or ("earthing" in tp) or ("jordkniv" in tp)

def _infer_kind(node_id: str, row=None) -> str:
    """
    kind used by later notebook sections:
      - line, breaker, sensor, indicator, comm, earthing, unknown
    """
    if not node_id:
        return "unknown"

    # If we have the row, earthing overrides '-B' heuristic
    if row is not None and _is_earthing_switch(row):
        return "earthing"

    # ID patterns
    if node_id.startswith("L") and node_id[1:].isdigit():
        return "line"
    if node_id.endswith("-COM"):
        return "comm"
    if "-S" in node_id:
        return "sensor"
    if "-M" in node_id:
        return "indicator"
    if "-B" in node_id:
        return "breaker"

    # Avoid treating 'A', 'B', 'IDK2' as structural nodes by default
    if node_id.isalpha() or node_id.startswith("IDK"):
        return "scope"

    return "unknown"


# ---------------- graph ----------------
G = nx.DiGraph()

def add_node(n: str, kind: str = None, **attrs):
    if not n:
        return
    if not G.has_node(n):
        G.add_node(n)
    if kind:
        G.nodes[n]["kind"] = kind
    for k, v in attrs.items():
        if v is not None:
            G.nodes[n][k] = v

def add_edge(u: str, v: str, relation: str):
    if not u or not v:
        return
    if u == v:
        return
    if not G.has_node(u):
        add_node(u, kind=_infer_kind(u))
    if not G.has_node(v):
        add_node(v, kind=_infer_kind(v))
    # allow multiple relations between same u->v? we keep first to avoid confusion
    if not G.has_edge(u, v):
        G.add_edge(u, v, relation=relation)


# ---------------- 0) add nodes + attrs ----------------
# We infer kind per row, so earthing gets correct kind even if it has '-B'
for _, r in df.iterrows():
    cid = _clean(r.get("Component ID"))
    if not cid:
        continue

    kind = _infer_kind(cid, row=r)

    add_node(
        cid,
        kind=kind,
        scopegroup=_clean(r.get("ScopeGroup")),
        fault_loc_key=_clean(r.get("fault_loc_key")),
        measuresat=_clean(r.get("MeasuresAt")),
        parent=_clean(r.get("ParentComponent")),
        type=_clean(r.get("Type")),
        component_name=_clean(r.get("Component Name")),
    )


# ---------------- 1) MAIN ladder via Upstream + Downstream ----------------
# Rule: ONLY line/breaker (NOT earthing) participate in main edges
for _, r in df.iterrows():
    cid = _clean(r.get("Component ID"))
    if not cid:
        continue

    kind = _infer_kind(cid, row=r)
    if kind not in ("line", "breaker"):
        continue
    if _is_earthing_switch(r):
        continue

    up = _clean(r.get("Upstream Node"))
    dn = _clean(r.get("Downstream Node"))

    if up:
        up_kind = _infer_kind(up)
        if up_kind in ("line", "breaker"):
            add_edge(up, cid, "main")
    if dn:
        dn_kind = _infer_kind(dn)
        if dn_kind in ("line", "breaker"):
            add_edge(cid, dn, "main")


# ---------------- 2) BRANCHES via BranchTo ----------------
# Rule: ONLY line/breaker (NOT earthing) can branch structurally
for _, r in df.iterrows():
    cid = _clean(r.get("Component ID"))
    if not cid:
        continue

    kind = _infer_kind(cid, row=r)
    if kind not in ("line", "breaker"):
        continue
    if _is_earthing_switch(r):
        continue

    for br in _split_csv(r.get("BranchTo")):
        br_kind = _infer_kind(br)
        if br_kind in ("line", "breaker"):
            add_edge(cid, br, "branch")


# ---------------- 3) ATTACHMENTS ----------------
# Priority:
#   (a) ParentComponent -> attach
#   (b) MeasuresAt      -> attach (ONLY if MeasuresAt resolves to a real node in G)
#   (c) Earthing special: if BranchTo points to host breaker/line, attach host -> earthing
for _, r in df.iterrows():
    cid = _clean(r.get("Component ID"))
    if not cid:
        continue

    kind = _infer_kind(cid, row=r)

    # Attach types we WANT as attachments
    if kind not in ("sensor", "indicator", "comm", "earthing"):
        continue

    parent = _clean(r.get("ParentComponent"))
    meas   = _clean(r.get("MeasuresAt"))

    if parent:
        add_edge(parent, cid, "attach")
        continue

    # fallback: MeasuresAt only if it exists as a node or looks like a real node id
    if meas and (meas in G.nodes or _infer_kind(meas) in ("line", "breaker")):
        add_edge(meas, cid, "attach")
        continue

    # earthing fallback: allow BranchTo as "host pointer"
    if kind == "earthing":
        hosts = _split_csv(r.get("BranchTo"))
        for h in hosts:
            if _infer_kind(h) in ("line", "breaker"):
                add_edge(h, cid, "attach")


# ---------------- 4) COM coverage (metadata only) ----------------
# Keep as edges, but DO NOT use for reachability/topology structure
for _, r in df.iterrows():
    cid = _clean(r.get("Component ID"))
    if not cid:
        continue
    kind = _infer_kind(cid, row=r)
    if kind != "comm":
        continue

    for tgt in _split_csv(r.get("LinkedTo")):
        # target may be any component; keep metadata edge
        add_edge(cid, tgt, "serves")


# ---------------- finalize ----------------
G_topology = G
topology = G

print("OK: Topology built from Funksjonstabell (truth).")
print("Nodes:", G.number_of_nodes(), "Edges:", G.number_of_edges())

# quick sanity peek
for k in ["A-B1", "B-B2", "C-B2", "C-B4", "F-B1", "F-B2"]:
    if k in G:
        mains = [v for v in G.successors(k) if G[k][v].get("relation") == "main"]
        brs   = [v for v in G.successors(k) if G[k][v].get("relation") == "branch"]
        att   = [v for v in G.successors(k) if G[k][v].get("relation") == "attach"]
        if mains or brs or att:
            print(f"{k}: main={mains} branch={brs} attach={att[:10]}{' ...' if len(att)>10 else ''}")


OK: Topology built from Funksjonstabell (truth).
Nodes: 86 Edges: 149
A-B1: main=['L1'] branch=[] attach=['A-B2', 'A-M1', 'A-M2', 'A-M3', 'A-S1', 'A-S2', 'A-S3', 'A-S4', 'A-S5', 'A-COM']
B-B2: main=['B-B4'] branch=['B-B3'] attach=['B-B1', 'B-COM']
C-B2: main=['C-B3'] branch=[] attach=['C-B1', 'C-COM']
C-B4: main=['L3'] branch=[] attach=['C-B5', 'C-M1', 'C-M2', 'C-M3', 'C-M4', 'C-S1', 'C-S2']
F-B1: main=[] branch=['L61'] attach=[]
F-B2: main=['F-B3'] branch=['F-B1'] attach=['F-COM']


In [5]:
# 2.6a [KODESJEKK] Full dump of ladder from topology (robust reach-sets)

#  - Prints full ladder from root with attachments + branches + main
#  - Computes BOTH:
#      reached_struct = reachable via main+branch only
#      reached_total  = reachable via main+branch+attach
#  - Reports unreached lists separately (no more misleading "attachments are unreached")

import networkx as nx
from collections import deque

assert "G_topology" in globals(), "Mangler G_topology fra 2.6"
G = G_topology

def rel(u, v):
    return G[u][v].get("relation")

def succ_by(u, r):
    return sorted([v for v in G.successors(u) if rel(u, v) == r])

def pick_root(preferred="A-B1"):
    if preferred in G:
        return preferred
    # prefer a breaker-like node if available
    cand = [n for n, d in G.nodes(data=True) if d.get("kind") in ("breaker", "line")]
    if cand:
        # pick one with in_degree==0 if possible
        roots = [n for n in cand if G.in_degree(n) == 0]
        return roots[0] if roots else cand[0]
    # fallback
    roots = [n for n in G.nodes if G.in_degree(n) == 0]
    return roots[0] if roots else list(G.nodes)[0]

def reachable_from(root, allowed_relations):
    """BFS reachability restricted to edges whose relation in allowed_relations."""
    seen = set([root])
    q = deque([root])
    while q:
        u = q.popleft()
        for v in G.successors(u):
            if rel(u, v) in allowed_relations and v not in seen:
                seen.add(v)
                q.append(v)
    return seen

def dump_subtree(u, indent="", seen_print=None):
    """
    Pretty print following:
      - attach: one-level list (no recursion)
      - branch: recursive
      - main: recursive
    Uses seen_print to avoid infinite loops if data has cycles.
    """
    if seen_print is None:
        seen_print = set()
    if u in seen_print:
        print(f"{indent}[ {u} ] (↺ visited)")
        return seen_print
    seen_print.add(u)

    print(f"{indent}[ {u} ]")

    # attachments (print only; they are counted via reached_total separately)
    atts = succ_by(u, "attach")
    for i, v in enumerate(atts):
        print(f"{indent}   ├─ [ {v} ]")

    # branches (recursive)
    brs = succ_by(u, "branch")
    for v in brs:
        print(f"{indent}   ├─▶ [ {v} ]")
        dump_subtree(v, indent + "   │   ", seen_print)

    # main (often 0/1; but print all)
    mains = succ_by(u, "main")
    for v in mains:
        print(f"{indent}   │")
        print(f"{indent}   ▼")
        dump_subtree(v, indent, seen_print)

    return seen_print


# ---------------- run ----------------
root = pick_root("A-B1")

print("\n=== FULL LADDER DUMP (from root) ===\n")
seen_print = dump_subtree(root)

# Reachability sets (THIS is what fixes your misleading "unreached")
reached_struct = reachable_from(root, allowed_relations={"main", "branch"})
reached_total  = reachable_from(root, allowed_relations={"main", "branch", "attach"})

unreached_struct = sorted(set(G.nodes) - reached_struct)
unreached_total  = sorted(set(G.nodes) - reached_total)

print("\n=== REACHABILITY SUMMARY ===")
print("root:", root)
print("nodes total:", G.number_of_nodes())
print("reached_struct (main+branch):", len(reached_struct))
print("reached_total  (main+branch+attach):", len(reached_total))

print("\n=== UNREACHED (STRUCTURAL: not reachable via main+branch) ===")
print("count:", len(unreached_struct))
print(unreached_struct)

print("\n=== UNREACHED (TOTAL: not reachable via main+branch+attach) ===")
print("count:", len(unreached_total))
print(unreached_total)

# Optional: show what is "attach-only" (reachable only when attachments are included)
attach_only = sorted(reached_total - reached_struct)
print("\n=== ATTACH-ONLY NODES (reachable only via attach edges) ===")
print("count:", len(attach_only))
print(attach_only[:80], "..." if len(attach_only) > 80 else "")



=== FULL LADDER DUMP (from root) ===

[ A-B1 ]
   ├─ [ A-B2 ]
   ├─ [ A-COM ]
   ├─ [ A-M1 ]
   ├─ [ A-M2 ]
   ├─ [ A-M3 ]
   ├─ [ A-S1 ]
   ├─ [ A-S2 ]
   ├─ [ A-S3 ]
   ├─ [ A-S4 ]
   ├─ [ A-S5 ]
   │
   ▼
[ L1 ]
   │
   ▼
[ B-B2 ]
   ├─ [ B-B1 ]
   ├─ [ B-COM ]
   ├─▶ [ B-B3 ]
   │   [ B-B3 ]
   │      ├─▶ [ L21 ]
   │      │   [ L21 ]
   │
   ▼
[ B-B4 ]
   ├─ [ B-B5 ]
   ├─ [ B-S1 ]
   │
   ▼
[ L2 ]
   │
   ▼
[ C-B2 ]
   ├─ [ C-B1 ]
   ├─ [ C-COM ]
   │
   ▼
[ C-B3 ]
   │
   ▼
[ C-B4 ]
   ├─ [ C-B5 ]
   ├─ [ C-M1 ]
   ├─ [ C-M2 ]
   ├─ [ C-M3 ]
   ├─ [ C-M4 ]
   ├─ [ C-S1 ]
   ├─ [ C-S2 ]
   │
   ▼
[ L3 ]
   ├─ [ IDK2-COM ]
   ├─ [ IDK2-S1 ]
   ├─ [ IDK2-S2 ]
   ├─ [ IDK2-S3 ]
   ├─ [ IDK2-S4 ]
   ├─ [ IDK2-S5 ]
   ├─ [ IDK2-S6 ]
   │
   ▼
[ D-B1 ]
   ├─ [ D-COM ]
   ├─ [ D-M1 ]
   │
   ▼
[ L4 ]
   │
   ▼
[ E-B1 ]
   ├─ [ E-COM ]
   ├─ [ E-S1 ]
   ├─ [ E-S2 ]
   ├─ [ E-S3 ]
   ├─ [ E-S4 ]
   ├─ [ E-S5 ]
   ├─ [ E-S6 ]
   │
   ▼
[ L5 ]
   │
   ▼
[ F-B2 ]
   ├─ [ F-COM ]
   ├─▶ [ F-

In [6]:
# 2.6b [KODESJEKK] Structural validation 

import networkx as nx

assert "G_topology" in globals(), "Mangler G_topology (kjør 2.6)"
G = G_topology

struct_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get("relation") in ("main", "branch")]
S = nx.DiGraph()
S.add_nodes_from(G.nodes(data=True))
S.add_edges_from(struct_edges)

problems = 0

# cycles
cyc = list(nx.simple_cycles(S))
if cyc:
    problems += 1
    print("❌ CYCLES found in (main+branch)! Example cycle:", cyc[0])

# >1 main successor
multi_main = []
for n in S.nodes:
    succ_main = [v for v in S.successors(n) if G[n][v].get("relation") == "main"]
    if len(succ_main) > 1:
        multi_main.append((n, succ_main))
if multi_main:
    problems += 1
    print("⚠️ Nodes with >1 MAIN successor (review):")
    for n, succs in multi_main[:20]:
        print(" ", n, "->", succs)

# scope touched
scope_struct = [(u, v) for (u, v) in struct_edges
                if G.nodes[u].get("kind") == "scope" or G.nodes[v].get("kind") == "scope"]
if scope_struct:
    problems += 1
    print("⚠️ Structural edges touching scope-nodes:", scope_struct[:10])

if problems == 0:
    print("✅ Topology structural validation OK (DAG, no ambiguous MAIN, no scope structural edges).")


✅ Topology structural validation OK (DAG, no ambiguous MAIN, no scope structural edges).


In [7]:
# 2.65 [SETUP] Derive LINE_IDS / BREAKER_IDS / SENSOR_IDS / COMM_IDS from topology (Funksjonstabell truth)

from typing import List

assert "topology" in globals(), "Mangler topology (bygges i 2.6)"

def _nodes_by_kind(kind: str) -> List[str]:
    return sorted([n for n, d in topology.nodes(data=True) if d.get("kind") == kind])

LINE_IDS: List[str]    = _nodes_by_kind("line")
BREAKER_IDS: List[str] = _nodes_by_kind("breaker")

# Sensor/indicator: du har 'kind' = sensor og indicator (fra _infer_kind)
SENSOR_IDS: List[str]  = sorted(
    [n for n, d in topology.nodes(data=True) if d.get("kind") in ("sensor", "indicator")]
)

COMM_IDS: List[str]    = _nodes_by_kind("comm")

print("Derived from Funksjonstabell/topology:")
print("  Lines   :", len(LINE_IDS), LINE_IDS[:20], "..." if len(LINE_IDS) > 20 else "")
print("  Breakers:", len(BREAKER_IDS), BREAKER_IDS[:20], "..." if len(BREAKER_IDS) > 20 else "")
print("  Sensors :", len(SENSOR_IDS), SENSOR_IDS[:20], "..." if len(SENSOR_IDS) > 20 else "")
print("  Comms   :", len(COMM_IDS), COMM_IDS)


Derived from Funksjonstabell/topology:
  Lines   : 10 ['L1', 'L2', 'L21', 'L3', 'L4', 'L5', 'L6', 'L61', 'L7', 'L8'] 
  Breakers: 14 ['A-B1', 'B-B2', 'B-B3', 'B-B4', 'C-B2', 'C-B3', 'C-B4', 'D-B1', 'E-B1', 'F-B1', 'F-B2', 'F-B3', 'G-B1', 'H-B1'] 
  Sensors : 46 ['A-M1', 'A-M2', 'A-M3', 'A-S1', 'A-S2', 'A-S3', 'A-S4', 'A-S5', 'B-S1', 'C-M1', 'C-M2', 'C-M3', 'C-M4', 'C-S1', 'C-S2', 'D-M1', 'E-S1', 'E-S2', 'E-S3', 'E-S4'] ...
  Comms   : 11 ['A-COM', 'B-COM', 'C-COM', 'D-COM', 'E-COM', 'F-COM', 'G-COM', 'H-COM', 'IDK2-COM', 'IDK5-COM', 'IDK8-COM']


In [8]:
# 2.66 [KODESJEKK] Quick peek: first 10 IDs from topology-derived lists

def _head10(xs):
    xs = list(xs) if xs is not None else []
    return xs[:10], len(xs)

h, n = _head10(LINE_IDS)
print(f"LINE_IDS (n={n}) first10:", h)

h, n = _head10(BREAKER_IDS)
print(f"BREAKER_IDS (n={n}) first10:", h)

h, n = _head10(SENSOR_IDS)
print(f"SENSOR_IDS (n={n}) first10:", h)

h, n = _head10(COMM_IDS)
print(f"COMM_IDS (n={n}) first10:", h)


LINE_IDS (n=10) first10: ['L1', 'L2', 'L21', 'L3', 'L4', 'L5', 'L6', 'L61', 'L7', 'L8']
BREAKER_IDS (n=14) first10: ['A-B1', 'B-B2', 'B-B3', 'B-B4', 'C-B2', 'C-B3', 'C-B4', 'D-B1', 'E-B1', 'F-B1']
SENSOR_IDS (n=46) first10: ['A-M1', 'A-M2', 'A-M3', 'A-S1', 'A-S2', 'A-S3', 'A-S4', 'A-S5', 'B-S1', 'C-M1']
COMM_IDS (n=11) first10: ['A-COM', 'B-COM', 'C-COM', 'D-COM', 'E-COM', 'F-COM', 'G-COM', 'H-COM', 'IDK2-COM', 'IDK5-COM']


In [9]:
# 2.7 [KODESJEKK] Sanity checks for topology built from Funksjonstabell

import networkx as nx

assert "G_topology" in globals() and G_topology.number_of_nodes() > 0, "G_topology er tom – sjekk parsing/kolonner"

# quick spot checks (do not hard-fail if your naming differs; prints warnings)
must_have = ["A-B1", "L1", "B-B2", "L2", "C-B2", "L3"]
missing = [n for n in must_have if n not in G_topology.nodes]
if missing:
    print("WARN: Mangler forventede noder:", missing)

# show a few example edges
sample_edges = list(G_topology.edges(data=True))[:1000]
print("Example edges (first 15):")
for u, v, d in sample_edges:
    print(" ", u, "->", v, d)

# downstream test if A-B1 exists
if "A-B1" in G_topology.nodes:
    desc = nx.descendants(G_topology, "A-B1")
    print("Downstream count from A-B1:", len(desc))


Example edges (first 15):
  A-B1 -> L1 {'relation': 'main'}
  A-B1 -> A-B2 {'relation': 'attach'}
  A-B1 -> A-M1 {'relation': 'attach'}
  A-B1 -> A-M2 {'relation': 'attach'}
  A-B1 -> A-M3 {'relation': 'attach'}
  A-B1 -> A-S1 {'relation': 'attach'}
  A-B1 -> A-S2 {'relation': 'attach'}
  A-B1 -> A-S3 {'relation': 'attach'}
  A-B1 -> A-S4 {'relation': 'attach'}
  A-B1 -> A-S5 {'relation': 'attach'}
  A-B1 -> A-COM {'relation': 'attach'}
  A-COM -> A-B1 {'relation': 'serves'}
  A-COM -> A-M1 {'relation': 'serves'}
  A-COM -> A-M2 {'relation': 'serves'}
  A-COM -> A-M3 {'relation': 'serves'}
  A-COM -> A-S1 {'relation': 'serves'}
  A-COM -> A-S2 {'relation': 'serves'}
  A-COM -> A-S3 {'relation': 'serves'}
  A-COM -> A-S4 {'relation': 'serves'}
  A-COM -> A-S5 {'relation': 'serves'}
  L1 -> B-B2 {'relation': 'main'}
  B-B2 -> B-B4 {'relation': 'main'}
  B-B2 -> B-B3 {'relation': 'branch'}
  B-B2 -> B-B1 {'relation': 'attach'}
  B-B2 -> B-COM {'relation': 'attach'}
  B-B3 -> L21 {'relatio

In [10]:
# 2.8 [SETUP] Topology helpers: upstream/downstream, and node attribute access

import networkx as nx

assert "topology" in globals(), "Mangler topology (skal komme fra 2.6)"

def topo_downstream(node: str, G: nx.DiGraph = None) -> set:
    G = topology if G is None else G
    if node not in G:
        return set()
    return set(nx.descendants(G, node))

def topo_upstream(node: str, G: nx.DiGraph = None) -> set:
    G = topology if G is None else G
    if node not in G:
        return set()
    return set(nx.ancestors(G, node))

def topo_node_attr(node: str, attr: str, default=None, G: nx.DiGraph = None):
    G = topology if G is None else G
    if node not in G:
        return default
    return G.nodes[node].get(attr, default)

def topo_nodes_by_attr(attr: str, value: str, G: nx.DiGraph = None) -> list:
    G = topology if G is None else G
    out = []
    for n, d in G.nodes(data=True):
        if d.get(attr) == value:
            out.append(n)
    return out

print("Topology helpers ready.")


Topology helpers ready.


In [11]:
# 2.9 [AGENT] Influence-set definition (scope-based)

# Defines how each fault_loc_key influences nearby scopes in the topology.
#
# - influence_set_for_fault_loc(k): 
#       Returns the set of scope-nodes within INFLUENCE_RADIUS of k.
#
# - build_influence_map(keys):
#       Returns a long DataFrame:
#           target_fault_loc_key | influence_component_id
#
# influence_component_id is a SCOPE identifier (e.g. "A", "D", "IDK2").
#
# INFLUENCE_RADIUS controls neighbourhood size.


import pandas as pd
import networkx as nx
import inspect

assert "G_topology" in globals(), "Mangler G_topology (topologi-grafen må være laget før 2.9)"

# -------------------------
# Config (one place)
# -------------------------
INFLUENCE_RADIUS = 2  # juster her senere (ikke i 16.x)

def _norm_scope(x) -> str:
    if x is None or pd.isna(x):
        return ""
    return str(x).strip()


def influence_set_for_fault_loc(fault_loc_key: str) -> set[str]:
    """
    Scope-basert influence set.
    - Hvis fault_loc_key finnes i grafen: return ego-neighborhood med INFLUENCE_RADIUS.
    - Hvis ikke: fallback = {fault_loc_key} (som scope-string).
    """
    node = _norm_scope(fault_loc_key)
    if node == "":
        return set()

    if node not in G_topology:
        return {node}

    H = nx.ego_graph(G_topology, node, radius=INFLUENCE_RADIUS)
    return set(map(str, H.nodes()))

def build_influence_map(target_fault_loc_keys) -> pd.DataFrame:
    """
    Returnerer lang DF:
      target_fault_loc_key | influence_component_id
    influence_component_id er SCOPE (ikke Component ID).
    """
    rows = []
    for k in target_fault_loc_keys:
        k_norm = _norm_scope(k)
        if k_norm == "":
            continue
        S = influence_set_for_fault_loc(k_norm)
        for infl in S:
            rows.append((k_norm, str(infl)))

    influence_map = pd.DataFrame(rows, columns=["target_fault_loc_key", "influence_component_id"])
    if influence_map.empty:
        influence_map = pd.DataFrame(columns=["target_fault_loc_key", "influence_component_id"])
    return influence_map

print("Influence-set functions loaded.")



Influence-set functions loaded.


In [12]:
# 7.1A [AGENT] STATUS MAPPING 

import pandas as pd

status_mapping = pd.DataFrame({
    "Status": [
        "Aktiv","Feil","Frakopling","Inne","Mellom","Normal","Ok","Start","Transient",
        "Ute","Utkopl","Utlst","Varsel","Permanent",
        "Grn","Rd","Rødd",
        "Perm.retn. Grønn","Perm.retn. Rødd",
        "Trans.retn.. Grønn","Trans.retn.. Rød","Trans.retn.. Rødd",
        "nan",
        "OK","FEIL",
    ],
    "category": [
        "ALARM","FAULT_INDICATION","OPEN","CLOSED","UNKNOWN","NORMAL","NORMAL","WARNING","NORMAL",
        "OPEN","TRIPPED","TRIPPED","WARNING","NORMAL",
        "EARTH_FAULT_IND","EARTH_FAULT_IND","EARTH_FAULT_IND",
        "EARTH_FAULT_IND","EARTH_FAULT_IND",
        "EARTH_FAULT_IND","EARTH_FAULT_IND","EARTH_FAULT_IND",
        "UNKNOWN",
        "NORMAL","FAULT_INDICATION",
    ]
})

status_mapping["Status"] = status_mapping["Status"].astype(str).str.strip()
status_mapping["category"] = status_mapping["category"].astype(str).str.strip()

display(status_mapping)


,Status,category
0,Aktiv,ALARM
1,Feil,FAULT_INDICATION
2,Frakopling,OPEN
3,Inne,CLOSED
4,Mellom,UNKNOWN
5,Normal,NORMAL
6,Ok,NORMAL
7,Start,WARNING
8,Transient,NORMAL
9,Ute,OPEN


In [13]:
# 7.1B [AGENT] LOAD ALL YEAR EVENTS + COM

import pandas as pd
from pathlib import Path

def load_all_year_events(years, data_dir: Path) -> pd.DataFrame:
    frames = []
    missing_normal = []
    missing_com = []

    for year in years:
        p = data_dir / f"{year}_events.xlsx"
        if p.exists():
            d = pd.read_excel(p)
            d.columns = d.columns.str.strip()
            d["year"] = year
            frames.append(d)
        else:
            missing_normal.append(str(p))

        pc = data_dir / f"{year}_events_COM.xlsx"
        if pc.exists():
            d = pd.read_excel(pc)
            d.columns = d.columns.str.strip()
            d["year"] = year
            frames.append(d)
        else:
            missing_com.append(str(pc))

    if not frames:
        raise RuntimeError("Ingen event-filer lasta (normal eller COM).")

    events_all = pd.concat(frames, ignore_index=True)

    required = ["Status", "Component ID", "timestamp", "year"]
    missing = [c for c in required if c not in events_all.columns]
    if missing:
        raise ValueError(f"Mangler kolonner etter load: {missing}. Har: {list(events_all.columns)}")

    events_all["Status"] = events_all["Status"].astype(str).str.strip()
    events_all["Component ID"] = events_all["Component ID"].astype(str).str.strip()
    events_all["timestamp"] = pd.to_datetime(events_all["timestamp"], format="mixed", utc=True, errors="coerce")

    n_bad = int(events_all["timestamp"].isna().sum())
    if n_bad:
        events_all = events_all.dropna(subset=["timestamp"]).reset_index(drop=True)

    if missing_normal:
        print(f"[WARN] Missing normal events files: {len(missing_normal)} (first 3):", missing_normal[:3])
    if missing_com:
        print(f"[INFO] Missing COM events files: {len(missing_com)} (first 3):", missing_com[:3])

    print(
    f"OK: events loaded | rows={len(events_all)} | years={sorted(events_all['year'].unique().tolist())} "
    f"| dropped_bad_timestamp={n_bad}"
)

    return events_all

events_all = load_all_year_events(years=YEARS, data_dir=DATA_DIR)


OK: events loaded | rows=95678 | years=[2018, 2019, 2020, 2021] | dropped_bad_timestamp=158


In [14]:
print("YEARS:", YEARS)
print("DATA_DIR:", DATA_DIR)

display(events_all.groupby("year").size())

# sjekk tidsspenn per år
span = (events_all.groupby("year")["timestamp"]
        .agg(["min","max","count"])
        .sort_index())
display(span)


YEARS: [2018, 2019, 2020, 2021]
DATA_DIR: C:\Python\ADA511_prosjekt\data


year
2018    89029
2019     3044
2020     2091
2021     1514
dtype: int64

,min,max,count
year,,,
2018,2018-01-04 02:49:49.049000+00:00,2018-12-31 14:12:19.855000+00:00,89029
2019,2019-01-01 00:37:55.455000+00:00,2019-12-31 14:40:33.425000+00:00,3044
2020,2020-01-01 15:05:04.337000+00:00,2020-12-31 21:06:51.964000+00:00,2091
2021,2021-01-01 03:48:56.127000+00:00,2021-12-29 19:02:47.554000+00:00,1514


In [15]:
# %% 7.2 [AGENT] Normalize event rows → sensor_evidence_df (minimal contract, backwards compatible)

import pandas as pd

assert "events_all" in globals(), "Mangler events_all (køyr 7.1B)"
assert "status_mapping" in globals(), "Mangler status_mapping (køyr 7.1A)"

def normalize_event_rows_status_to_category(
    events_all: pd.DataFrame,
    status_mapping: pd.DataFrame,
) -> pd.DataFrame:
    """
    Row-level normalization:
      - Map raw Status text → semantic category via status_mapping
      - Standardize sensor_id = Component ID
      - Keep one row per original event (no time-window aggregation here)
    """
    ev = events_all.copy()

    # ensure clean join keys
    ev["Status"] = ev["Status"].astype(str).str.strip()
    ev["Component ID"] = ev["Component ID"].astype(str).str.strip()

    ev = ev.merge(status_mapping, on="Status", how="left")
    ev["category"] = ev["category"].fillna("UNKNOWN")

    ev["sensor_id"] = ev["Component ID"]
    return ev

# Funksjonsnavn beholdt for konsistens i resten av notebooken

def preprocess_events_with_mapping(events_all: pd.DataFrame, status_mapping: pd.DataFrame) -> pd.DataFrame:
    return normalize_event_rows_status_to_category(events_all, status_mapping)

events_pp = preprocess_events_with_mapping(events_all, status_mapping)

# sensor_evidence_df: rå eventrader (ingen tidsaggregasjon)

sensor_evidence_df = events_pp[["sensor_id", "timestamp", "year", "Status", "category"]].copy()
sensor_evidence_df = sensor_evidence_df.rename(columns={"timestamp": "window_start"})
sensor_evidence_df["window_start"] = pd.to_datetime(sensor_evidence_df["window_start"], utc=True)

# --- Kodesjekk (kompakt mapping-rapport) ---

n_rows    = len(sensor_evidence_df)
n_sensors = sensor_evidence_df["sensor_id"].nunique()
n_years   = sensor_evidence_df["year"].nunique()

vc_cat = sensor_evidence_df["category"].value_counts(dropna=False)
n_unknown = int(vc_cat.get("UNKNOWN", 0))

print(
    "OK: sensor_evidence_df built | "
    f"rows={n_rows} | sensors={n_sensors} | years={n_years} | "
    f"UNKNOWN={n_unknown} ({(n_unknown/max(n_rows,1)):.2%})"
)

print("Top categories:", vc_cat.head(8).to_dict())

if n_unknown > 0:
    top_unknown = (
        sensor_evidence_df.loc[sensor_evidence_df["category"]=="UNKNOWN", "Status"]
        .value_counts(dropna=False)
        .head(10)
        .to_dict()
    )
    print("Top UNKNOWN Status:", top_unknown)



OK: sensor_evidence_df built | rows=95678 | sensors=57 | years=4 | UNKNOWN=18 (0.02%)
Top categories: {'NORMAL': 84087, 'FAULT_INDICATION': 10520, 'WARNING': 491, 'OPEN': 196, 'CLOSED': 181, 'EARTH_FAULT_IND': 106, 'TRIPPED': 74, 'UNKNOWN': 18}
Top UNKNOWN Status: {'Mellom': 18}


In [16]:
# 10.1 [RESEARCH] Build helper structures for topology logic

import networkx as nx

# Identify IDK sensors (earth-fault indicators)
IDK_SENSORS = [s for s in SENSOR_IDS if "IDK" in s]

def scope_of_sensor_id(sensor_id: str) -> str:
    """
    ScopeGroup = delen før første '-'
    Eksempel: 'C-COM' -> 'C', 'A-S1' -> 'A'
    """
    if not isinstance(sensor_id, str) or "-" not in sensor_id:
        return str(sensor_id)
    return sensor_id.split("-", 1)[0].strip()

def sensors_in_same_scope(sensor_list, scope: str):
    """
    Returner alle sensorar i sensor_list som tilhøyrer same scope.
    """
    scope = str(scope).strip()
    prefix = scope + "-"
    return [s for s in sensor_list if isinstance(s, str) and s.startswith(prefix)]

def topo_nodes_in_scope(topology, scope: str):
    """
    Returner alle noder i topologi-grafen som tilhøyrer same scope (basert på prefix 'SCOPE-').
    """
    scope = str(scope).strip()
    prefix = scope + "-"
    return [n for n in topology.nodes if isinstance(n, str) and n.startswith(prefix)]


In [17]:
# %% 10.2 [AGENT] Topology Consistency Engine

import pandas as pd
import numpy as np
import networkx as nx

assert "sensor_evidence_df" in globals(), "Mangler sensor_evidence_df (køyr 7.2)"
assert "topology" in globals(), "Mangler topology (NetworkX graph)"

def scope_of_sensor_id(s: str) -> str:
    s = "" if s is None else str(s)
    return s.split("-", 1)[0].strip()


def build_topology_consistency_features(eventflow: pd.DataFrame, topology) -> pd.DataFrame:

    ef = eventflow.copy()

    # --- Standardiser tid ---
    ef["window_start"] = pd.to_datetime(ef["window_start"], utc=True, errors="coerce")
    ef = ef.dropna(subset=["window_start"]).copy()
    ef["timestamp"] = ef["window_start"]
    ef["window_start_15min"] = ef["timestamp"].dt.floor("15min")

    ef["sensor_id"] = ef["sensor_id"].astype(str).str.strip()
    ef["sensor_scope"] = ef["sensor_id"].map(scope_of_sensor_id)

    # --------------------------------------------------
    # RULE 1: Earth-fault without follow-up trip
    # --------------------------------------------------
    ef["topo_earthfault_missing_trip"] = 0

    if "category" in ef.columns:
        for idx, row in ef.iterrows():
            if row["category"] == "EARTH_FAULT_IND":
                sid = row["sensor_id"]

                downstream = [n for n in topology.nodes
                              if nx.has_path(topology, sid, n)]

                timewin = row["timestamp"] + pd.Timedelta(seconds=30)

                trips = ef[
                    (ef["sensor_id"].isin(downstream)) &
                    (ef["timestamp"] > row["timestamp"]) &
                    (ef["timestamp"] <= timewin) &
                    (ef["category"] == "TRIPPED")
                ]

                if trips.empty:
                    ef.at[idx, "topo_earthfault_missing_trip"] = 1

    # --------------------------------------------------
    # RULE 2: Trip without preceding earth-fault
    # --------------------------------------------------
    ef["topo_trip_without_earthfault"] = 0

    if "category" in ef.columns:
        for idx, row in ef.iterrows():
            if row["category"] == "TRIPPED":
                bid = row["sensor_id"]
                upstream = [n for n in topology.nodes
                            if nx.has_path(topology, n, bid)]

                timewin = row["timestamp"] - pd.Timedelta(seconds=30)

                earths = ef[
                    (ef["sensor_id"].isin(upstream)) &
                    (ef["timestamp"] >= timewin) &
                    (ef["timestamp"] <= row["timestamp"]) &
                    (ef["category"] == "EARTH_FAULT_IND")
                ]

                if earths.empty:
                    ef.at[idx, "topo_trip_without_earthfault"] = 1

    # --------------------------------------------------
    # RULE 3: Inconsistent direction (multiple IDK same ts)
    # --------------------------------------------------
    ef["topo_inconsistent_direction"] = 0

    if "category" in ef.columns:
        blocks = ef[ef["category"] == "EARTH_FAULT_IND"]
        grouped = blocks.groupby("timestamp")

        for ts, grp in grouped:
            if len(grp["sensor_id"].unique()) > 1:
                ef.loc[grp.index, "topo_inconsistent_direction"] = 1

    # --------------------------------------------------
    # RULE 4: COM missing propagation
    # --------------------------------------------------
    ef["topo_comms_missing_propagation"] = 0

    COM_PROBLEM_CATEGORIES = {"FAULT_INDICATION", "WARNING", "UNKNOWN"}

    is_com = ef["sensor_id"].str.contains("COM", na=False)

    noncom_exists = (
        ef.loc[~is_com]
          .groupby(["window_start_15min", "sensor_scope"])
          .size()
          .rename("n_noncom")
          .reset_index()
    )

    ef = ef.merge(noncom_exists, on=["window_start_15min", "sensor_scope"], how="left")
    ef["n_noncom"] = ef["n_noncom"].fillna(0).astype(int)

    com_problem = is_com & ef["category"].isin(COM_PROBLEM_CATEGORIES)
    ef.loc[com_problem & (ef["n_noncom"] > 0), "topo_comms_missing_propagation"] = 1

    ef = ef.drop(columns=["n_noncom"])

    # --------------------------------------------------
    # RULE 5: Illegal sequence (no dependency anymore)
    # --------------------------------------------------
    if "impossible_transition" in ef.columns:
        ef["topo_illegal_sequence"] = ef["impossible_transition"].fillna(0).astype(int)
    else:
        ef["topo_illegal_sequence"] = 0

    # --------------------------------------------------
    # RULE 6: Interlock violation
    # --------------------------------------------------
    INTERLOCK_PAIRS = [
        ("A-B1", "A-B2"),
        ("B-B1", "B-B2"),
        ("B-B4", "B-B5"),
        ("C-B1", "C-B2"),
        ("C-B4", "C-B5"),
    ]

    ef["topo_illegal_interlock"] = 0

    closed = ef[ef["category"] == "CLOSED"].copy()

    if not closed.empty:
        agg = (
            closed.groupby(["window_start_15min", "sensor_scope", "sensor_id"], as_index=False)
                  .size()
                  .assign(is_closed=1)
        )

        wide = agg.pivot_table(
            index=["window_start_15min", "sensor_scope"],
            columns="sensor_id",
            values="is_closed",
            fill_value=0,
        )

        bad_windows = set()
        for (a, b) in INTERLOCK_PAIRS:
            if a in wide.columns and b in wide.columns:
                bad_idx = wide[(wide[a] == 1) & (wide[b] == 1)].index
                bad_windows.update(list(bad_idx))

        if bad_windows:
            bad_df = pd.DataFrame(list(bad_windows),
                                  columns=["window_start_15min", "sensor_scope"])

            ef = ef.merge(
                bad_df.assign(topo_illegal_interlock=1),
                on=["window_start_15min", "sensor_scope"],
                how="left",
                suffixes=("", "_flag")
            )

            ef["topo_illegal_interlock"] = (
                ef["topo_illegal_interlock_flag"].fillna(0).astype(int)
            )

            ef = ef.drop(columns=["topo_illegal_interlock_flag"])

    return ef


In [18]:
# 10.2a [AGENT] Build topo_features (required by 16.12)

assert "sensor_evidence_df" in globals(), "Mangler sensor_evidence_df (køyr 7.2)"
assert "topology" in globals(), "Mangler topology (køyr 2.6)"
assert "build_topology_consistency_features" in globals(), "Mangler build_topology_consistency_features (køyr 10.2)"

topo_features = build_topology_consistency_features(
    sensor_evidence_df,
    topology,
)

# minimal, nyttig output
need_cols = ["sensor_id", "window_start_15min", "topo_comms_missing_propagation", "topo_illegal_interlock"]
missing = [c for c in need_cols if c not in topo_features.columns]
assert not missing, f"topo_features manglar kolonner: {missing}. Har: {list(topo_features.columns)[:30]}"

print(
    "OK: topo_features built | "
    f"rows={len(topo_features)} | "
    f"comms_missing={int(topo_features['topo_comms_missing_propagation'].sum())} | "
    f"illegal_interlock={int(topo_features['topo_illegal_interlock'].sum())}"
)



OK: topo_features built | rows=95678 | comms_missing=49 | illegal_interlock=0


In [19]:
# 10.3 [RESEARCH] Topology anomaly summary


topo_summary = topo_features[
    [
        "topo_earthfault_missing_trip",
        "topo_trip_without_earthfault",
        "topo_inconsistent_direction",
        "topo_comms_missing_propagation",
        "topo_illegal_sequence"
    ]
].sum()

print("Topology-based anomaly counts:")
display(topo_summary)


Topology-based anomaly counts:


topo_earthfault_missing_trip      106
topo_trip_without_earthfault       74
topo_inconsistent_direction         0
topo_comms_missing_propagation     49
topo_illegal_sequence               0
dtype: int64

In [20]:
# 10.4 [SETUP] Safe sensor roles + activity + simple fault patterns


import pandas as pd
import numpy as np

assert "ft" in globals(), "Mangler ft"
assert "events_pp" in globals(), "Mangler events_pp (preprocessed events table)"

# ------------------------------------------------------------
# 10.4.0  Standardise event time column once: timestamp
# ------------------------------------------------------------
ev = events_pp.copy()

if "timestamp" in ev.columns:
    ev["timestamp"] = pd.to_datetime(ev["timestamp"], utc=True, errors="coerce")
elif "window_start" in ev.columns:
    ev["timestamp"] = pd.to_datetime(ev["window_start"], utc=True, errors="coerce")
else:
    raise AssertionError("events_pp manglar både 'timestamp' og 'window_start'")

# hygiene
assert "sensor_id" in ev.columns, "events_pp manglar sensor_id"
ev["sensor_id"] = ev["sensor_id"].astype(str).str.strip()
ev = ev.dropna(subset=["timestamp", "sensor_id"]).copy()

# ------------------------------------------------------------
# 10.4.1 Sensor-role = Component Name + role_simple
# ------------------------------------------------------------
sensor_meta = ft[["Component ID", "Component Name"]].copy()
sensor_meta["Component ID"] = sensor_meta["Component ID"].astype(str).str.strip()

sensor_meta["sensor_role"] = sensor_meta["Component Name"].astype(str).str.strip()

def simplify_role(name: str, cid: str) -> str:
    name_l = (name or "").lower()
    cid_u  = (cid or "").upper()
    if name_l == "earth-fault indicator":
        return "earth_fault_indicator"
    if name_l == "short-circuit indicator":
        return "short_circuit_indicator"
    if cid_u.endswith("-COM"):
        return "communication_node"
    return "other"

sensor_meta["role_simple"] = sensor_meta.apply(
    lambda row: simplify_role(str(row["Component Name"]), str(row["Component ID"])),
    axis=1,
)

# Merge roles into events (safe replacement for transition_enriched)
events_enriched = ev.merge(
    sensor_meta[["Component ID", "sensor_role", "role_simple"]],
    left_on="sensor_id",
    right_on="Component ID",
    how="left",
)

# ------------------------------------------------------------
# 10.4.2 Per-sensor activity stats (uses event rows directly)
#   NOTE: This counts event rows per sensor_id (not transitions)
# ------------------------------------------------------------
cat_col = "category" if "category" in events_enriched.columns else None

def _count_eq(series, val):
    return int((series == val).sum())

if cat_col is None:
    # fallback: just count total rows per sensor
    sensor_event_stats = (
        events_enriched.groupby("sensor_id")
        .agg(
            total_events=("timestamp", "count"),
            roles=("sensor_role", lambda r: ",".join(sorted(set(str(x) for x in r if pd.notna(x))))),
        )
        .reset_index()
    )
    sensor_event_stats["earth_fault_events"] = 0
    sensor_event_stats["short_circuit_events"] = 0
else:
    sensor_event_stats = (
        events_enriched.groupby("sensor_id")
        .agg(
            total_events=(cat_col, "count"),
            earth_fault_events=("role_simple", lambda r: _count_eq(r, "earth_fault_indicator")),
            short_circuit_events=("role_simple", lambda r: _count_eq(r, "short_circuit_indicator")),
            roles=("sensor_role", lambda r: ",".join(sorted(set(str(x) for x in r if pd.notna(x))))),
        )
        .reset_index()
    )

# Flag top 5% activity
if not sensor_event_stats.empty:
    q95 = sensor_event_stats["total_events"].quantile(0.95)
    sensor_event_stats["high_activity_suspect"] = (sensor_event_stats["total_events"] > q95).astype(int)
else:
    sensor_event_stats["high_activity_suspect"] = []

print("Per-sensor eventstatistikk (topp 20 etter total_events):")
display(sensor_event_stats.sort_values("total_events", ascending=False).head(20))

# ------------------------------------------------------------
# 10.4.3 sensor_id -> fault_loc_key mapping (from ft)
# ------------------------------------------------------------
assert "fault_loc_key" in ft.columns, "ft manglar fault_loc_key"

sensor_loc_map = (
    ft[["Component ID", "fault_loc_key"]]
      .dropna(subset=["fault_loc_key"])
      .drop_duplicates(subset=["Component ID"])
      .rename(columns={"Component ID": "sensor_id", "fault_loc_key": "location"})
      .reset_index(drop=True)
)

sensor_loc_map["sensor_id"] = sensor_loc_map["sensor_id"].astype(str).str.strip()
sensor_loc_map["location"]  = sensor_loc_map["location"].astype(str).str.strip()

print("OK: sensor_loc_map laget (fra Funksjonstabell). Head:")
display(sensor_loc_map.head(10))

# Add location to events_enriched via map (prefer this over parsing "A-S1" -> "A")
events_enriched = events_enriched.merge(sensor_loc_map, on="sensor_id", how="left")

# ------------------------------------------------------------
# 10.4.4 Solo fault pattern (15-min): only one location has fault-indicator events
# ------------------------------------------------------------
fault_events = events_enriched[
    events_enriched["role_simple"].isin(["earth_fault_indicator", "short_circuit_indicator"])
].copy()

if fault_events.empty:
    print("Ingen hendelser fra Earth-fault / Short-circuit indikatorer.")
    solo_fault_df = pd.DataFrame(columns=["fault_window", "active_locations", "solo_fault_location"])
else:
    fault_events["fault_window"] = fault_events["timestamp"].dt.floor("15min")

    solo_rows = []
    for win_ts, grp in fault_events.groupby("fault_window"):
        locs = sorted([x for x in grp["location"].dropna().unique().tolist() if str(x).strip() != ""])
        solo_rows.append({
            "fault_window": win_ts,
            "active_locations": locs,
            "solo_fault_location": 1 if len(locs) == 1 else 0,
        })

    solo_fault_df = pd.DataFrame(solo_rows)

print("\nAntall tidsvinduer med solo fault location (mistenkelig):",
      int(solo_fault_df["solo_fault_location"].sum()) if not solo_fault_df.empty else 0)
display(solo_fault_df.head(20))

# ------------------------------------------------------------
# 10.4.5 Expand solo-fault windows to 5-min (3 bins inside 15-min)
# ------------------------------------------------------------
if not solo_fault_df.empty:
    solo_expanded = solo_fault_df.explode("active_locations").rename(columns={"active_locations": "location"})

    solo_with_sensors = solo_expanded.merge(
        sensor_loc_map,  # location=fault_loc_key
        on="location",
        how="left",
    )

    display(solo_with_sensors.head(20))

    rows = []
    for _, r in solo_with_sensors.iterrows():
        if int(r.get("solo_fault_location", 0)) != 1:
            continue
        sid = r.get("sensor_id")
        if pd.isna(sid):
            continue

        base_ts = r["fault_window"]
        year = int(base_ts.year)

        for offset_min in (0, 5, 10):
            rows.append({
                "sensor_id": str(sid).strip(),
                "year": year,
                "window_start": base_ts + pd.Timedelta(minutes=offset_min),
                "solo_fault_location": 1,
            })

    solo_fault_5min = pd.DataFrame(rows).drop_duplicates()
else:
    solo_fault_5min = pd.DataFrame(columns=["sensor_id", "year", "window_start", "solo_fault_location"])

print("solo_fault_5min shape:", solo_fault_5min.shape)
display(solo_fault_5min.head())

# ------------------------------------------------------------
# SAFETY NOTE (for you): none of these objects are merged into merged_15
# ------------------------------------------------------------


Per-sensor eventstatistikk (topp 20 etter total_events):


,sensor_id,total_events,earth_fault_events,short_circuit_events,roles,high_activity_suspect
47,IDK5-COM,25121,0,0,SMS,1
52,IDK8-COM,25084,0,0,SMS,1
40,IDK2-COM,25079,0,0,SMS,1
23,E-COM,6164,0,0,IEC104,0
10,B-COM,5762,0,0,IEC104,0
43,IDK2-S3,4781,4781,0,Earth-fault Indicator,0
34,G-COM,1770,0,0,IEC104,0
17,C-COM,441,0,0,IEC104,0
3,A-S1,287,287,0,Earth-fault Indicator,0
14,C-B3,152,0,0,Circuit breaker,0


OK: sensor_loc_map laget (fra Funksjonstabell). Head:


,sensor_id,location
0,A-B1,A
1,A-B2,A
2,A-M1,A
3,A-M2,A
4,A-M3,A
5,A-S1,A
6,A-S2,A
7,A-S3,A
8,A-S4,A
9,A-S5,A



Antall tidsvinduer med solo fault location (mistenkelig): 166


,fault_window,active_locations,solo_fault_location
0,2018-04-04 08:45:00+00:00,[C],1
1,2018-06-20 13:45:00+00:00,[IDK2],1
2,2018-06-20 14:15:00+00:00,[IDK2],1
3,2018-06-21 08:30:00+00:00,"[IDK2, IDK5]",0
4,2018-06-21 10:00:00+00:00,"[IDK2, IDK5]",0
5,2018-06-22 08:15:00+00:00,[IDK5],1
6,2018-06-22 10:30:00+00:00,"[IDK2, IDK5]",0
7,2018-06-22 11:00:00+00:00,"[IDK2, IDK5]",0
8,2018-06-22 12:00:00+00:00,[IDK2],1
9,2018-06-22 13:00:00+00:00,[IDK2],1


,fault_window,location,solo_fault_location,sensor_id
0,2018-04-04 08:45:00+00:00,C,1,C-B1
1,2018-04-04 08:45:00+00:00,C,1,C-B2
2,2018-04-04 08:45:00+00:00,C,1,C-B3
3,2018-04-04 08:45:00+00:00,C,1,C-B4
4,2018-04-04 08:45:00+00:00,C,1,C-B5
5,2018-04-04 08:45:00+00:00,C,1,C-M1
6,2018-04-04 08:45:00+00:00,C,1,C-M2
7,2018-04-04 08:45:00+00:00,C,1,C-M3
8,2018-04-04 08:45:00+00:00,C,1,C-M4
9,2018-04-04 08:45:00+00:00,C,1,C-S1


solo_fault_5min shape: (5004, 4)


,sensor_id,year,window_start,solo_fault_location
0,C-B1,2018,2018-04-04 08:45:00+00:00,1
1,C-B1,2018,2018-04-04 08:50:00+00:00,1
2,C-B1,2018,2018-04-04 08:55:00+00:00,1
3,C-B2,2018,2018-04-04 08:45:00+00:00,1
4,C-B2,2018,2018-04-04 08:50:00+00:00,1


In [21]:
# 10.5 [RESEARCH] Årsfordeling events

import pandas as pd

assert "events_all" in globals(), "Mangler events_all"

print("Årsfordeling i events_all:")
display(events_all["year"].value_counts().sort_index())

if "fault_events" in globals():
    print("\nÅrsfordeling i fault_events (jordfeil/kortslutning indikatorar):")
    if "year" in fault_events.columns:
        display(fault_events["year"].value_counts().sort_index())
    else:
        # fallback if year missing
        _tmp = fault_events.copy()
        if "timestamp" in _tmp.columns:
            _tmp["year"] = pd.to_datetime(_tmp["timestamp"], utc=True, errors="coerce").dt.year
            display(_tmp["year"].value_counts().sort_index())
else:
    print("\nNB: fault_events finst ikkje (køyr 10.4x først om du vil).")


Årsfordeling i events_all:


year
2018    89029
2019     3044
2020     2091
2021     1514
Name: count, dtype: int64


Årsfordeling i fault_events (jordfeil/kortslutning indikatorar):


year
2018    5122
2019     240
2020     139
2021      57
Name: count, dtype: int64

In [22]:
print(events_all[["timestamp","year"]].sort_values("timestamp").head(3))
print(events_all[["timestamp","year"]].sort_values("timestamp").tail(3))


                            timestamp  year
6499 2018-01-04 02:49:49.049000+00:00  2018
8456 2018-01-05 12:48:50.608000+00:00  2018
8457 2018-01-05 12:52:56.742000+00:00  2018
                             timestamp  year
95496 2021-12-29 18:35:17.529000+00:00  2021
95497 2021-12-29 18:55:26.543000+00:00  2021
95498 2021-12-29 19:02:47.554000+00:00  2021


In [23]:
# 15.0 [AGENT] Load + standardize measurements


import pandas as pd

def scopegroup_from_component_id(component_id: str) -> str | None:
    if not isinstance(component_id, str) or "-" not in component_id:
        return None
    return component_id.split("-", 1)[0].strip()

def load_measurements_all_years(years=YEARS, mn_dir=MN_DIR) -> pd.DataFrame:
    frames = []
    for year in years:
        meas_dir = mn_dir / f"{year}_measurements"
        parts = sorted(meas_dir.glob("part_*.parquet"))
        assert parts, f"Ingen part_*.parquet funnet i {meas_dir}"

        dfy = pd.concat((pd.read_parquet(p) for p in parts), ignore_index=True)
        dfy["year"] = year
        frames.append(dfy)

    dfm_all = pd.concat(frames, ignore_index=True)
    return dfm_all







In [24]:
# 15.1 [AGENT] Last + standardiser måledata → dfm2

# ------------------------------------------------------------
# 0) Load
# ------------------------------------------------------------
dfm = load_measurements_all_years(years=YEARS, mn_dir=MN_DIR)

# ------------------------------------------------------------
# 1) Minimumskrav + robust tidskolonne (og tvungen UTC)
# ------------------------------------------------------------
needed = ["component_id", "quality_ok"]
missing = [c for c in needed if c not in dfm.columns]
assert not missing, f"Mangler kolonner i dfm: {missing}. Har: {list(dfm.columns)[:30]}"

time_col_candidates = ["window_start_5min", "timestamp", "time", "datetime", "window_start_15min"]
time_col = next((c for c in time_col_candidates if c in dfm.columns), None)
assert time_col is not None, f"Fant ingen tidskolonne. Har: {list(dfm.columns)[:30]}"

# UTC-standard (same prinsipp som 0.x)
dfm[time_col] = pd.to_datetime(dfm[time_col], utc=True, errors="coerce")
dfm = dfm[dfm[time_col].notna()].copy()

# Standardiser tidskolonnenamn: window_start_5min
if time_col != "window_start_5min":
    dfm = dfm.rename(columns={time_col: "window_start_5min"})
    time_col = "window_start_5min"

# ------------------------------------------------------------
# 2) Standardiser ID + quality_ok
# ------------------------------------------------------------
dfm["component_id"] = dfm["component_id"].astype(str).str.strip()
dfm["quality_ok"] = dfm["quality_ok"].astype(bool)

# ------------------------------------------------------------
# 3) Finn måleverdi-kolonne og standardiser → meas_value
#    (16.10 må kunne lese denne deterministisk)
# ------------------------------------------------------------
value_candidates = ["meas_value", "value", "measurement_value", "meas", "val", "reading"]
value_col = next((c for c in value_candidates if c in dfm.columns), None)

if value_col is None:
    # fallback: prøv å finne første numeriske kolonne som ikkje er meta
    meta = {"year", "component_id", "quality_ok", "scope", "date", "window_start_5min"}
    numeric_cols = [c for c in dfm.columns if c not in meta and pd.api.types.is_numeric_dtype(dfm[c])]
    assert numeric_cols, (
        "Fant ingen måleverdi-kolonne (meas_value/value/...). "
        "Legg til 'meas_value' i loaderen eller oppdater value_candidates."
    )
    value_col = numeric_cols[0]

if value_col != "meas_value":
    dfm = dfm.rename(columns={value_col: "meas_value"})

# ------------------------------------------------------------
# 4) Scope + date (det 15.2 forventer)
# ------------------------------------------------------------
dfm["scope"] = dfm["component_id"].apply(scopegroup_from_component_id)
dfm["date"]  = dfm["window_start_5min"].dt.floor("D")

# ------------------------------------------------------------
# 5) Year (dersom ikkje allereie)
# ------------------------------------------------------------
if "year" not in dfm.columns:
    dfm["year"] = dfm["window_start_5min"].dt.year.astype(int)

# ------------------------------------------------------------
# 6) Derive meas_type from Funksjonstabell
#
# Note:
#   In the current stateless agent, only voltage (kV) measurements
#   are used for evidential modelling (see 16.x → E_kV_15min).
#   The mapping below keeps support for power/current for future
#   extensions, but they are not used in the present inference model.
# ------------------------------------------------------------


COL_COMP = "Component ID"
COL_SIG  = "Measured / Status Signals"

assert COL_COMP in ft.columns, f"Mangler {COL_COMP} i ft"
assert COL_SIG in ft.columns, f"Mangler {COL_SIG} i ft"

ft_map = (
    ft[[COL_COMP, COL_SIG]]
      .dropna()
      .drop_duplicates(subset=[COL_COMP])
      .copy()
)

ft_map[COL_COMP] = ft_map[COL_COMP].astype(str).str.strip()

def meas_type_from_signal(sig: str) -> str:
    s = (sig or "").upper()
    if "KV" in s or "U_" in s or "VOLT" in s:
        return "voltage"
    if "MW" in s or "KW" in s or " P" in s or "POWER" in s:
        return "power"
    if " A" in s or s.startswith("A") or "I_" in s or "CURR" in s:
        return "current"
    return "unknown"

ft_map["meas_type"] = ft_map[COL_SIG].apply(meas_type_from_signal)

type_map = dict(zip(ft_map[COL_COMP], ft_map["meas_type"]))

dfm["meas_type"] = dfm["component_id"].map(type_map).fillna("unknown")


# ------------------------------------------------------------
# 7) Endelig standardisert dfm2 (det resten av pipeline bruker)
# ------------------------------------------------------------
dfm2 = dfm[
    ["year", "component_id", "window_start_5min", "date", "scope", "quality_ok", "meas_value", "meas_type"]
].copy()

# Hygiene
dfm2["meas_value"] = pd.to_numeric(dfm2["meas_value"], errors="coerce")
#dfm2 = dfm2[dfm2["meas_value"].notna()].copy()

print("dfm2 rader:", len(dfm2))
print("År-fordeling:\n", dfm2["year"].value_counts().sort_index())
print("meas_type fordeling:\n", dfm2["meas_type"].value_counts(dropna=False))
print("Tidsrange:", dfm2["window_start_5min"].min(), "→", dfm2["window_start_5min"].max())
print("Kolonner:", list(dfm2.columns))


dfm2 rader: 1096905
År-fordeling:
 year
2018    244438
2019    291187
2020    281024
2021    280256
Name: count, dtype: int64
meas_type fordeling:
 meas_type
voltage    553902
current    271502
power      271501
Name: count, dtype: int64
Tidsrange: 2018-01-01 01:00:00+00:00 → 2021-12-31 23:45:00+00:00
Kolonner: ['year', 'component_id', 'window_start_5min', 'date', 'scope', 'quality_ok', 'meas_value', 'meas_type']


In [25]:
# 15.1.1 [STATUS]
assert "dfm2" in globals(), "Mangler dfm2 (køyr 15.1 først)"
needed = ["year","component_id","window_start_5min","date","scope","quality_ok","meas_value","meas_type"]
missing = [c for c in needed if c not in dfm2.columns]
assert not missing, f"dfm2 manglar kolonner: {missing}"


In [26]:
# 15.2 [AGENT] Lag `meas_daily` (daglig evidens) fra `dfm`


import pandas as pd

# dfm må allerede finnes fra 15.1 og inneholde: date, scope, quality_ok
assert "date" in dfm.columns and "scope" in dfm.columns and "quality_ok" in dfm.columns

meas_daily_scope = (
    dfm.groupby(["date", "scope"], as_index=False)
       .agg(
           meas_total=("quality_ok", "size"),
           meas_valid=("quality_ok", lambda s: int(s.sum())),
           meas_invalid=("quality_ok", lambda s: int((~s).sum()))
       )
)

meas_daily_scope["meas_present"] = meas_daily_scope["meas_total"] > 0
meas_daily_scope["meas_invalid_rate"] = meas_daily_scope["meas_invalid"] / meas_daily_scope["meas_total"]

# Diskret evidens (enkel start)
def validity_state(r: float) -> str:
    if r == 0:
        return "all_valid"
    if r <= 0.2:
        return "some_invalid"
    return "many_invalid"

meas_daily_scope["E_meas_present"] = meas_daily_scope["meas_present"].map({True: "present", False: "absent"})
meas_daily_scope["E_meas_validity"] = meas_daily_scope["meas_invalid_rate"].apply(validity_state)

# Sjekk
n_rows   = len(meas_daily_scope)
n_days   = meas_daily_scope["date"].nunique()
n_scopes = meas_daily_scope["scope"].nunique()

# Fordeling av evidens
present_counts  = meas_daily_scope["E_meas_present"].value_counts().to_dict()
validity_counts = meas_daily_scope["E_meas_validity"].value_counts().to_dict()

worst_scope = (
    meas_daily_scope.groupby("scope")["meas_invalid_rate"]
    .mean()
    .sort_values(ascending=False)
    .head(3)
)

# Konverter til prosent med 1 desimal
worst_scope_pct = {
    k: f"{v*100:.1f}%" for k, v in worst_scope.items()
}


print(
    "OK: meas_daily_scope | "
    f"rows={n_rows} | days={n_days} | scopes={n_scopes}"
)
print("E_meas_present:", present_counts)
print("E_meas_validity:", validity_counts)
print("Worst scopes (mean invalid rate):", worst_scope_pct)



OK: meas_daily_scope | rows=4290 | days=1461 | scopes=3
E_meas_present: {'present': 4290}
E_meas_validity: {'all_valid': 3672, 'many_invalid': 404, 'some_invalid': 214}
Worst scopes (mean invalid rate): {'C': '7.7%', 'D': '6.0%', 'A': '1.9%'}


In [27]:
# 16.10 [AGENT] kV-evidens per 5-min vindu (per komponent) – REN dfm2-kontrakt

import pandas as pd
import numpy as np

assert "dfm2" in globals(), "Mangler dfm2 (køyr 15.1 først)."
df = dfm2.copy()

# ------------------------------------------------------------
# 1) Filtrer til spenningsmålingar (robust for gamle labels)
# ------------------------------------------------------------
voltage_labels = {"voltage", "kV", "KV"}
df = df[df["meas_type"].isin(voltage_labels)].copy()

# ------------------------------------------------------------
# 2) Sikre UTC-aware 5-min vindu (skal alt vere frå 15.1)
# ------------------------------------------------------------
assert "window_start_5min" in df.columns, "dfm2 manglar window_start_5min (skal kome frå 15.1)"
df["window_start_5min"] = pd.to_datetime(df["window_start_5min"], utc=True, errors="coerce")
df = df[df["window_start_5min"].notna()].copy()

# ------------------------------------------------------------
# 3) Tersklar per component_id
# ------------------------------------------------------------
KV_OK_BY_COMP = {
    "DEFAULT": (20.1, 22.9),
    "D-M1": (21.0, 26.0),
}

def kv_in_ok(v: float, comp_id: str) -> bool:
    lo, hi = KV_OK_BY_COMP.get(str(comp_id), KV_OK_BY_COMP["DEFAULT"])
    return (v >= lo) and (v <= hi)

# ------------------------------------------------------------
# 4) Klassifiser rad → present / absent_or_invalid / unknown
#    - unknown dersom quality_ok=False eller meas_value manglar
#    - absent_or_invalid dersom quality_ok=True men verdi utanfor terskel
# ------------------------------------------------------------
df["meas_value"] = pd.to_numeric(df["meas_value"], errors="coerce")

def classify_row(quality_ok: bool, v: float, comp_id: str) -> str:
    if not bool(quality_ok):
        return "unknown"
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "unknown"
    return "present" if kv_in_ok(float(v), comp_id) else "absent_or_invalid"

df["E_kV_present_row"] = [
    classify_row(q, v, cid)
    for q, v, cid in zip(df["quality_ok"].values, df["meas_value"].values, df["component_id"].values)
]

# ------------------------------------------------------------
# 5) Aggreger per (window_start_5min, component_id)
# ------------------------------------------------------------
def agg_window(s: pd.Series) -> str:
    if (s == "present").any():
        return "present"
    if (s == "absent_or_invalid").any():
        return "absent_or_invalid"
    return "unknown"

meas_5min_kV_evidence = (
    df.groupby(["window_start_5min", "component_id"])["E_kV_present_row"]
      .apply(agg_window)
      .reset_index(name="E_kV_present")
)


# Små sanity prints (kan fjernast i “super-ren”)
print("meas_5min_kV_evidence rows:", len(meas_5min_kV_evidence))
print("E_kV_present value_counts:\n", meas_5min_kV_evidence["E_kV_present"].value_counts(dropna=False))
print("time range:", meas_5min_kV_evidence["window_start_5min"].min(), "→", meas_5min_kV_evidence["window_start_5min"].max())


meas_5min_kV_evidence rows: 543003
E_kV_present value_counts:
 E_kV_present
present              367052
absent_or_invalid    138232
unknown               37719
Name: count, dtype: int64
time range: 2018-01-01 01:00:00+00:00 → 2021-12-31 23:45:00+00:00


In [28]:
# %% 16.12 [AGENT] Build merged_15 (deterministisk): meas + events + topo (no fallback)

import pandas as pd
import numpy as np

assert "ft" in globals(), "Mangler ft (køyr 0.x først)."
assert "sensor_evidence_df" in globals(), "Mangler sensor_evidence_df (event pipeline)."
assert "meas_5min_kV_evidence" in globals(), "Mangler meas_5min_kV_evidence (køyr 16.10)."
assert "topo_features" in globals(), (
    "Mangler topo_features. I ren stateless skal topo_features byggast før 16.12 "
    "(min: topo_comms_missing_propagation, topo_illegal_interlock)."
)

# -------------------------
# Helpers (små og stabile)
# -------------------------
def scope_of(x: str) -> str:
    x = "" if x is None else str(x)
    return x.split("-", 1)[0].strip()

def agg_present(series: pd.Series) -> str:
    s = series.dropna().astype(str)
    if len(s) == 0:
        return "unknown"
    if (s == "present").any():
        return "present"
    if (s == "absent_or_invalid").any():
        return "absent_or_invalid"
    return "unknown"

def agg_obj(series: pd.Series):
    s = series.dropna()
    if len(s) == 0:
        return pd.NA
    vocab = set(s.astype(str).unique())
    if vocab.issubset({"present", "absent_or_invalid", "unknown"}):
        return agg_present(s.astype(str))
    # fallback: første verdi (stabilt)
    return s.iloc[0]

# -------------------------
# A) events_df_base: 15-min + scope + minimumskolonner
# -------------------------
events_df_base = sensor_evidence_df.copy()

need = {"window_start", "sensor_id"}
missing = [c for c in need if c not in events_df_base.columns]
assert not missing, f"sensor_evidence_df manglar: {missing}. Har: {list(events_df_base.columns)[:30]}"

events_df_base["window_start"] = pd.to_datetime(events_df_base["window_start"], utc=True, errors="coerce")
events_df_base = events_df_base.dropna(subset=["window_start"]).copy()

events_df_base["window_start_15min"] = events_df_base["window_start"].dt.floor("15min")
events_df_base["sensor_id"] = events_df_base["sensor_id"].astype(str).str.strip()
events_df_base["sensor_scope"] = events_df_base["sensor_id"].map(scope_of)

# -------------------------
# B) events_topo_df: plukk ut berre topo-kolonner + merge ÉIN gong
# -------------------------
tf = topo_features.copy()

# robust time key i topo_features → window_start_15min
if "window_start_15min" not in tf.columns:
    if "window_start" in tf.columns:
        tf["window_start"] = pd.to_datetime(tf["window_start"], utc=True, errors="coerce")
        tf["window_start_15min"] = tf["window_start"].dt.floor("15min")
    elif "timestamp" in tf.columns:
        tf["timestamp"] = pd.to_datetime(tf["timestamp"], utc=True, errors="coerce")
        tf["window_start_15min"] = tf["timestamp"].dt.floor("15min")
    else:
        raise AssertionError("topo_features manglar window_start_15min og (window_start/timestamp).")

required_topo = {"sensor_id", "window_start_15min", "topo_comms_missing_propagation", "topo_illegal_interlock"}
missing_topo = [c for c in required_topo if c not in tf.columns]
assert not missing_topo, f"topo_features manglar: {missing_topo}. Har: {list(tf.columns)[:40]}"

tf["sensor_id"] = tf["sensor_id"].astype(str).str.strip()
tf["window_start_15min"] = pd.to_datetime(tf["window_start_15min"], utc=True, errors="coerce")
tf = tf.dropna(subset=["sensor_id", "window_start_15min"]).copy()

events_topo_df = (
    tf.groupby(["sensor_id", "window_start_15min"], as_index=False)[
        ["topo_comms_missing_propagation", "topo_illegal_interlock"]
    ].max()
)

# merge topo inn i events_df_base (deterministisk, ingen suffix-fiksar)
events_df = events_df_base.merge(events_topo_df, on=["sensor_id", "window_start_15min"], how="left")
events_df["topo_comms_missing_propagation"] = events_df["topo_comms_missing_propagation"].fillna(0).astype(int)
events_df["topo_illegal_interlock"] = events_df["topo_illegal_interlock"].fillna(0).astype(int)

# -------------------------
# C) Measurements: 5-min → 15-min per component_id (+ scope)
# -------------------------
meas_kv = meas_5min_kV_evidence.copy()

need_m = {"window_start_5min", "component_id", "E_kV_present"}
missing_m = [c for c in need_m if c not in meas_kv.columns]
assert not missing_m, f"meas_5min_kV_evidence manglar: {missing_m}. Har: {list(meas_kv.columns)[:40]}"

meas_kv["window_start_5min"] = pd.to_datetime(meas_kv["window_start_5min"], utc=True, errors="coerce")
meas_kv = meas_kv.dropna(subset=["window_start_5min"]).copy()

meas_kv["window_start_15min"] = meas_kv["window_start_5min"].dt.floor("15min")
meas_kv["component_id"] = meas_kv["component_id"].astype(str).str.strip()
meas_kv["component_scope"] = meas_kv["component_id"].map(scope_of)

meas_kv_15 = (
    meas_kv.groupby(["window_start_15min", "component_id", "component_scope"])["E_kV_present"]
           .apply(agg_present)
           .reset_index()
           .rename(columns={"E_kV_present": "E_kV_present_15min"})
)

# -------------------------
# D) Funksjonstabell mapping
#    - event-map: binary
#    - meas-map:  continuous/discrete
# -------------------------
ft_local = ft.copy()

# finn kolonnenavn som faktisk finnes
COL_COMP = "component_id" if "component_id" in ft_local.columns else "Component ID"
COL_DT   = "data_type"    if "data_type"    in ft_local.columns else "Data Type"

assert COL_COMP in ft_local.columns, f"ft mangler Component ID/component_id. Har: {list(ft_local.columns)[:40]}"
assert "fault_loc_key" in ft_local.columns, f"ft mangler fault_loc_key. Har: {list(ft_local.columns)[:40]}"
assert COL_DT in ft_local.columns, f"ft mangler Data Type/data_type. Har: {list(ft_local.columns)[:40]}"

dt = ft_local[COL_DT].astype(str).str.strip().str.lower()

is_meas  = dt.isin(["continuous", "discrete"])
is_event = dt.eq("binary")

map_meas = (
    ft_local.loc[is_meas, [COL_COMP, "fault_loc_key"]]
      .dropna(subset=[COL_COMP, "fault_loc_key"])
      .drop_duplicates(subset=[COL_COMP])
      .rename(columns={COL_COMP: "component_id"})
)

map_event = (
    ft_local.loc[is_event, [COL_COMP, "fault_loc_key"]]
      .dropna(subset=[COL_COMP, "fault_loc_key"])
      .drop_duplicates(subset=[COL_COMP])
      .rename(columns={COL_COMP: "sensor_id"})
)


# -------------------------
# E) Map to fault_loc_key
# -------------------------
events_mapped = events_df.merge(map_event, on="sensor_id", how="left")
meas_mapped   = meas_kv_15.merge(map_meas, on="component_id", how="left")

events_mapped = events_mapped[events_mapped["fault_loc_key"].notna()].copy()
meas_mapped   = meas_mapped[meas_mapped["fault_loc_key"].notna()].copy()

# -------------------------
# F) EVENTS → 15min × fault_loc_key
# -------------------------
event_e_cols = [c for c in events_mapped.columns if c.startswith("E_")]

if len(events_mapped) == 0:
    events_15min_faultloc = pd.DataFrame(
        {"window_start_15min": pd.to_datetime([], utc=True), "fault_loc_key": pd.Series([], dtype="object")}
    )
else:
    agg_dict = {}
    for c in event_e_cols:
        if pd.api.types.is_numeric_dtype(events_mapped[c]) or pd.api.types.is_bool_dtype(events_mapped[c]):
            agg_dict[c] = "max"
        else:
            agg_dict[c] = agg_obj

    # Ta med topo-felt også (dei er ikkje E_*)
    agg_dict["topo_comms_missing_propagation"] = "max"
    agg_dict["topo_illegal_interlock"] = "max"

    events_15min_faultloc = (
        events_mapped
        .groupby(["window_start_15min", "fault_loc_key"], as_index=False)
        .agg(agg_dict)
    )

events_15min_faultloc["window_start_15min"] = pd.to_datetime(events_15min_faultloc["window_start_15min"], utc=True)

# -------------------------
# G) MEASUREMENTS → 15min × fault_loc_key
# -------------------------
meas_15min_faultloc = (
    meas_mapped
    .groupby(["window_start_15min", "fault_loc_key"])["E_kV_present_15min"]
    .apply(agg_present)
    .reset_index()
    .rename(columns={"E_kV_present_15min": "E_kV_15min"})
)
meas_15min_faultloc["window_start_15min"] = pd.to_datetime(meas_15min_faultloc["window_start_15min"], utc=True)

# -------------------------
# H) Merge (måledrevet universe)
# -------------------------
merged_15 = meas_15min_faultloc.merge(
    events_15min_faultloc,
    on=["window_start_15min", "fault_loc_key"],
    how="left"
)

# -------------------------
# I) Contracts
# -------------------------
must_have = ["window_start_15min", "fault_loc_key", "E_kV_15min"]
missing2 = [c for c in must_have if c not in merged_15.columns]
assert not missing2, f"merged_15 manglar: {missing2}"

dups = merged_15.duplicated(["window_start_15min", "fault_loc_key"]).sum()
assert dups == 0, f"Duplikate rader i merged_15: {dups}"

# --- Kort, meir beskrivande output ---
n_rows    = len(merged_15)
n_windows = merged_15["window_start_15min"].nunique()
n_locs    = merged_15["fault_loc_key"].nunique()
kv_counts = merged_15["E_kV_15min"].value_counts(dropna=False).to_dict()

print(f"OK: merged_15 built (meas-driven) | rows={n_rows} | windows={n_windows} | locs={n_locs}")
print("E_kV_15min:", kv_counts)



OK: merged_15 built (meas-driven) | rows=411734 | windows=140232 | locs=3
E_kV_15min: {'present': 367052, 'absent_or_invalid': 29222, 'unknown': 15460}


In [29]:
# 16.12.1 [AGENT] Expand to FULL location universe (ALL fault_loc_key) per window_start_15min (REN)

import pandas as pd
import numpy as np

assert "ft" in globals(), "Mangler ft (køyr 0.x først)."
assert "merged_15" in globals(), "Mangler merged_15 (køyr 16.12 først)."
assert "events_mapped" in globals(), "Mangler events_mapped (skal bli laga i 16.12)."

# -------------------------
# 1) Alle fault_loc_key frå ft (global)
# -------------------------
all_faultloc = (
    ft["fault_loc_key"]
      .astype("string").str.strip()
      .replace({"nan": pd.NA, "NaN": pd.NA, "": pd.NA})
      .dropna()
      .unique()
)
all_faultloc = sorted(all_faultloc)

# -------------------------
# 2) Alle vinduer frå merged_15 (tidsakse)
# -------------------------
assert "window_start_15min" in merged_15.columns, "merged_15 manglar window_start_15min"

win_meas = pd.to_datetime(merged_15["window_start_15min"], utc=True, errors="coerce").dropna().unique()

win_ev = pd.to_datetime(events_mapped["window_start_15min"], utc=True, errors="coerce").dropna().unique()

all_windows = np.sort(np.unique(np.concatenate([win_meas, win_ev])))


base = (
    pd.MultiIndex.from_product(
        [all_windows, all_faultloc],
        names=["window_start_15min", "fault_loc_key"]
    )
    .to_frame(index=False)
)


# -------------------------
# 3) Merge inn ALT frå merged_15 (men base-univers styrer rader)
# -------------------------
m = merged_15.copy()
m["window_start_15min"] = pd.to_datetime(m["window_start_15min"], utc=True, errors="coerce")

# NA-safe keys (ikkje lag 'nan'-tekst)
m["fault_loc_key"] = m["fault_loc_key"].astype("string").str.strip()
m["fault_loc_key"] = m["fault_loc_key"].replace({"nan": pd.NA, "NaN": pd.NA, "": pd.NA})

m = m.dropna(subset=["window_start_15min", "fault_loc_key"]).copy()

out = base.merge(m, on=["window_start_15min", "fault_loc_key"], how="left")


# -------------------------
# 4) Aggregate events → binary E_event_any per 15min × fault_loc_key
# -------------------------
ev = events_mapped.copy()
ev["window_start_15min"] = pd.to_datetime(ev["window_start_15min"], utc=True, errors="coerce")
ev["fault_loc_key"] = ev["fault_loc_key"].astype("string").str.strip()
ev["fault_loc_key"] = ev["fault_loc_key"].replace({"nan": pd.NA, "NaN": pd.NA, "": pd.NA})
ev = ev.dropna(subset=["window_start_15min", "fault_loc_key"]).copy()

events_15_any = (
    ev.groupby(["window_start_15min", "fault_loc_key"], as_index=False)
      .size()
      .rename(columns={"size": "n_events"})
)
events_15_any["E_event_any"] = (events_15_any["n_events"] > 0).astype(int)
events_15_any = events_15_any[["window_start_15min", "fault_loc_key", "E_event_any"]]

# fjern ev. gammal kolonne før merge (unngå _x/_y)
out = out.drop(columns=["E_event_any"], errors="ignore")

out = out.merge(events_15_any, on=["window_start_15min", "fault_loc_key"], how="left")
out["E_event_any"] = out["E_event_any"].fillna(0).astype(int)


# E_kV_15min skal kome frå 16.12-merge (feil tidleg om ikkje)
assert "E_kV_15min" in out.columns, "Mangler E_kV_15min (køyr 16.12 først)"

# Standard vidare
merged_15 = out

# --- output ---
n_rows = len(merged_15)

n_kv = int(merged_15["E_kV_15min"].notna().sum())
n_event = int((merged_15["E_event_any"] == 1).sum())

print(f"FULL universe: {n_rows:,} rows")
print(f"kV evidence universe-coverage: {n_kv:,} rows ({n_kv/n_rows:.2%})")
print(f"Event-active windows: {n_event:,} rows ({n_event/n_rows:.2%})")




FULL universe: 1,542,563 rows
kV evidence universe-coverage: 411,734 rows (26.69%)
Event-active windows: 6,604 rows (0.43%)


In [30]:
# 16.12a [AGENT] Derive E_comm_fail_any (REN: frå merged_15 sine topo-kolonner)

import pandas as pd
import numpy as np

assert "merged_15" in globals(), "Mangler merged_15"
df = merged_15.copy()

# standard keys (hygiene)
df["window_start_15min"] = pd.to_datetime(df["window_start_15min"], utc=True, errors="coerce")
df["fault_loc_key"] = df["fault_loc_key"].astype(str).str.strip()

# topo_comms_missing_propagation skal kome frå 16.12 via topo_features->events->fault_loc
if "topo_comms_missing_propagation" not in df.columns:
    raise AssertionError(
        "merged_15 manglar topo_comms_missing_propagation. "
        "Sjekk 16.12: events_15min_faultloc må ta med topo_comms_missing_propagation."
    )

df["E_comm_fail_any"] = df["topo_comms_missing_propagation"].fillna(0).astype(int).clip(0, 1)

merged_15 = df

print("OK: E_comm_fail_any lagt til (frå topo_comms_missing_propagation).")
print(merged_15["E_comm_fail_any"].value_counts(dropna=False).head(10))


OK: E_comm_fail_any lagt til (frå topo_comms_missing_propagation).
E_comm_fail_any
0    1542563
Name: count, dtype: int64


In [31]:
# 16.12b [AGENT] Contract: topo_illegal_interlock finst (REN)

assert "merged_15" in globals(), "Mangler merged_15"

if "topo_illegal_interlock" not in merged_15.columns:
    raise AssertionError(
        "merged_15 manglar topo_illegal_interlock. "
        "Sjekk 16.12: events_15min_faultloc må ta med topo_illegal_interlock."
    )

# normaliser til int 0/1
merged_15["topo_illegal_interlock"] = merged_15["topo_illegal_interlock"].fillna(0).astype(int).clip(0, 1)

n_total = len(merged_15)
n_illegal = int(merged_15["topo_illegal_interlock"].sum())

print(f"Interlock violations: {n_illegal:,} of {n_total:,} rows ({n_illegal/n_total:.4%})")

if n_illegal == 0:
    print("→ No interlock violations detected in dataset.")



Interlock violations: 0 of 1,542,563 rows (0.0000%)
→ No interlock violations detected in dataset.


In [32]:
# 16.12c [AGENT] Derive evidence variables:

#   - E_topo_consistency (kV vs events)
#   - E_interlock (projection of topo_illegal_interlock into {none,present})


import pandas as pd
import numpy as np

assert "merged_15" in globals(), "Mangler merged_15"
assert "E_kV_15min" in merged_15.columns, "Mangler E_kV_15min"
assert "E_event_any" in merged_15.columns, "Mangler E_event_any"

df = merged_15.copy()

# -------------------------
# 1) Topo-consistency (3-state): unknown/consistent/inconsistent
# -------------------------
kv_state = (
    df["E_kV_15min"]
    .map({
        "present": "normal",
        "normal": "normal",
        "absent_or_invalid": "abnormal",
        "abnormal": "abnormal",
        "unknown": "unknown",
    })
    .fillna("unknown")
)

event_present = df["E_event_any"].fillna(0).astype(int).eq(1)

df["E_topo_consistency"] = "unknown"

mask_known = kv_state.ne("unknown")
mask_inconsistent = (
    (kv_state.eq("abnormal") & (~event_present)) |
    (kv_state.eq("normal") & (event_present))
) & mask_known

df.loc[mask_known & (~mask_inconsistent), "E_topo_consistency"] = "consistent"
df.loc[mask_inconsistent, "E_topo_consistency"] = "inconsistent"

# -------------------------
# 2) Interlock evidence (2-state): none/present
# -------------------------
df["E_interlock"] = "none"
if "topo_illegal_interlock" in df.columns:
    bad = df["topo_illegal_interlock"].fillna(0).astype(int).eq(1)
    df.loc[bad, "E_interlock"] = "present"
else:
    # i ren pipeline skal denne finnast; men vi feilar ikkje hardt her
    print("[WARN] topo_illegal_interlock manglar -> E_interlock blir 'none'")

merged_15 = df

n_total = len(merged_15)

# 1) topo-consistency
vc = merged_15["E_topo_consistency"].value_counts(dropna=False)
print("Topo-consistency (derived from E_kV_15min vs E_event_any):")
print(vc)
print(f"→ inconsistent rate: {int(vc.get('inconsistent', 0)):,}/{n_total:,} ({int(vc.get('inconsistent', 0))/max(n_total,1):.2%})")

# 2) interlock
n_present = int((merged_15["E_interlock"] == "present").sum())
print("\nInterlock evidence (derived from topo_illegal_interlock):")
print(merged_15["E_interlock"].value_counts(dropna=False))
print(f"→ present: {n_present:,}/{n_total:,} ({n_present/max(n_total,1):.4%})")



Topo-consistency (derived from E_kV_15min vs E_event_any):
E_topo_consistency
unknown         1146289
consistent       366575
inconsistent      29699
Name: count, dtype: int64
→ inconsistent rate: 29,699/1,542,563 (1.93%)

Interlock evidence (derived from topo_illegal_interlock):
E_interlock
none    1542563
Name: count, dtype: int64
→ present: 0/1,542,563 (0.0000%)


In [33]:
# 16.12d [KODESJEKK] Universe sanity

import pandas as pd

assert "events_mapped" in globals(), "Mangler events_mapped (kjør 16.12)"
assert "meas_mapped" in globals(), "Mangler meas_mapped (kjør 16.12)"
assert "merged_15" in globals(), "Mangler merged_15 (kjør 16.12)"

print(f"Events uten fault_loc_key etter mapping: {events_mapped['fault_loc_key'].isna().sum()} / {len(events_mapped)}")
print(f"Målinger uten fault_loc_key etter mapping: {meas_mapped['fault_loc_key'].isna().sum()} / {len(meas_mapped)}")

print("OK: merged_15 (AGENT-univers) laget.")
print("Tidsrom:", merged_15["window_start_15min"].min(), "→", merged_15["window_start_15min"].max())
print("Antall rader:", len(merged_15))

# Event-kolonner i merged_15 = alle E_* unntatt E_kV_15min
event_cols = [c for c in merged_15.columns if c.startswith("E_") and c != "E_kV_15min"]

if len(event_cols) > 0:
    print(
        "Andel rader med event-observasjon (minst én event-kolonne ikke-NaN):",
        merged_15[event_cols].notna().any(axis=1).mean()
    )
else:
    print("Ingen event_cols (E_* unntatt E_kV_15min) funnet i merged_15.")


Events uten fault_loc_key etter mapping: 0 / 95678
Målinger uten fault_loc_key etter mapping: 0 / 543003
OK: merged_15 (AGENT-univers) laget.
Tidsrom: 2018-01-01 01:00:00+00:00 → 2021-12-31 23:45:00+00:00
Antall rader: 1542563
Andel rader med event-observasjon (minst én event-kolonne ikke-NaN): 1.0


In [34]:
# 16.12e [KODESJEKK]



# Sjekk måledekning per fault_loc_key i merged_15.
# NB: E_kV finnes først etter 16.15, så her bruker vi E_kV_15min.


assert "merged_15" in globals(), "Mangler merged_15 (kjør 16.12 først)"
assert "fault_loc_key" in merged_15.columns, "Mangler fault_loc_key i merged_15"

# Finn riktig kolonne (tåler litt navnevariasjon)
if "E_kV_15min" in merged_15.columns:
    meas_col = "E_kV_15min"
elif "E_kV_present_15min" in merged_15.columns:
    meas_col = "E_kV_present_15min"
else:
    raise KeyError(
        "Fant ikke E_kV_15min eller E_kV_present_15min i merged_15. "
        "Sjekk 16.12 rename/kolonnenavn."
    )

# 1) Antall rader med måle-observasjon (ikke-NaN) per lokasjon
coverage = (
    merged_15.groupby("fault_loc_key")[meas_col]
    .apply(lambda s: s.notna().sum())
    .sort_values(ascending=False)
)

print(f"Bruker kolonne: {meas_col}")
display(coverage.head(30))

# 2) (valgfritt) Hvis du vil se fordeling av 'present/absent_or_invalid'
if merged_15[meas_col].dtype == "object":
    display(
        merged_15.groupby("fault_loc_key")[meas_col]
        .value_counts(dropna=False)
        .unstack(fill_value=0)
        .head(30)
    )




Bruker kolonne: E_kV_15min


fault_loc_key
A       140232
D       140232
C       131270
B            0
E            0
F            0
G            0
H            0
IDK2         0
IDK5         0
IDK8         0
Name: E_kV_15min, dtype: int64

E_kV_15min,absent_or_invalid,present,unknown,NaN
fault_loc_key,,,,
A,20444,117187,2601,1
B,0,0,0,140233
C,8751,114130,8389,8963
D,27,135735,4470,1
E,0,0,0,140233
F,0,0,0,140233
G,0,0,0,140233
H,0,0,0,140233
IDK2,0,0,0,140233


In [35]:
# 16.12f [KODESJEKK]


meas_15min_faultloc["fault_loc_key"].value_counts()


fault_loc_key
A    140232
D    140232
C    131270
Name: count, dtype: Int64

In [36]:
# 16.12g [KODESJEKK]


events_15min_faultloc["fault_loc_key"].value_counts()


fault_loc_key
B       3109
G       1321
E        846
C        371
IDK5     206
IDK2     205
IDK8     160
A        147
F         98
H         82
D         59
Name: count, dtype: Int64

In [37]:
# 16.12h [AGENT] Aggregert interlock-evidens (globalt per 15-min) (REN)

import pandas as pd
import numpy as np

assert "merged_15" in globals(), "Mangler merged_15"

df = merged_15.copy()
df["window_start_15min"] = pd.to_datetime(df["window_start_15min"], utc=True, errors="coerce")
df["fault_loc_key"] = df["fault_loc_key"].astype(str).str.strip()

if "topo_illegal_interlock" in df.columns:
    interlock_flag = df["topo_illegal_interlock"].fillna(0).astype(int).clip(0, 1)
elif "E_interlock" in df.columns:
    interlock_flag = (df["E_interlock"].fillna("none").astype(str).str.lower() == "present").astype(int)
else:
    raise AssertionError("Mangler topo_illegal_interlock eller E_interlock (køyr 16.12c først)")

tmp = df[["window_start_15min", "fault_loc_key"]].copy()
tmp["interlock_flag"] = interlock_flag.values

# max per (ts15, loc) => tel kvar loc maks 1 gong per ts15
per_loc = (
    tmp.groupby(["window_start_15min", "fault_loc_key"], as_index=False)["interlock_flag"]
       .max()
)

counts = (
    per_loc.groupby("window_start_15min", as_index=False)["interlock_flag"]
           .sum()
           .rename(columns={"interlock_flag": "interlock_count"})
)

def _bucket(n: int) -> str:
    if n <= 0: return "none"
    if n <= 2: return "few"
    return "many"

counts["E_interlock_agg"] = counts["interlock_count"].astype(int).map(_bucket)

df = df.drop(columns=["E_interlock_agg", "interlock_count"], errors="ignore")
df = df.merge(counts[["window_start_15min", "E_interlock_agg", "interlock_count"]],
              on="window_start_15min", how="left")

df["E_interlock_agg"] = df["E_interlock_agg"].fillna("none")
df["interlock_count"] = df["interlock_count"].fillna(0).astype(int)

merged_15 = df

n_rows = len(merged_15)
n_windows = merged_15["window_start_15min"].nunique()

# Én rad per vindu (for å ikke telle samme vindu 100 ganger)
w = merged_15[["window_start_15min", "E_interlock_agg", "interlock_count"]].drop_duplicates("window_start_15min")

vc = w["E_interlock_agg"].value_counts()
n_with_any = int((w["interlock_count"] > 0).sum())

print("Global interlock aggregation (per 15-min window):")
print(f"Windows: {len(w):,} | Rows (full universe): {n_rows:,}")
print(f"Windows with ≥1 interlock violation: {n_with_any:,}/{len(w):,} ({n_with_any/len(w):.2%})")
print("Bucket distribution (windows):", vc.to_dict())
print(f"Interlock_count per window: min={int(w['interlock_count'].min())}, max={int(w['interlock_count'].max())}")



Global interlock aggregation (per 15-min window):
Windows: 140,233 | Rows (full universe): 1,542,563
Windows with ≥1 interlock violation: 0/140,233 (0.00%)
Bucket distribution (windows): {'none': 140233}
Interlock_count per window: min=0, max=0


In [38]:
# 16.13 [KODESJEKK] Full-universe sanity: where is evidence, and do events join?

import pandas as pd

assert "merged_15" in globals(), "Mangler merged_15"
df = merged_15

event_cols = [c for c in df.columns if c.startswith("E_") and c != "E_kV_15min"]

meas_present = df["E_kV_15min"].notna()
event_present = df[event_cols].notna().any(axis=1) if len(event_cols) > 0 else pd.Series(False, index=df.index)

any_evidence = meas_present | event_present

by_loc = (
    df.assign(
        meas_present=meas_present,
        event_present=event_present,
        any_evidence=any_evidence
    )
    .groupby("fault_loc_key")
    .agg(
        n_rows=("fault_loc_key", "size"),
        any_evidence_rate=("any_evidence", "mean"),
        meas_rate=("meas_present", "mean"),
        event_rate=("event_present", "mean"),
    )
    .sort_values(["any_evidence_rate", "meas_rate", "event_rate"], ascending=False)
)

print("=== 16.13b [KODESJEKK] ===")
print("Rader:", len(df))
print("Unike fault_loc_key:", df["fault_loc_key"].nunique())
print("Unike vinduer:", df["window_start_15min"].nunique())
print("\n--- Evidens-oversikt ---")
print("Andel rader med måle-evidens (E_kV_15min != NaN):", round(meas_present.mean(), 4))
print("Event-evidenskolonner:", len(event_cols))
print("Andel rader med event-evidens (minst én event-kolonne ikke-NaN):", round(event_present.mean(), 4))
print("Andel rader med NOE evidens (måling eller event):", round(any_evidence.mean(), 4))

print("\n--- Topp 10 fault_loc_key (etter evidens-rate) ---")
display(by_loc.head(10))


=== 16.13b [KODESJEKK] ===
Rader: 1542563
Unike fault_loc_key: 11
Unike vinduer: 140233

--- Evidens-oversikt ---
Andel rader med måle-evidens (E_kV_15min != NaN): 0.2669
Event-evidenskolonner: 5
Andel rader med event-evidens (minst én event-kolonne ikke-NaN): 1.0
Andel rader med NOE evidens (måling eller event): 1.0

--- Topp 10 fault_loc_key (etter evidens-rate) ---


,n_rows,any_evidence_rate,meas_rate,event_rate
fault_loc_key,,,,
A,140233,1.0,0.999993,1.0
D,140233,1.0,0.999993,1.0
C,140233,1.0,0.936085,1.0
B,140233,1.0,0.000000,1.0
E,140233,1.0,0.000000,1.0
F,140233,1.0,0.000000,1.0
G,140233,1.0,0.000000,1.0
H,140233,1.0,0.000000,1.0
IDK2,140233,1.0,0.000000,1.0


In [39]:
# 16.13b [KODESJEKK] Top-10 locations by event-rate and meas-rate (sanity after FULL-universe join)

import pandas as pd

assert "merged_15" in globals(), "Mangler merged_15 (kjør 16.13 først)"

df = merged_15

# Preconditions
assert "fault_loc_key" in df.columns, "merged_15 mangler fault_loc_key"
assert "window_start_15min" in df.columns, "merged_15 mangler window_start_15min"
assert "E_event_any" in df.columns, "merged_15 mangler E_event_any (bruk 16.13-erstatningen)"
assert "E_kV_15min" in df.columns, "merged_15 mangler E_kV_15min (skal komme fra 16.13)"

# Make sure types are reasonable
df["fault_loc_key"] = df["fault_loc_key"].astype(str).str.strip()

meas_present = df["E_kV_15min"].notna()
event_present = df["E_event_any"].fillna(0).astype(int).eq(1)

by_loc = (
    df.assign(meas_present=meas_present, event_present=event_present)
      .groupby("fault_loc_key")
      .agg(
          n_rows=("fault_loc_key", "size"),
          meas_rate=("meas_present", "mean"),
          event_any_rate=("event_present", "mean"),
      )
)

print("=== 16.13c [KODESJEKK] Evidence rates by location ===")
print("Rows:", len(df),
      "| Windows:", df["window_start_15min"].nunique(),
      "| Locations:", df["fault_loc_key"].nunique())
print("Overall meas_rate:", round(meas_present.mean(), 4),
      "| Overall event_any_rate:", round(event_present.mean(), 6))

print("\n--- Top 10 fault_loc_key by meas_rate ---")
display(by_loc.sort_values(["meas_rate", "event_any_rate"], ascending=False).head(10))

print("\n--- Top 10 fault_loc_key by event_any_rate ---")
display(by_loc.sort_values(["event_any_rate", "meas_rate"], ascending=False).head(10))

# Optional: quick check of vocab in E_kV_15min
print("\n--- E_kV_15min value_counts (top) ---")
display(df["E_kV_15min"].value_counts(dropna=False).head(10))


=== 16.13c [KODESJEKK] Evidence rates by location ===
Rows: 1542563 | Windows: 140233 | Locations: 11
Overall meas_rate: 0.2669 | Overall event_any_rate: 0.004281

--- Top 10 fault_loc_key by meas_rate ---


,n_rows,meas_rate,event_any_rate
fault_loc_key,,,
A,140233,0.999993,0.001048
D,140233,0.999993,0.000421
C,140233,0.936085,0.002646
B,140233,0.000000,0.022170
G,140233,0.000000,0.009420
E,140233,0.000000,0.006033
IDK5,140233,0.000000,0.001469
IDK2,140233,0.000000,0.001462
IDK8,140233,0.000000,0.001141



--- Top 10 fault_loc_key by event_any_rate ---


,n_rows,meas_rate,event_any_rate
fault_loc_key,,,
B,140233,0.000000,0.022170
G,140233,0.000000,0.009420
E,140233,0.000000,0.006033
C,140233,0.936085,0.002646
IDK5,140233,0.000000,0.001469
IDK2,140233,0.000000,0.001462
IDK8,140233,0.000000,0.001141
A,140233,0.999993,0.001048
F,140233,0.000000,0.000699



--- E_kV_15min value_counts (top) ---


E_kV_15min
NaN                  1130829
present               367052
absent_or_invalid      29222
unknown                15460
Name: count, dtype: int64

In [40]:
# 16.14 [AGENT] Likelihood specification (REN) + TOPO_LH + SEQUENCE_LH  (SINGLE SOURCE OF TRUTH)
#
# NOTE:
# - These TOPO_LH and SEQUENCE_LH values match the ones previously defined in 16.14b and 16.14c.
# - Keep ONLY this cell; delete/disable 16.14b and 16.14c to avoid accidental overwrites.

import numpy as np

# Hypotheses
HYPOTHESES = ["Normal", "SensorFault", "CommFault"]

# Evidence vocab
E_KV_VALUES       = ["normal", "abnormal", "unknown"]
E_EVENT_VALUES    = ["none", "present"]
E_EVENT_UC_VALUES = ["none", "present"]  # event under comm-failure

# --------------------------------------------------------
# Core 3D likelihood table: P(E_kV, E_event, E_event_under_comm | H)
# (Keep EXACTLY as before)
# --------------------------------------------------------
likelihood = {
    "Normal": {
        ("normal","none","none"):        0.40,
        ("normal","present","none"):     0.04,
        ("normal","present","present"):  0.0001,

        ("abnormal","none","none"):      0.05,
        ("abnormal","present","none"):   0.01,
        ("abnormal","present","present"):0.0001,

        ("unknown","none","none"):       0.45,
        ("unknown","present","none"):    0.05,
        ("unknown","present","present"): 0.0001,
    },

    "SensorFault": {
        ("normal","none","none"):        0.06,
        ("normal","present","none"):     0.05,
        ("normal","present","present"):  0.0001,

        ("abnormal","none","none"):      0.55,
        ("abnormal","present","none"):   0.25,
        ("abnormal","present","present"):0.01,

        ("unknown","none","none"):       0.05,
        ("unknown","present","none"):    0.03,
        ("unknown","present","present"): 0.01,
    },

    "CommFault": {
        ("normal","none","none"):        0.05,
        ("normal","present","none"):     0.05,
        ("normal","present","present"):  0.10,

        ("abnormal","none","none"):      0.05,
        ("abnormal","present","none"):   0.05,
        ("abnormal","present","present"):0.15,

        ("unknown","none","none"):       0.35,
        ("unknown","present","none"):    0.10,
        ("unknown","present","present"): 0.10,
    }
}

# -------------------------
# Normalize likelihoods (safety)
# -------------------------
for h, lh in likelihood.items():
    Z = float(sum(lh.values()))
    assert Z > 0, f"Zero likelihood mass for {h}"
    for k in list(lh.keys()):
        lh[k] = float(lh[k]) / Z

for h in HYPOTHESES:
    total = float(sum(likelihood[h].values()))
    assert abs(total - 1.0) < 1e-9, f"Likelihoods for {h} sum to {total}, not 1.0"

# --------------------------------------------------------
# TOPO_LH: P(E_topo_consistency | H)
# (Use the VALUES that were previously in 16.14b)
# --------------------------------------------------------
TOPO_LH = {
    "Normal": {
        "consistent":   0.90,
        "inconsistent": 0.01,
        "unknown":      0.09,
    },
    "SensorFault": {
        "consistent":   0.20,
        "inconsistent": 0.70,
        "unknown":      0.10,
    },
    "CommFault": {
        "consistent":   0.30,
        "inconsistent": 0.40,
        "unknown":      0.30,
    },
}

for h in HYPOTHESES:
    Z = float(sum(TOPO_LH[h].values()))
    assert abs(Z - 1.0) < 1e-9, f"TOPO_LH for {h} sums to {Z}"

# --------------------------------------------------------
# SEQUENCE_LH: P(E_event_sequence | H)
# --------------------------------------------------------
SEQUENCE_LH = {
    "Normal":      {"none": 0.98, "present": 0.02},
    "SensorFault": {"none": 0.30, "present": 0.70},
    "CommFault":   {"none": 0.15, "present": 0.85},
}

for h in HYPOTHESES:
    Z = float(sum(SEQUENCE_LH[h].values()))
    assert abs(Z - 1.0) < 1e-9, f"SEQUENCE_LH for {h} sums to {Z}"

print("OK: 16.14 likelihood (3D) + TOPO_LH + SEQUENCE_LH defined (single source of truth).")


OK: 16.14 likelihood (3D) + TOPO_LH + SEQUENCE_LH defined (single source of truth).


In [41]:
# 16.14d [AGENT] E_event_sequence per (window_start_15min, fault_loc_key)

import pandas as pd
import numpy as np

assert "merged_15" in globals(), "Mangler merged_15 (køyr 16.12.2 først)"
assert "events_mapped" in globals(), "Mangler events_mapped (skal bli laga i 16.12)"

ev = events_mapped.copy()
ev["window_start_15min"] = pd.to_datetime(ev["window_start_15min"], utc=True, errors="coerce")
ev["fault_loc_key"] = ev["fault_loc_key"].astype(str).str.strip()
ev = ev.dropna(subset=["window_start_15min", "fault_loc_key"]).copy()

seq = (
    ev.groupby(["window_start_15min", "fault_loc_key"], as_index=False)
      .size()
      .rename(columns={"size": "n_events"})
)
seq["E_event_sequence"] = np.where(seq["n_events"] >= 2, "present", "none")
seq = seq[["window_start_15min", "fault_loc_key", "E_event_sequence"]]

df = merged_15.copy()
df["window_start_15min"] = pd.to_datetime(df["window_start_15min"], utc=True, errors="coerce")
df["fault_loc_key"] = df["fault_loc_key"].astype(str).str.strip()

df = df.drop(columns=["E_event_sequence"], errors="ignore")
df = df.merge(seq, on=["window_start_15min", "fault_loc_key"], how="left")
df["E_event_sequence"] = df["E_event_sequence"].fillna("none")
df.loc[~df["E_event_sequence"].isin(["none", "present"]), "E_event_sequence"] = "none"

merged_15 = df

n_rows = len(merged_15)
n_present = int((merged_15["E_event_sequence"] == "present").sum())

# hvor mange unike 15-min vinduer som har minst én lokasjon med "present"
w_present = (
    merged_15.loc[merged_15["E_event_sequence"] == "present", "window_start_15min"]
    .nunique()
)
n_windows = merged_15["window_start_15min"].nunique()

print("Event-sequence evidence (>=2 events within same 15-min window at same fault_loc_key):")
print(f"Rows with E_event_sequence='present': {n_present:,}/{n_rows:,} ({n_present/n_rows:.2%})")
print(f"Windows with ≥1 'present' location:   {w_present:,}/{n_windows:,} ({w_present/max(n_windows,1):.2%})")



Event-sequence evidence (>=2 events within same 15-min window at same fault_loc_key):
Rows with E_event_sequence='present': 3,947/1,542,563 (0.26%)
Windows with ≥1 'present' location:   3,649/140,233 (2.60%)


In [42]:
# 16.14e [CLEAN] Clean state before posterior update

# endrer ikke evidens, endrer ikke data, gjør ingen inferens, påvirker ikke modellstruktur

assert "merged_15" in globals(), "Mangler merged_15"

out = merged_15.copy()

POSTERIOR_COLS = ["P_Normal", "P_SensorFault", "P_CommFault"]

to_drop = [c for c in POSTERIOR_COLS if c in out.columns]
out = out.drop(columns=to_drop, errors="ignore")

# Remove duplicated column names (keep first)
if out.columns.duplicated().any():
    out = out.loc[:, ~out.columns.duplicated()]

merged_15 = out
print("OK: Clean state ready. Run 16.15 → 16.16.")


OK: Clean state ready. Run 16.15 → 16.16.


In [43]:
# 16.15 [AGENT] Posterior update (REN, stateless)

import pandas as pd
import numpy as np

# --- Contract checks ---
assert "merged_15" in globals(), "Mangler merged_15"
assert "likelihood" in globals(), "Mangler likelihood (16.14)"
assert "TOPO_LH" in globals(), "Mangler TOPO_LH (16.14)"
assert "SEQUENCE_LH" in globals(), "Mangler SEQUENCE_LH (16.14)"

REQUIRED_COLS = ["E_kV_15min", "E_event_any", "E_comm_fail_any", "E_topo_consistency"]
for c in REQUIRED_COLS:
    assert c in merged_15.columns, f"merged_15 manglar {c}"

# --- Agent priors ---
AGENT_PRIORS = {"Normal": 0.90, "SensorFault": 0.05, "CommFault": 0.05}
HYPOTHESES = list(AGENT_PRIORS.keys())

# Likelihood for interlock evidence (lokal – OK å halde her)
INTERLOCK_LH = {
    "Normal":      {"none": 1.0, "present": 0.001},
    "SensorFault": {"none": 1.0, "present": 0.70},
    "CommFault":   {"none": 1.0, "present": 0.75},
}
INTERLOCK_AGG_LH = {
    "Normal":      {"none": 1.0, "few": 0.05, "many": 0.001},
    "SensorFault": {"none": 1.0, "few": 0.60, "many": 0.15},
    "CommFault":   {"none": 1.0, "few": 0.80, "many": 0.95},
}

def posterior_update(
    df: pd.DataFrame,
    likelihood_dict: dict,
    topo_lh: dict,
    seq_lh: dict,
    priors: dict = AGENT_PRIORS,
    mask=None,
) -> pd.DataFrame:
    out = df.copy()

    if mask is None:
        mask = pd.Series(True, index=out.index)

    # --------------------------------------------------------
    # 1) Derived / compact evidence (idempotent)
    # --------------------------------------------------------
    out.loc[mask, "E_kV"] = (
        out.loc[mask, "E_kV_15min"]
        .map({
            "present": "normal",
            "absent_or_invalid": "abnormal",
            "normal": "normal",
            "abnormal": "abnormal",
            "unknown": "unknown",
        })
        .fillna("unknown")
    )

    out.loc[mask, "E_event"] = (
        out.loc[mask, "E_event_any"]
        .fillna(0).astype(int)
        .map({1: "present", 0: "none"})
        .fillna("none")
    )

    out["E_comm_fail_any"] = out["E_comm_fail_any"].fillna(0).astype(int).clip(0, 1)

    out.loc[mask, "E_event_under_comm"] = np.where(
        (out.loc[mask, "E_event_any"].fillna(0).astype(int).eq(1))
        & (out.loc[mask, "E_comm_fail_any"].eq(1)),
        "present",
        "none",
    )

    # Ensure sequence exists (default none)
    if "E_event_sequence" not in out.columns:
        out["E_event_sequence"] = "none"
    out["E_event_sequence"] = out["E_event_sequence"].fillna("none")
    out.loc[~out["E_event_sequence"].isin(["none", "present"]), "E_event_sequence"] = "none"

    # Ensure interlock vars exist
    if "E_interlock" not in out.columns:
        out["E_interlock"] = "none"
    out["E_interlock"] = out["E_interlock"].fillna("none")
    out.loc[~out["E_interlock"].isin(["none", "present"]), "E_interlock"] = "none"

    if "E_interlock_agg" not in out.columns:
        out["E_interlock_agg"] = "none"
    out["E_interlock_agg"] = out["E_interlock_agg"].fillna("none")
    out.loc[~out["E_interlock_agg"].isin(["none", "few", "many"]), "E_interlock_agg"] = "none"

    # --------------------------------------------------------
    # 2) Posterior per row (radvis, stateless)
    # --------------------------------------------------------
    def posterior_row(r):
        etopo = r.get("E_topo_consistency", "unknown")
        if etopo not in ("consistent", "inconsistent", "unknown"):
            etopo = "unknown"

        eint = r.get("E_interlock", "none")
        if eint not in ("none", "present"):
            eint = "none"

        eagg = r.get("E_interlock_agg", "none")
        if eagg not in ("none", "few", "many"):
            eagg = "none"

        eseq = r.get("E_event_sequence", "none")
        if eseq not in ("none", "present"):
            eseq = "none"

        key = (r["E_kV"], r["E_event"], r["E_event_under_comm"])

        unnorm = {}
        for h in HYPOTHESES:
            unnorm[h] = (
                likelihood_dict[h][key]
                * topo_lh[h].get(etopo, topo_lh[h]["unknown"])
                * INTERLOCK_LH[h].get(eint, 1.0)
                * INTERLOCK_AGG_LH[h].get(eagg, 1.0)
                * seq_lh[h].get(eseq, 1.0)   # <-- bruker global SEQUENCE_LH
                * priors[h]
            )

        Z = float(sum(unnorm.values()))
        if Z <= 0 or not np.isfinite(Z):
            return pd.Series({f"P_{h}": 1.0 / len(HYPOTHESES) for h in HYPOTHESES})

        return pd.Series({f"P_{h}": unnorm[h] / Z for h in HYPOTHESES})

    post = out.loc[mask].apply(posterior_row, axis=1)

    for h in HYPOTHESES:
        out.loc[mask, f"P_{h}"] = post[f"P_{h}"]

    return out

# ------------------------------------------------------------
# 3) Run posterior update (batch or runtime slice)
# ------------------------------------------------------------
RUNTIME_MASK = globals().get("RUNTIME_MASK", None)

merged_15 = posterior_update(
    merged_15,
    likelihood_dict=likelihood,
    topo_lh=TOPO_LH,
    seq_lh=SEQUENCE_LH,
    priors=AGENT_PRIORS,
    mask=RUNTIME_MASK,
)

# ------------------------------------------------------------
# 4) Sanity check
# ------------------------------------------------------------
mask_chk = RUNTIME_MASK if RUNTIME_MASK is not None else pd.Series(True, index=merged_15.index)
s = merged_15.loc[mask_chk, ["P_Normal", "P_SensorFault", "P_CommFault"]].sum(axis=1)

print("OK: 16.15 posterior ferdig.")
print("Posterior sanity (min/max sum):", float(s.min()), float(s.max()))


OK: 16.15 posterior ferdig.
Posterior sanity (min/max sum): 1.0 1.0


In [44]:
# ============================================================
# 16.15b [KODESJEKK] Likelihood key-space completeness (valid keys only)
# ============================================================

from itertools import product

E_KV_DOMAIN = ["normal", "abnormal", "unknown"]
E_EVENT_DOMAIN = ["present", "none"]
E_EVENT_UNDER_COMM_DOMAIN = ["present", "none"]

# Only valid combinations:
# If E_event_under_comm == "present", then E_event must be "present"
EXPECTED_KEYS = set()
for kv, ev, uc in product(E_KV_DOMAIN, E_EVENT_DOMAIN, E_EVENT_UNDER_COMM_DOMAIN):
    if uc == "present" and ev != "present":
        continue
    EXPECTED_KEYS.add((kv, ev, uc))

print("Valid keys expected:", len(EXPECTED_KEYS))

for h, lh in likelihood.items():
    lh_keys = set(lh.keys())
    missing = EXPECTED_KEYS - lh_keys
    extra = lh_keys - EXPECTED_KEYS

    print(f"\nHypothesis: {h}")
    print(f"  Keys expected : {len(EXPECTED_KEYS)}")
    print(f"  Keys defined  : {len(lh_keys)}")

    if missing:
        print("  MISSING keys:")
        for k in sorted(missing):
            print("   ", k)
    else:
        print("  OK: No missing keys")

    if extra:
        print("  EXTRA keys (unused):")
        for k in sorted(extra):
            print("   ", k)
    else:
        print("  OK: No extra keys")

    assert not missing, f"Likelihood for {h} manglar {len(missing)} keys"

print("\nOK: Likelihood key-space is complete for all hypotheses (valid keys).")


Valid keys expected: 9

Hypothesis: Normal
  Keys expected : 9
  Keys defined  : 9
  OK: No missing keys
  OK: No extra keys

Hypothesis: SensorFault
  Keys expected : 9
  Keys defined  : 9
  OK: No missing keys
  OK: No extra keys

Hypothesis: CommFault
  Keys expected : 9
  Keys defined  : 9
  OK: No missing keys
  OK: No extra keys

OK: Likelihood key-space is complete for all hypotheses (valid keys).


In [45]:
print("merged_15 nan-string:", int((out["fault_loc_key"].astype("string") == "nan").sum()))
print("merged_15 NA:", int(out["fault_loc_key"].isna().sum()))


merged_15 nan-string: 0
merged_15 NA: 0


In [46]:
# %% 16.16 [AGENT] Utilities + Expected Utility + Decision

import pandas as pd
import numpy as np

assert "merged_15" in globals(), "Mangler merged_15 (køyr 16.15 først)"
assert "ft" in globals(), "Mangler ft (køyr 0.x først)"

REQUIRED_POST_COLS = ["P_Normal", "P_SensorFault", "P_CommFault"]
for c in REQUIRED_POST_COLS:
    assert c in merged_15.columns, f"Mangler {c} (køyr 16.15 først)"

# ------------------------------------------------------------
# 0) Build utilities_by_loc once (from global ft)
# ------------------------------------------------------------
UTIL_COLS = [
    "SensorIntegrity_Utility",
    "StatusConsistency_Utility",
    "CyberSecurity_Utility",
    "DataIntegrity_Utility",
    "DigitalTwinImpact_Utility",
]

def build_utils_by_loc_from_ft(ft_df: pd.DataFrame) -> pd.DataFrame:
    missing = [c for c in (["fault_loc_key"] + UTIL_COLS) if c not in ft_df.columns]
    assert not missing, f"ft manglar utility-kolonner: {missing}"

    tmp = ft_df[["fault_loc_key"] + UTIL_COLS].copy()
    tmp = tmp[tmp["fault_loc_key"].notna()].copy()
    tmp["fault_loc_key"] = tmp["fault_loc_key"].astype(str).str.strip()

    for c in UTIL_COLS:
        tmp[c] = pd.to_numeric(tmp[c], errors="coerce")

    utils_by_loc = (
        tmp.groupby("fault_loc_key")[UTIL_COLS]
           .min()
           .add_suffix("_worst")
           .reset_index()
    )

    counts = tmp.groupby("fault_loc_key").size().rename("n_rows").reset_index()
    utils_by_loc = counts.merge(utils_by_loc, on="fault_loc_key", how="left")

    return utils_by_loc

# Lazy global cache (bygg berre éin gong)
if "UTILS_BY_LOC" not in globals():
    UTILS_BY_LOC = build_utils_by_loc_from_ft(ft)

# ------------------------------------------------------------
# 1) Decision update
# ------------------------------------------------------------
def decision_update(df: pd.DataFrame, utils_by_loc: pd.DataFrame, mask=None) -> pd.DataFrame:
    out = df.copy()

    # mask -> posisjonell bool-array
    if mask is None:
        mask_arr = np.ones(len(out), dtype=bool)
    else:
        mask_arr = np.asarray(mask, dtype=bool)
        assert mask_arr.shape[0] == len(out), f"Mask-lengde {mask_arr.shape[0]} != df-lengde {len(out)}"

    # hygiene keys
    out["fault_loc_key"] = out["fault_loc_key"].astype("string").str.strip()


    # dropp gamle worst-kolonner før merge (collision-safe)
    worst_cols = [c for c in out.columns if c.endswith("_worst")] + ["n_rows"]
    out = out.drop(columns=worst_cols, errors="ignore")

    # merge utilities (m:1)
    out = out.merge(utils_by_loc, on="fault_loc_key", how="left", validate="m:1")

    NEED_UTIL = [
        "SensorIntegrity_Utility_worst",
        "StatusConsistency_Utility_worst",
        "CyberSecurity_Utility_worst",
        "DataIntegrity_Utility_worst",
        "DigitalTwinImpact_Utility_worst",
    ]


    bad = out.loc[mask_arr, ["fault_loc_key"] + NEED_UTIL].copy()
    bad_rows = bad[bad[NEED_UTIL].isna().any(axis=1)]

    print("Rows in mask with any missing utility:", len(bad_rows))
    print("Distinct fault_loc_key with missing utility:", bad_rows["fault_loc_key"].nunique())
    display(
        bad_rows["fault_loc_key"].value_counts().head(20)
    )
    display(bad_rows.head(20))

    for c in NEED_UTIL:
        assert c in out.columns, f"Mangler {c} etter merge (sjekk UTILS_BY_LOC)"

    # slice (posisjonell)
    U_SI = out.loc[mask_arr, "SensorIntegrity_Utility_worst"]
    U_SC = out.loc[mask_arr, "StatusConsistency_Utility_worst"]
    U_CS = out.loc[mask_arr, "CyberSecurity_Utility_worst"]
    U_DI = out.loc[mask_arr, "DataIntegrity_Utility_worst"]
    U_DT = out.loc[mask_arr, "DigitalTwinImpact_Utility_worst"]

    COST = {"Ignore": 0.00, "Inspect": -0.10, "Isolate": -0.15}

    def mean2(a, b):
        return np.nanmean(np.vstack([a.to_numpy(), b.to_numpy()]), axis=0)

    # --------------------------------------------------
    # 2) Utilities U(action, hypothesis)
    #   NB: Beheld din eksisterande struktur (kan strammast seinare)
    # --------------------------------------------------
    out.loc[mask_arr, "U_Ignore_Normal"]      = COST["Ignore"]
    out.loc[mask_arr, "U_Ignore_SensorFault"] = COST["Ignore"] + mean2(U_SI, U_DI) * (-1)
    out.loc[mask_arr, "U_Ignore_CommFault"]   = COST["Ignore"] + mean2(U_CS, U_DI) * (-1)

    out.loc[mask_arr, "U_Inspect_Normal"]      = COST["Inspect"] - 0.02
    out.loc[mask_arr, "U_Inspect_SensorFault"] = COST["Inspect"] + mean2(U_SI, U_DI)
    out.loc[mask_arr, "U_Inspect_CommFault"]   = COST["Inspect"] + (U_DI.to_numpy() * 0.2)

    out.loc[mask_arr, "U_Isolate_Normal"]      = COST["Isolate"] - 0.10
    out.loc[mask_arr, "U_Isolate_SensorFault"] = COST["Isolate"] + 0.4 * mean2(U_SC, U_DI)
    out.loc[mask_arr, "U_Isolate_CommFault"]   = COST["Isolate"] + 1.2 * mean2(U_CS, U_DT)

    # --------------------------------------------------
    # 3) Expected utility + decision
    # --------------------------------------------------
    out.loc[mask_arr, "EU_Ignore"] = (
        out.loc[mask_arr, "P_Normal"]      * out.loc[mask_arr, "U_Ignore_Normal"] +
        out.loc[mask_arr, "P_SensorFault"] * out.loc[mask_arr, "U_Ignore_SensorFault"] +
        out.loc[mask_arr, "P_CommFault"]   * out.loc[mask_arr, "U_Ignore_CommFault"]
    )

    out.loc[mask_arr, "EU_Inspect"] = (
        out.loc[mask_arr, "P_Normal"]      * out.loc[mask_arr, "U_Inspect_Normal"] +
        out.loc[mask_arr, "P_SensorFault"] * out.loc[mask_arr, "U_Inspect_SensorFault"] +
        out.loc[mask_arr, "P_CommFault"]   * out.loc[mask_arr, "U_Inspect_CommFault"]
    )

    out.loc[mask_arr, "EU_Isolate"] = (
        out.loc[mask_arr, "P_Normal"]      * out.loc[mask_arr, "U_Isolate_Normal"] +
        out.loc[mask_arr, "P_SensorFault"] * out.loc[mask_arr, "U_Isolate_SensorFault"] +
        out.loc[mask_arr, "P_CommFault"]   * out.loc[mask_arr, "U_Isolate_CommFault"]
    )

    out.loc[mask_arr, "Decision"] = (
        out.loc[mask_arr, ["EU_Ignore", "EU_Inspect", "EU_Isolate"]]
          .idxmax(axis=1)
          .str.replace("EU_", "", regex=False)
    )

    assert set(out.loc[mask_arr, "Decision"].unique()) <= {"Ignore", "Inspect", "Isolate"}
    return out

# ------------------------------------------------------------
# 2) Run decision update (batch eller slice via RUNTIME_MASK)
# ------------------------------------------------------------
RUNTIME_MASK = globals().get("RUNTIME_MASK", None)

merged_15 = decision_update(
    merged_15,
    utils_by_loc=UTILS_BY_LOC,
    mask=RUNTIME_MASK,
)

print("OK: 16.16 ferdig.")

mask_s = np.ones(len(merged_15), dtype=bool) if RUNTIME_MASK is None else np.asarray(RUNTIME_MASK, dtype=bool)

print("\nDecision distribution:")
display(merged_15.loc[mask_s, "Decision"].value_counts())

print("\nMean posterior by Decision:")
display(
    merged_15.loc[mask_s]
    .groupby("Decision")[["P_Normal", "P_SensorFault", "P_CommFault"]]
    .mean()
)


Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst


OK: 16.16 ferdig.

Decision distribution:


Decision
Ignore     1509185
Inspect      29713
Isolate       3665
Name: count, dtype: int64


Mean posterior by Decision:


,P_Normal,P_SensorFault,P_CommFault
Decision,,,
Ignore,0.981873,0.001688,0.016439
Inspect,0.071538,0.900112,0.028350
Isolate,0.055395,0.071904,0.872701


In [47]:
# 16.xx [DIAG] Utility matrix actually used by the agent (from 16.16 columns)

import numpy as np
import pandas as pd

assert "merged_15" in globals(), "Mangler merged_15"
need = [
    "fault_loc_key",
    "U_Ignore_Normal","U_Ignore_SensorFault","U_Ignore_CommFault",
    "U_Inspect_Normal","U_Inspect_SensorFault","U_Inspect_CommFault",
    "U_Isolate_Normal","U_Isolate_SensorFault","U_Isolate_CommFault",
]
for c in need:
    assert c in merged_15.columns, f"Mangler {c} (køyr 16.16 først)"

mask_s = np.ones(len(merged_15), dtype=bool) if globals().get("RUNTIME_MASK", None) is None \
         else np.asarray(RUNTIME_MASK, dtype=bool)

# --------- A) one location (pick one you care about) ----------
loc = "A"   # <-- endre her ved behov

one = merged_15.loc[mask_s & (merged_15["fault_loc_key"] == loc), need].head(1)
assert len(one) == 1, f"Fant ingen rader for loc={loc} i mask"

U_one = pd.DataFrame(
    {
        "Normal":      [float(one["U_Ignore_Normal"]),  float(one["U_Inspect_Normal"]),  float(one["U_Isolate_Normal"])],
        "SensorFault": [float(one["U_Ignore_SensorFault"]), float(one["U_Inspect_SensorFault"]), float(one["U_Isolate_SensorFault"])],
        "CommFault":   [float(one["U_Ignore_CommFault"]),   float(one["U_Inspect_CommFault"]),   float(one["U_Isolate_CommFault"])],
    },
    index=["Ignore","Inspect","Isolate"]
)
print(f"\nUtility matrix used by agent (example row) for fault_loc_key={loc}:")
display(U_one)

# --------- B) per-location utility summary (should be constant within loc) ----------
U_cols = [c for c in need if c.startswith("U_")]

# take first row per loc (utilities should not vary within a loc)
U_by_loc = (
    merged_15.loc[mask_s, ["fault_loc_key"] + U_cols]
    .groupby("fault_loc_key", as_index=False)
    .first()
    .sort_values("fault_loc_key")
)

print("\nPer-location utility matrix (one row per loc, as used by agent):")
display(U_by_loc)



Utility matrix used by agent (example row) for fault_loc_key=A:


C:\Users\jh10341\AppData\Local\Temp\ipykernel_23808\3615401714.py:27: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  "Normal":      [float(one["U_Ignore_Normal"]),  float(one["U_Inspect_Normal"]),  float(one["U_Isolate_Normal"])],
C:\Users\jh10341\AppData\Local\Temp\ipykernel_23808\3615401714.py:28: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  "SensorFault": [float(one["U_Ignore_SensorFault"]), float(one["U_Inspect_SensorFault"]), float(one["U_Isolate_SensorFault"])],
C:\Users\jh10341\AppData\Local\Temp\ipykernel_23808\3615401714.py:29: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  "CommFault":   [float(one["U_Ignore_CommFault"]),   float(one["U_Inspect_CommFault"]),   float(one["U_Isolate_Comm

,Normal,SensorFault,CommFault
Ignore,0.00,-0.60,-0.50
Inspect,-0.12,0.50,0.04
Isolate,-0.25,0.09,0.21



Per-location utility matrix (one row per loc, as used by agent):


,fault_loc_key,U_Ignore_Normal,U_Ignore_SensorFault,U_Ignore_CommFault,U_Inspect_Normal,U_Inspect_SensorFault,U_Inspect_CommFault,U_Isolate_Normal,U_Isolate_SensorFault,U_Isolate_CommFault
0,A,0.0,-0.6,-0.5,-0.12,0.5,0.04,-0.25,0.09,0.21
1,B,0.0,-0.3,-0.3,-0.12,0.2,-0.04,-0.25,-0.03,0.21
2,C,0.0,-0.3,-0.3,-0.12,0.2,-0.04,-0.25,-0.03,0.21
3,D,0.0,-0.3,-0.4,-0.12,0.2,-0.04,-0.25,-0.03,0.33
4,E,0.0,-0.3,-0.3,-0.12,0.2,-0.04,-0.25,-0.03,0.21
5,F,0.0,-0.3,-0.3,-0.12,0.2,-0.04,-0.25,-0.03,0.21
6,G,0.0,-0.3,-0.3,-0.12,0.2,-0.04,-0.25,-0.03,0.15
7,H,0.0,-0.3,-0.3,-0.12,0.2,-0.04,-0.25,-0.03,0.09
8,IDK2,0.0,-0.3,-0.3,-0.12,0.2,-0.04,-0.25,-0.03,0.33
9,IDK5,0.0,-0.4,-0.3,-0.12,0.3,-0.04,-0.25,-0.03,0.33


In [48]:
# 16.xx [DIAG] Verify EU vs Decision (stateless)

sample = merged_15.sample(5, random_state=1)

for _, r in sample.iterrows():
    print("\nLoc:", r["fault_loc_key"])

    print("Posterior:",
          r["P_Normal"],
          r["P_SensorFault"],
          r["P_CommFault"])

    print("EU_Ignore :", r["EU_Ignore"])
    print("EU_Inspect:", r["EU_Inspect"])
    print("EU_Isolate:", r["EU_Isolate"])

    print("Chosen decision:", r["Decision"])



Loc: G
Posterior: 0.9764369134120507 0.002030241691871854 0.021532844896077475
EU_Ignore : -0.007068925976384798
EU_Inspect: -0.11762769506691483
EU_Isolate: -0.24094020886935724
Chosen decision: Ignore

Loc: D
Posterior: 0.9990850298360126 0.000560879760308056 0.00035409040367947954
EU_Ignore : -0.0003099000895642086
EU_Inspect: -0.11979219124440707
EU_Isolate: -0.24967123401859817
Chosen decision: Ignore

Loc: B
Posterior: 0.9764369134120507 0.002030241691871854 0.021532844896077475
EU_Ignore : -0.007068925976384798
EU_Inspect: -0.11762769506691483
EU_Isolate: -0.23964823817559258
Chosen decision: Ignore

Loc: H
Posterior: 0.9764369134120507 0.002030241691871854 0.021532844896077475
EU_Ignore : -0.007068925976384798
EU_Inspect: -0.11762769506691483
EU_Isolate: -0.24223217956312187
Chosen decision: Ignore

Loc: G
Posterior: 0.9764369134120507 0.002030241691871854 0.021532844896077475
EU_Ignore : -0.007068925976384798
EU_Inspect: -0.11762769506691483
EU_Isolate: -0.24094020886935724
C

In [49]:
# 16.17 [KODESJEKK] Decision statistics per fault_loc_key


import pandas as pd

assert "merged_15" in globals(), "Mangler merged_15"
df = merged_15.copy()

assert "fault_loc_key" in df.columns, "Mangler fault_loc_key (kjør 16.12)"
assert "Decision" in df.columns, "Mangler Decision (kjør 16.16)"

totals = df.groupby("fault_loc_key").size().rename("n_rows").reset_index()

counts = (
    df.pivot_table(
        index="fault_loc_key",
        columns="Decision",
        values="window_start_15min",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

for col in ["Ignore", "Inspect", "Isolate"]:
    if col not in counts.columns:
        counts[col] = 0

shares = counts.copy()
shares["n_rows"] = shares[["Ignore", "Inspect", "Isolate"]].sum(axis=1)
shares["Ignore_share"]  = shares["Ignore"]  / shares["n_rows"]
shares["Inspect_share"] = shares["Inspect"] / shares["n_rows"]
shares["Isolate_share"] = shares["Isolate"] / shares["n_rows"]

stats = (
    totals.merge(counts, on="fault_loc_key", how="left")
          .merge(shares[["fault_loc_key", "Ignore_share", "Inspect_share", "Isolate_share"]],
                 on="fault_loc_key", how="left")
          .sort_values("fault_loc_key")
)

print("Decision statistics per fault_loc_key:")
display(stats)

print("\nGlobal decision distribution:")
global_counts = df["Decision"].value_counts().reindex(["Ignore", "Inspect", "Isolate"], fill_value=0)
global_shares = (global_counts / global_counts.sum()).rename(lambda x: f"{x}_share")
display(pd.concat([global_counts, global_shares]))




Decision statistics per fault_loc_key:


,fault_loc_key,n_rows,Ignore,Inspect,Isolate,Ignore_share,Inspect_share,Isolate_share
0,A,140233,119684,20548,1,0.853465,0.146528,0.000007
1,B,140233,137987,0,2246,0.983984,0.000000,0.016016
2,C,140233,131131,9094,8,0.935094,0.064849,0.000057
3,D,140233,140157,71,5,0.999458,0.000506,0.000036
4,E,140233,139615,0,618,0.995593,0.000000,0.004407
5,F,140233,140222,0,11,0.999922,0.000000,0.000078
6,G,140233,139876,0,357,0.997454,0.000000,0.002546
7,H,140233,140228,0,5,0.999964,0.000000,0.000036
8,IDK2,140233,140076,0,157,0.998880,0.000000,0.001120
9,IDK5,140233,140089,0,144,0.998973,0.000000,0.001027



Global decision distribution:


Decision
Ignore           1.509185e+06
Inspect          2.971300e+04
Isolate          3.665000e+03
Ignore_share     9.783620e-01
Inspect_share    1.926210e-02
Isolate_share    2.375916e-03
Name: count, dtype: float64

In [50]:
# 16.18 [KODESJEKK] Monthly decision statistics (fault_loc_key × month)


import pandas as pd

assert "merged_15" in globals(), "Mangler merged_15"
df = merged_15.copy()

assert "Decision" in df.columns, "Mangler Decision (kjør 16.16)"
assert "window_start_15min" in df.columns, "Mangler window_start_15min"
assert "fault_loc_key" in df.columns, "Mangler fault_loc_key (kjør 16.12)"

df["month"] = pd.to_datetime(df["window_start_15min"], utc=True).dt.to_period("M").astype(str)

counts_m = (
    df.pivot_table(
        index=["fault_loc_key", "month"],
        columns="Decision",
        values="window_start_15min",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

for col in ["Ignore", "Inspect", "Isolate"]:
    if col not in counts_m.columns:
        counts_m[col] = 0

counts_m["n_rows"] = counts_m[["Ignore", "Inspect", "Isolate"]].sum(axis=1)

counts_m["Ignore_share"]  = counts_m["Ignore"]  / counts_m["n_rows"]
counts_m["Inspect_share"] = counts_m["Inspect"] / counts_m["n_rows"]
counts_m["Isolate_share"] = counts_m["Isolate"] / counts_m["n_rows"]

monthly_stats = counts_m[[
    "fault_loc_key", "month", "n_rows",
    "Ignore", "Inspect", "Isolate",
    "Ignore_share", "Inspect_share", "Isolate_share"
]].sort_values(["fault_loc_key", "month"])

print("Monthly decision statistics (fault_loc_key × month):")
display(monthly_stats)

print("\nOverall monthly decision distribution (all fault_loc_key):")
overall = (
    df.pivot_table(
        index="month",
        columns="Decision",
        values="window_start_15min",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

for col in ["Ignore", "Inspect", "Isolate"]:
    if col not in overall.columns:
        overall[col] = 0

overall["n_rows"] = overall[["Ignore", "Inspect", "Isolate"]].sum(axis=1)
overall["Ignore_share"]  = overall["Ignore"]  / overall["n_rows"]
overall["Inspect_share"] = overall["Inspect"] / overall["n_rows"]
overall["Isolate_share"] = overall["Isolate"] / overall["n_rows"]

display(overall.sort_values("month"))



C:\Users\jh10341\AppData\Local\Temp\ipykernel_23808\929330643.py:13: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["month"] = pd.to_datetime(df["window_start_15min"], utc=True).dt.to_period("M").astype(str)


Monthly decision statistics (fault_loc_key × month):


Decision,fault_loc_key,month,n_rows,Ignore,Inspect,Isolate,Ignore_share,Inspect_share,Isolate_share
0,A,2018-01,2972,7,2965,0,0.002355,0.997645,0.000000
1,A,2018-02,2688,11,2677,0,0.004092,0.995908,0.000000
2,A,2018-03,2976,0,2976,0,0.000000,1.000000,0.000000
3,A,2018-04,2880,9,2871,0,0.003125,0.996875,0.000000
4,A,2018-05,2976,4,2972,0,0.001344,0.998656,0.000000
...,...,...,...,...,...,...,...,...,...
523,IDK8,2021-08,2976,2975,0,1,0.999664,0.000000,0.000336
524,IDK8,2021-09,2880,2880,0,0,1.000000,0.000000,0.000000
525,IDK8,2021-10,2976,2976,0,0,1.000000,0.000000,0.000000
526,IDK8,2021-11,2880,2880,0,0,1.000000,0.000000,0.000000



Overall monthly decision distribution (all fault_loc_key):


Decision,month,Ignore,Inspect,Isolate,n_rows,Ignore_share,Inspect_share,Isolate_share
0,2018-01,29687,2965,40,32692,0.908081,0.090695,0.001224
1,2018-02,26832,2678,58,29568,0.907468,0.090571,0.001962
2,2018-03,29702,2976,58,32736,0.907319,0.090909,0.001772
3,2018-04,28639,2976,65,31680,0.904009,0.093939,0.002052
4,2018-05,29716,2975,45,32736,0.907747,0.090879,0.001375
5,2018-06,28218,2918,544,31680,0.890720,0.092109,0.017172
6,2018-07,29655,3026,55,32736,0.905883,0.092436,0.001680
7,2018-08,32167,140,429,32736,0.982619,0.004277,0.013105
8,2018-09,31400,180,100,31680,0.991162,0.005682,0.003157
9,2018-10,32654,7,75,32736,0.997495,0.000214,0.002291


In [51]:

# 16.19 [KODESJEKK] Location labels (NON-MUTATING)


# Create a reporting-only view
merged_15_view = merged_15.copy()

merged_15_view["loc_label"] = (
    merged_15_view["fault_loc_key"]
    .astype(str)
    .str.replace("_", "-", regex=False)
)

# Example lightweight inspection (safe)
merged_15_view[["fault_loc_key", "loc_label"]].drop_duplicates().head()


,fault_loc_key,loc_label
0,A,A
1,B,B
2,C,C
3,D,D
4,E,E


In [52]:
# 16.20 [KODESJEKK] Agent-kontrakt: konsistens, duplikater, posterior-sum


#   Sjekker at:
#     - merged_15 finnes og har forventede nøkler/kolonner
#     - ingen duplikater på (window_start_15min, fault_loc_key)
#     - posterior-kolonner finnes og summerer til ~1
#     - EU-kolonner finnes og Decision er gyldig
#     - utilities ikke mangler for lokasjoner (valgfritt sanity)


import pandas as pd
import numpy as np

assert "merged_15" in globals(), "Mangler merged_15"
df = merged_15.copy()

# 1) Nøkkel-kolonner
must_have = ["window_start_15min", "fault_loc_key", "E_kV_15min"]
missing = [c for c in must_have if c not in df.columns]
assert not missing, f"merged_15 mangler: {missing}. Har: {list(df.columns)[:50]}"

# 2) Duplikater i universet
dup_mask = df.duplicated(["window_start_15min", "fault_loc_key"])
n_dups = int(dup_mask.sum())
if n_dups > 0:
    print(f"ADVARSEL: {n_dups} duplikate rader i (window_start_15min, fault_loc_key). Viser 10 eksempel:")
    display(df.loc[dup_mask, ["window_start_15min", "fault_loc_key"]].head(10))
else:
    print("OK: Ingen duplikater i (window_start_15min, fault_loc_key).")

# 3) Posterior-sjekk
post_cols = ["P_Normal", "P_SensorFault", "P_CommFault"]
missing_post = [c for c in post_cols if c not in df.columns]
assert not missing_post, f"Mangler posterior-kolonner: {missing_post}. Kjør 16.15."

post_sum = df[post_cols].sum(axis=1)
max_abs_err = float((post_sum - 1.0).abs().max())
mean_abs_err = float((post_sum - 1.0).abs().mean())

print(f"Posterior-sum | max abs avvik: {max_abs_err:.3e}, mean abs avvik: {mean_abs_err:.3e}")

# vis rader med størst avvik hvis noe er rart
bad = df.loc[(post_sum - 1.0).abs() > 1e-8, ["window_start_15min", "fault_loc_key"] + post_cols].copy()
if len(bad) > 0:
    print(f"ADVARSEL: {len(bad)} rader med posterior-sum-avvik > 1e-8 (viser 10):")
    display(bad.head(10))
else:
    print("OK: Posterior summerer til 1 (innen toleranse).")

# 4) EU/Decision-sjekk
eu_cols = ["EU_Ignore", "EU_Inspect", "EU_Isolate"]
missing_eu = [c for c in eu_cols if c not in df.columns]
assert not missing_eu, f"Mangler EU-kolonner: {missing_eu}. Kjør 16.16."

assert "Decision" in df.columns, "Mangler Decision. Kjør 16.16."
valid_decisions = {"Ignore", "Inspect", "Isolate"}
bad_dec = df.loc[~df["Decision"].isin(valid_decisions), ["Decision"]]
assert len(bad_dec) == 0, f"Ugyldige Decision-verdier funnet: {bad_dec['Decision'].unique()}"

print("OK: EU-kolonner og Decision finnes og er gyldige.")
print("\nDecision fordeling:")
display(df["Decision"].value_counts())

# 5) Utilities coverage (valgfri sanity)
# Hvis utilities ble merget inn i 16.16, skal disse vanligvis finnes
util_check_cols = [
    "SensorIntegrity_Utility_worst",
    "StatusConsistency_Utility_worst",
    "CyberSecurity_Utility_worst",
    "DataIntegrity_Utility_worst",
    "DigitalTwinImpact_Utility_worst",
]
have_utils = [c for c in util_check_cols if c in df.columns]
if len(have_utils) == len(util_check_cols):
    miss_utils_rows = df[have_utils].isna().any(axis=1).mean()
    print(f"\nUtilities coverage: andel rader med minst én NaN utility = {miss_utils_rows:.3%}")
else:
    print("\nMerk: Fant ikke alle utility-kolonner i merged_15 (ok hvis du ikke beholder dem i df).")


OK: Ingen duplikater i (window_start_15min, fault_loc_key).
Posterior-sum | max abs avvik: 0.000e+00, mean abs avvik: 0.000e+00
OK: Posterior summerer til 1 (innen toleranse).
OK: EU-kolonner og Decision finnes og er gyldige.

Decision fordeling:


Decision
Ignore     1509185
Inspect      29713
Isolate       3665
Name: count, dtype: int64


Utilities coverage: andel rader med minst én NaN utility = 0.000%


In [53]:
# 16.20x [KODESJEKK]

baseline_view = (
    merged_15.loc[
        merged_15["fault_loc_key"].isin(["C"]),
        ["window_start_15min","fault_loc_key",
         "P_Normal","P_SensorFault","P_CommFault","Decision"]
    ]
    .copy()
)

display(baseline_view.head(10))


,window_start_15min,fault_loc_key,P_Normal,P_SensorFault,P_CommFault,Decision
2,2018-01-01 01:00:00+00:00,C,0.976437,0.00203,0.021533,Ignore
13,2018-01-01 01:15:00+00:00,C,0.976437,0.00203,0.021533,Ignore
24,2018-01-01 01:30:00+00:00,C,0.976437,0.00203,0.021533,Ignore
35,2018-01-01 01:45:00+00:00,C,0.976437,0.00203,0.021533,Ignore
46,2018-01-01 02:00:00+00:00,C,0.976437,0.00203,0.021533,Ignore
57,2018-01-01 02:15:00+00:00,C,0.976437,0.00203,0.021533,Ignore
68,2018-01-01 02:30:00+00:00,C,0.976437,0.00203,0.021533,Ignore
79,2018-01-01 02:45:00+00:00,C,0.976437,0.00203,0.021533,Ignore
90,2018-01-01 03:00:00+00:00,C,0.976437,0.00203,0.021533,Ignore
101,2018-01-01 03:15:00+00:00,C,0.976437,0.00203,0.021533,Ignore


In [54]:
merged_15["P_SensorFault"].quantile([0.5,0.9,0.99,0.999])
merged_15.groupby("fault_loc_key")["Decision"].value_counts(normalize=True).head(60)


fault_loc_key  Decision
A              Ignore      0.853465
               Inspect     0.146528
               Isolate     0.000007
B              Ignore      0.983984
               Isolate     0.016016
C              Ignore      0.935094
               Inspect     0.064849
               Isolate     0.000057
D              Ignore      0.999458
               Inspect     0.000506
               Isolate     0.000036
E              Ignore      0.995593
               Isolate     0.004407
F              Ignore      0.999922
               Isolate     0.000078
G              Ignore      0.997454
               Isolate     0.002546
H              Ignore      0.999964
               Isolate     0.000036
IDK2           Ignore      0.998880
               Isolate     0.001120
IDK5           Ignore      0.998973
               Isolate     0.001027
IDK8           Ignore      0.999194
               Isolate     0.000806
Name: proportion, dtype: float64

In [55]:
# %% 18.event_sequence_ONLY [EVAL] Single-evidence injection test (E_event_sequence only)

import pandas as pd
import numpy as np

LOCS = ["A", "C"]
INJ_START = pd.to_datetime("2020-01-01 00:00:00+00:00", utc=True)
INJ_END   = pd.to_datetime("2020-01-01 23:45:00+00:00", utc=True)

TMIN = INJ_START - pd.Timedelta(hours=6)
TMAX = INJ_END   + pd.Timedelta(hours=6)

POST_COLS = ["P_Normal","P_SensorFault","P_CommFault"]
DEC_COL = "Decision"
KEYS = ["window_start_15min","fault_loc_key"]

def _print(title):
    print("\n" + "="*96)
    print(title)
    print("="*96)

def _subset(df):
    out = df.copy()
    out["window_start_15min"] = pd.to_datetime(out["window_start_15min"], utc=True, errors="coerce")
    out = out.dropna(subset=["window_start_15min"]).copy()
    out["fault_loc_key"] = out["fault_loc_key"].astype(str).str.strip()

    m = (
        out["fault_loc_key"].isin(LOCS) &
        (out["window_start_15min"] >= TMIN) &
        (out["window_start_15min"] <= TMAX)
    )
    out = out.loc[m].copy()

    if "E_event_sequence" not in out.columns:
        out["E_event_sequence"] = "none"

    out["E_event_sequence"] = out["E_event_sequence"].fillna("none")
    return out

def _inject_event_sequence_only(df):
    out = df.copy()
    m_inj = (out["window_start_15min"] >= INJ_START) & (out["window_start_15min"] <= INJ_END)
    out.loc[m_inj, "E_event_sequence"] = "present"
    return out

def _run(df):
    out = posterior_update(
        df.copy(),
        likelihood_dict=likelihood,
        topo_lh=TOPO_LH,
        seq_lh=SEQUENCE_LH,
        priors=AGENT_PRIORS,
        mask=None
    )
    out = decision_update(out, utils_by_loc=UTILS_BY_LOC, mask=None)
    return out

BASE_IN = _subset(merged_15)
INJ_IN  = _inject_event_sequence_only(BASE_IN)

m_inj_time = (
    (BASE_IN["window_start_15min"] >= INJ_START) &
    (BASE_IN["window_start_15min"] <= INJ_END)
)

_print("BEFORE injection (E_event_sequence)")
print(BASE_IN.loc[m_inj_time, "E_event_sequence"].value_counts(dropna=False))

_print("AFTER injection (E_event_sequence := present)")
print(INJ_IN.loc[m_inj_time, "E_event_sequence"].value_counts(dropna=False))

_print("AGENT run — BASELINE")
BASE_OUT = _run(BASE_IN)

_print("AGENT run — INJECTED")
INJ_OUT = _run(INJ_IN)

mB = (
    (pd.to_datetime(BASE_OUT["window_start_15min"], utc=True) >= INJ_START) &
    (pd.to_datetime(BASE_OUT["window_start_15min"], utc=True) <= INJ_END) &
    (BASE_OUT["fault_loc_key"].isin(LOCS))
)

mI = (
    (pd.to_datetime(INJ_OUT["window_start_15min"], utc=True) >= INJ_START) &
    (pd.to_datetime(INJ_OUT["window_start_15min"], utc=True) <= INJ_END) &
    (INJ_OUT["fault_loc_key"].isin(LOCS))
)

B = BASE_OUT.loc[mB, KEYS + POST_COLS + [DEC_COL]]
I = INJ_OUT.loc[mI, KEYS + POST_COLS + [DEC_COL]]

MER = B.merge(I, on=KEYS, suffixes=("_base","_inj"), how="inner")

for c in POST_COLS:
    MER[f"d_{c}"] = MER[f"{c}_inj"] - MER[f"{c}_base"]

_print("POSTERIOR DIFF SUMMARY")
for c in POST_COLS:
    print(c, "max|Δ| =", MER[f"d_{c}"].abs().max())

decision_change_rate = (MER["Decision_base"] != MER["Decision_inj"]).mean()
print("\nDecisionChangeRate:", decision_change_rate)

_print("Decision distribution per location")
for loc in LOCS:
    print("\nLOC:", loc)
    print("Baseline:", B.loc[B["fault_loc_key"]==loc, DEC_COL].value_counts().to_dict())
    print("Injected:", I.loc[I["fault_loc_key"]==loc, DEC_COL].value_counts().to_dict())



BEFORE injection (E_event_sequence)
E_event_sequence
none    176
Name: count, dtype: int64

AFTER injection (E_event_sequence := present)
E_event_sequence
present    176
Name: count, dtype: int64

AGENT run — BASELINE
Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst



AGENT run — INJECTED
Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst



POSTERIOR DIFF SUMMARY
P_Normal max|Δ| = 0.13894035948838124
P_SensorFault max|Δ| = 0.05464834958113181
P_CommFault max|Δ| = 0.08429200990724935

DecisionChangeRate: 0.5

Decision distribution per location

LOC: A
Baseline: {'Ignore': 88}
Injected: {'Inspect': 88}

LOC: C
Baseline: {'Ignore': 88}
Injected: {'Ignore': 88}


In [56]:
# %% 18.interlock_AGG_ONLY [EVAL] Single-evidence injection test (E_interlock_agg only)

import pandas as pd
import numpy as np

LOCS = ["A", "C"]
INJ_START = pd.to_datetime("2020-01-01 00:00:00+00:00", utc=True)
INJ_END   = pd.to_datetime("2020-01-01 23:45:00+00:00", utc=True)

TMIN = INJ_START - pd.Timedelta(hours=6)
TMAX = INJ_END   + pd.Timedelta(hours=6)

POST_COLS = ["P_Normal","P_SensorFault","P_CommFault"]
DEC_COL = "Decision"
KEYS = ["window_start_15min","fault_loc_key"]

def _print(title):
    print("\n" + "="*96)
    print(title)
    print("="*96)

def _subset(df):
    out = df.copy()
    out["window_start_15min"] = pd.to_datetime(out["window_start_15min"], utc=True, errors="coerce")
    out = out.dropna(subset=["window_start_15min"]).copy()
    out["fault_loc_key"] = out["fault_loc_key"].astype(str).str.strip()

    m = (
        out["fault_loc_key"].isin(LOCS) &
        (out["window_start_15min"] >= TMIN) &
        (out["window_start_15min"] <= TMAX)
    )
    out = out.loc[m].copy()

    if "E_interlock_agg" not in out.columns:
        out["E_interlock_agg"] = "none"

    out["E_interlock_agg"] = out["E_interlock_agg"].fillna("none")
    return out

def _inject_interlock_agg_only(df):
    out = df.copy()
    m_inj = (out["window_start_15min"] >= INJ_START) & (out["window_start_15min"] <= INJ_END)
    out.loc[m_inj, "E_interlock_agg"] = "many"   # sterkaste bucket
    return out

def _run(df):
    out = posterior_update(
        df.copy(),
        likelihood_dict=likelihood,
        topo_lh=TOPO_LH,
        seq_lh=SEQUENCE_LH,
        priors=AGENT_PRIORS,
        mask=None
    )
    out = decision_update(out, utils_by_loc=UTILS_BY_LOC, mask=None)
    return out

BASE_IN = _subset(merged_15)
INJ_IN  = _inject_interlock_agg_only(BASE_IN)

m_inj_time = (
    (BASE_IN["window_start_15min"] >= INJ_START) &
    (BASE_IN["window_start_15min"] <= INJ_END)
)

_print("BEFORE injection (E_interlock_agg)")
print(BASE_IN.loc[m_inj_time, "E_interlock_agg"].value_counts(dropna=False))

_print("AFTER injection (E_interlock_agg := many)")
print(INJ_IN.loc[m_inj_time, "E_interlock_agg"].value_counts(dropna=False))

_print("AGENT run — BASELINE")
BASE_OUT = _run(BASE_IN)

_print("AGENT run — INJECTED")
INJ_OUT = _run(INJ_IN)

mB = (
    (pd.to_datetime(BASE_OUT["window_start_15min"], utc=True) >= INJ_START) &
    (pd.to_datetime(BASE_OUT["window_start_15min"], utc=True) <= INJ_END) &
    (BASE_OUT["fault_loc_key"].isin(LOCS))
)

mI = (
    (pd.to_datetime(INJ_OUT["window_start_15min"], utc=True) >= INJ_START) &
    (pd.to_datetime(INJ_OUT["window_start_15min"], utc=True) <= INJ_END) &
    (INJ_OUT["fault_loc_key"].isin(LOCS))
)

B = BASE_OUT.loc[mB, KEYS + POST_COLS + [DEC_COL]]
I = INJ_OUT.loc[mI, KEYS + POST_COLS + [DEC_COL]]

MER = B.merge(I, on=KEYS, suffixes=("_base","_inj"), how="inner")

for c in POST_COLS:
    MER[f"d_{c}"] = MER[f"{c}_inj"] - MER[f"{c}_base"]

_print("POSTERIOR DIFF SUMMARY")
for c in POST_COLS:
    print(c, "max|Δ| =", MER[f"d_{c}"].abs().max())

decision_change_rate = (MER["Decision_base"] != MER["Decision_inj"]).mean()
print("\nDecisionChangeRate:", decision_change_rate)

_print("Decision distribution per location")
for loc in LOCS:
    print("\nLOC:", loc)
    print("Baseline:", B.loc[B["fault_loc_key"]==loc, DEC_COL].value_counts().to_dict())
    print("Injected:", I.loc[I["fault_loc_key"]==loc, DEC_COL].value_counts().to_dict())



BEFORE injection (E_interlock_agg)
E_interlock_agg
none    176
Name: count, dtype: int64

AFTER injection (E_interlock_agg := many)
E_interlock_agg
many    176
Name: count, dtype: int64

AGENT run — BASELINE
Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst



AGENT run — INJECTED
Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst



POSTERIOR DIFF SUMMARY
P_Normal max|Δ| = 0.29530720171445324
P_SensorFault max|Δ| = 0.058703556362571574
P_CommFault max|Δ| = 0.23660364535188164

DecisionChangeRate: 1.0

Decision distribution per location

LOC: A
Baseline: {'Ignore': 88}
Injected: {'Inspect': 88}

LOC: C
Baseline: {'Ignore': 88}
Injected: {'Inspect': 88}


In [57]:
# %% 18.interlock_ONLY [EVAL] Single-evidence injection test (E_interlock only)

import pandas as pd
import numpy as np

LOCS = ["A", "C"]
INJ_START = pd.to_datetime("2020-01-01 00:00:00+00:00", utc=True)
INJ_END   = pd.to_datetime("2020-01-01 23:45:00+00:00", utc=True)

TMIN = INJ_START - pd.Timedelta(hours=6)
TMAX = INJ_END   + pd.Timedelta(hours=6)

POST_COLS = ["P_Normal","P_SensorFault","P_CommFault"]
DEC_COL = "Decision"
KEYS = ["window_start_15min","fault_loc_key"]

def _print(title):
    print("\n" + "="*96)
    print(title)
    print("="*96)

def _subset(df):
    out = df.copy()
    out["window_start_15min"] = pd.to_datetime(out["window_start_15min"], utc=True, errors="coerce")
    out = out.dropna(subset=["window_start_15min"]).copy()
    out["fault_loc_key"] = out["fault_loc_key"].astype(str).str.strip()

    m = (
        out["fault_loc_key"].isin(LOCS) &
        (out["window_start_15min"] >= TMIN) &
        (out["window_start_15min"] <= TMAX)
    )
    out = out.loc[m].copy()

    # normalise required cols
    out["E_event_any"] = pd.to_numeric(out["E_event_any"], errors="coerce").fillna(0).astype(int).clip(0,1)
    out["E_comm_fail_any"] = pd.to_numeric(out["E_comm_fail_any"], errors="coerce").fillna(0).astype(int).clip(0,1)

    if "E_interlock" not in out.columns:
        out["E_interlock"] = "none"

    out["E_interlock"] = out["E_interlock"].fillna("none")
    return out

def _inject_interlock_only(df):
    out = df.copy()
    m_inj = (out["window_start_15min"] >= INJ_START) & (out["window_start_15min"] <= INJ_END)
    out.loc[m_inj, "E_interlock"] = "present"
    return out

def _run(df):
    out = posterior_update(
        df.copy(),
        likelihood_dict=likelihood,
        topo_lh=TOPO_LH,
        seq_lh=SEQUENCE_LH,
        priors=AGENT_PRIORS,
        mask=None
    )
    out = decision_update(out, utils_by_loc=UTILS_BY_LOC, mask=None)
    return out

BASE_IN = _subset(merged_15)
INJ_IN  = _inject_interlock_only(BASE_IN)

m_inj_time = (
    (BASE_IN["window_start_15min"] >= INJ_START) &
    (BASE_IN["window_start_15min"] <= INJ_END)
)

_print("BEFORE injection (E_interlock)")
print(BASE_IN.loc[m_inj_time, "E_interlock"].value_counts(dropna=False))

_print("AFTER injection (E_interlock := present)")
print(INJ_IN.loc[m_inj_time, "E_interlock"].value_counts(dropna=False))

_print("AGENT run — BASELINE")
BASE_OUT = _run(BASE_IN)

_print("AGENT run — INJECTED")
INJ_OUT = _run(INJ_IN)

mB = (
    (pd.to_datetime(BASE_OUT["window_start_15min"], utc=True) >= INJ_START) &
    (pd.to_datetime(BASE_OUT["window_start_15min"], utc=True) <= INJ_END) &
    (BASE_OUT["fault_loc_key"].isin(LOCS))
)

mI = (
    (pd.to_datetime(INJ_OUT["window_start_15min"], utc=True) >= INJ_START) &
    (pd.to_datetime(INJ_OUT["window_start_15min"], utc=True) <= INJ_END) &
    (INJ_OUT["fault_loc_key"].isin(LOCS))
)

B = BASE_OUT.loc[mB, KEYS + POST_COLS + [DEC_COL]]
I = INJ_OUT.loc[mI, KEYS + POST_COLS + [DEC_COL]]

MER = B.merge(I, on=KEYS, suffixes=("_base","_inj"), how="inner")

for c in POST_COLS:
    MER[f"d_{c}"] = MER[f"{c}_inj"] - MER[f"{c}_base"]

_print("POSTERIOR DIFF SUMMARY")
for c in POST_COLS:
    print(c, "max|Δ| =", MER[f"d_{c}"].abs().max())

decision_change_rate = (MER["Decision_base"] != MER["Decision_inj"]).mean()
print("\nDecisionChangeRate:", decision_change_rate)

_print("Decision distribution per location")
for loc in LOCS:
    print("\nLOC:", loc)
    print("Baseline:", B.loc[B["fault_loc_key"]==loc, DEC_COL].value_counts().to_dict())
    print("Injected:", I.loc[I["fault_loc_key"]==loc, DEC_COL].value_counts().to_dict())



BEFORE injection (E_interlock)
E_interlock
none    176
Name: count, dtype: int64

AFTER injection (E_interlock := present)
E_interlock
present    176
Name: count, dtype: int64

AGENT run — BASELINE
Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst



AGENT run — INJECTED
Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst



POSTERIOR DIFF SUMMARY
P_Normal max|Δ| = 0.39623465858990337
P_SensorFault max|Δ| = 0.2363444817854932
P_CommFault max|Δ| = 0.15989017680441014

DecisionChangeRate: 1.0

Decision distribution per location

LOC: A
Baseline: {'Ignore': 88}
Injected: {'Inspect': 88}

LOC: C
Baseline: {'Ignore': 88}
Injected: {'Inspect': 88}


In [58]:
# %% 18.event_comm_COMBO [EVAL] Combined injection: E_event_any + E_comm_fail_any

import pandas as pd
import numpy as np

LOCS = ["A", "C"]
INJ_START = pd.to_datetime("2020-01-01 00:00:00+00:00", utc=True)
INJ_END   = pd.to_datetime("2020-01-01 23:45:00+00:00", utc=True)

TMIN = INJ_START - pd.Timedelta(hours=6)
TMAX = INJ_END   + pd.Timedelta(hours=6)

POST_COLS = ["P_Normal","P_SensorFault","P_CommFault"]
DEC_COL = "Decision"
KEYS = ["window_start_15min","fault_loc_key"]

def _print(title):
    print("\n" + "="*96)
    print(title)
    print("="*96)

def _subset(df):
    out = df.copy()
    out["window_start_15min"] = pd.to_datetime(out["window_start_15min"], utc=True, errors="coerce")
    out = out.dropna(subset=["window_start_15min"]).copy()
    out["fault_loc_key"] = out["fault_loc_key"].astype(str).str.strip()

    m = (
        out["fault_loc_key"].isin(LOCS) &
        (out["window_start_15min"] >= TMIN) &
        (out["window_start_15min"] <= TMAX)
    )
    out = out.loc[m].copy()

    out["E_event_any"] = pd.to_numeric(out["E_event_any"], errors="coerce").fillna(0).astype(int).clip(0,1)
    out["E_comm_fail_any"] = pd.to_numeric(out["E_comm_fail_any"], errors="coerce").fillna(0).astype(int).clip(0,1)

    return out

def _inject_combo(df):
    out = df.copy()
    m_inj = (out["window_start_15min"] >= INJ_START) & (out["window_start_15min"] <= INJ_END)
    out.loc[m_inj, "E_event_any"] = 1
    out.loc[m_inj, "E_comm_fail_any"] = 1
    return out

def _compact_triplet(df):
    out = df.copy()

    out["E_kV"] = (
        out["E_kV_15min"]
        .map({"present":"normal","absent_or_invalid":"abnormal","normal":"normal","abnormal":"abnormal","unknown":"unknown"})
        .fillna("unknown")
    )

    out["E_event"] = out["E_event_any"].map({1:"present",0:"none"}).fillna("none")

    out["E_event_under_comm"] = np.where(
        (out["E_event_any"].eq(1)) & (out["E_comm_fail_any"].eq(1)),
        "present", "none"
    )

    out["LH_key3"] = list(zip(out["E_kV"], out["E_event"], out["E_event_under_comm"]))
    return out

def _run(df):
    out = posterior_update(
        df.copy(),
        likelihood_dict=likelihood,
        topo_lh=TOPO_LH,
        seq_lh=SEQUENCE_LH,
        priors=AGENT_PRIORS,
        mask=None
    )
    out = decision_update(out, utils_by_loc=UTILS_BY_LOC, mask=None)
    return out

BASE_IN = _subset(merged_15)
INJ_IN  = _inject_combo(BASE_IN)

m_inj_time = (
    (BASE_IN["window_start_15min"] >= INJ_START) &
    (BASE_IN["window_start_15min"] <= INJ_END)
)

b_view = _compact_triplet(BASE_IN.loc[m_inj_time].copy())
i_view = _compact_triplet(INJ_IN .loc[m_inj_time].copy())

_print("LH_key3 value_counts (inj-window)")
print("BASELINE:")
print(b_view["LH_key3"].value_counts())
print("\nINJECTED:")
print(i_view["LH_key3"].value_counts())

_print("AGENT run — BASELINE")
BASE_OUT = _run(BASE_IN)

_print("AGENT run — INJECTED")
INJ_OUT = _run(INJ_IN)

mB = (
    (pd.to_datetime(BASE_OUT["window_start_15min"], utc=True) >= INJ_START) &
    (pd.to_datetime(BASE_OUT["window_start_15min"], utc=True) <= INJ_END) &
    (BASE_OUT["fault_loc_key"].isin(LOCS))
)

mI = (
    (pd.to_datetime(INJ_OUT["window_start_15min"], utc=True) >= INJ_START) &
    (pd.to_datetime(INJ_OUT["window_start_15min"], utc=True) <= INJ_END) &
    (INJ_OUT["fault_loc_key"].isin(LOCS))
)

B = BASE_OUT.loc[mB, KEYS + POST_COLS + [DEC_COL]]
I = INJ_OUT.loc[mI, KEYS + POST_COLS + [DEC_COL]]

MER = B.merge(I, on=KEYS, suffixes=("_base","_inj"), how="inner")

for c in POST_COLS:
    MER[f"d_{c}"] = MER[f"{c}_inj"] - MER[f"{c}_base"]

_print("POSTERIOR DIFF SUMMARY")
for c in POST_COLS:
    print(c, "max|Δ| =", MER[f"d_{c}"].abs().max())

decision_change_rate = (MER["Decision_base"] != MER["Decision_inj"]).mean()
print("\nDecisionChangeRate:", decision_change_rate)

_print("Decision distribution per location")
for loc in LOCS:
    print("\nLOC:", loc)
    print("Baseline:", B.loc[B["fault_loc_key"]==loc, DEC_COL].value_counts().to_dict())
    print("Injected:", I.loc[I["fault_loc_key"]==loc, DEC_COL].value_counts().to_dict())



LH_key3 value_counts (inj-window)
BASELINE:
LH_key3
(normal, none, none)    176
Name: count, dtype: int64

INJECTED:
LH_key3
(normal, present, present)    176
Name: count, dtype: int64

AGENT run — BASELINE
Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst



AGENT run — INJECTED
Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst



POSTERIOR DIFF SUMMARY
P_Normal max|Δ| = 0.7386046053695883
P_SensorFault max|Δ| = 0.00041400021271185093
P_CommFault max|Δ| = 0.7381906051568764

DecisionChangeRate: 1.0

Decision distribution per location

LOC: A
Baseline: {'Ignore': 88}
Injected: {'Isolate': 88}

LOC: C
Baseline: {'Ignore': 88}
Injected: {'Isolate': 88}


In [59]:
# %% 18.event_ONLY [EVAL] Single-evidence injection test (E_event_any only)

import pandas as pd
import numpy as np

LOCS = ["A", "C"]
INJ_START = pd.to_datetime("2020-01-01 00:00:00+00:00", utc=True)
INJ_END   = pd.to_datetime("2020-01-01 23:45:00+00:00", utc=True)

TMIN = INJ_START - pd.Timedelta(hours=6)
TMAX = INJ_END   + pd.Timedelta(hours=6)

POST_COLS = ["P_Normal","P_SensorFault","P_CommFault"]
DEC_COL = "Decision"
KEYS = ["window_start_15min","fault_loc_key"]

def _print(title):
    print("\n" + "="*96)
    print(title)
    print("="*96)

def _subset(df):
    out = df.copy()
    out["window_start_15min"] = pd.to_datetime(out["window_start_15min"], utc=True, errors="coerce")
    out = out.dropna(subset=["window_start_15min"]).copy()
    out["fault_loc_key"] = out["fault_loc_key"].astype(str).str.strip()

    m = (
        out["fault_loc_key"].isin(LOCS) &
        (out["window_start_15min"] >= TMIN) &
        (out["window_start_15min"] <= TMAX)
    )
    out = out.loc[m].copy()

    out["E_event_any"] = pd.to_numeric(out["E_event_any"], errors="coerce").fillna(0).astype(int).clip(0,1)
    out["E_comm_fail_any"] = pd.to_numeric(out["E_comm_fail_any"], errors="coerce").fillna(0).astype(int).clip(0,1)

    return out

def _inject_event_only(df):
    out = df.copy()
    m_inj = (out["window_start_15min"] >= INJ_START) & (out["window_start_15min"] <= INJ_END)
    out.loc[m_inj, "E_event_any"] = 1
    return out

def _compact_triplet(df):
    out = df.copy()

    out["E_kV"] = (
        out["E_kV_15min"]
        .map({"present":"normal","absent_or_invalid":"abnormal","normal":"normal","abnormal":"abnormal","unknown":"unknown"})
        .fillna("unknown")
    )

    out["E_event"] = out["E_event_any"].map({1:"present",0:"none"}).fillna("none")

    out["E_event_under_comm"] = np.where(
        (out["E_event_any"].eq(1)) & (out["E_comm_fail_any"].eq(1)),
        "present", "none"
    )

    out["LH_key3"] = list(zip(out["E_kV"], out["E_event"], out["E_event_under_comm"]))
    return out

def _run(df):
    out = posterior_update(
        df.copy(),
        likelihood_dict=likelihood,
        topo_lh=TOPO_LH,
        seq_lh=SEQUENCE_LH,
        priors=AGENT_PRIORS,
        mask=None
    )
    out = decision_update(out, utils_by_loc=UTILS_BY_LOC, mask=None)
    return out

BASE_IN = _subset(merged_15)
INJ_IN  = _inject_event_only(BASE_IN)

m_inj_time = (
    (BASE_IN["window_start_15min"] >= INJ_START) &
    (BASE_IN["window_start_15min"] <= INJ_END)
)

b_view = _compact_triplet(BASE_IN.loc[m_inj_time].copy())
i_view = _compact_triplet(INJ_IN .loc[m_inj_time].copy())

_print("BEFORE injection (E_event_any)")
print(b_view["E_event_any"].value_counts(dropna=False))

_print("AFTER injection (E_event_any := 1)")
print(i_view["E_event_any"].value_counts(dropna=False))

_print("LH_key3 value_counts (inj-window)")
print("BASELINE:")
print(b_view["LH_key3"].value_counts(dropna=False))
print("\nINJECTED:")
print(i_view["LH_key3"].value_counts(dropna=False))

_print("LIKELIHOOD sanity (event effect)")
k1 = ("normal","none","none")
k2 = ("normal","present","none")

for h in ["Normal","SensorFault","CommFault"]:
    v1 = float(likelihood[h].get(k1, np.nan))
    v2 = float(likelihood[h].get(k2, np.nan))
    print(h, "L", k1, "=", v1, " | L", k2, "=", v2)

_print("AGENT run — BASELINE")
BASE_OUT = _run(BASE_IN)

_print("AGENT run — INJECTED")
INJ_OUT = _run(INJ_IN)

mB = (
    (pd.to_datetime(BASE_OUT["window_start_15min"], utc=True) >= INJ_START) &
    (pd.to_datetime(BASE_OUT["window_start_15min"], utc=True) <= INJ_END) &
    (BASE_OUT["fault_loc_key"].isin(LOCS))
)

mI = (
    (pd.to_datetime(INJ_OUT["window_start_15min"], utc=True) >= INJ_START) &
    (pd.to_datetime(INJ_OUT["window_start_15min"], utc=True) <= INJ_END) &
    (INJ_OUT["fault_loc_key"].isin(LOCS))
)

B = BASE_OUT.loc[mB, KEYS + POST_COLS + [DEC_COL]]
I = INJ_OUT.loc[mI, KEYS + POST_COLS + [DEC_COL]]

MER = B.merge(I, on=KEYS, suffixes=("_base","_inj"), how="inner")

for c in POST_COLS:
    MER[f"d_{c}"] = MER[f"{c}_inj"] - MER[f"{c}_base"]

_print("POSTERIOR DIFF SUMMARY")
for c in POST_COLS:
    print(c, "max|Δ| =", MER[f"d_{c}"].abs().max())

decision_change_rate = (MER["Decision_base"] != MER["Decision_inj"]).mean()
print("\nDecisionChangeRate:", decision_change_rate)



BEFORE injection (E_event_any)
E_event_any
0    176
Name: count, dtype: int64

AFTER injection (E_event_any := 1)
E_event_any
1    176
Name: count, dtype: int64

LH_key3 value_counts (inj-window)
BASELINE:
LH_key3
(normal, none, none)    176
Name: count, dtype: int64

INJECTED:
LH_key3
(normal, present, none)    176
Name: count, dtype: int64

LIKELIHOOD sanity (event effect)
Normal L ('normal', 'none', 'none') = 0.3998800359892033  | L ('normal', 'present', 'none') = 0.039988003598920324
SensorFault L ('normal', 'none', 'none') = 0.0594000594000594  | L ('normal', 'present', 'none') = 0.049500049500049506
CommFault L ('normal', 'none', 'none') = 0.05  | L ('normal', 'present', 'none') = 0.05

AGENT run — BASELINE
Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst



AGENT run — INJECTED
Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst



POSTERIOR DIFF SUMMARY
P_Normal max|Δ| = 0.0072403982415950585
P_SensorFault max|Δ| = 0.004079245642922302
P_CommFault max|Δ| = 0.0031611525986727585

DecisionChangeRate: 0.0


In [60]:
# %% 18.comm_ONLY [EVAL] Single-evidence injection test (E_comm_fail_any only)

import pandas as pd
import numpy as np

# -----------------------------
# SETTINGS
# -----------------------------
LOCS = ["A", "C"]
INJ_START = pd.to_datetime("2020-01-01 00:00:00+00:00", utc=True)
INJ_END   = pd.to_datetime("2020-01-01 23:45:00+00:00", utc=True)

TMIN = INJ_START - pd.Timedelta(hours=6)
TMAX = INJ_END   + pd.Timedelta(hours=6)

# -----------------------------
# CONTRACTS
# -----------------------------
assert "merged_15" in globals()
assert "likelihood" in globals()
assert "posterior_update" in globals()
assert "decision_update" in globals()
assert "TOPO_LH" in globals()
assert "SEQUENCE_LH" in globals()
assert "AGENT_PRIORS" in globals()
assert "UTILS_BY_LOC" in globals()


POST_COLS = ["P_Normal","P_SensorFault","P_CommFault"]
DEC_COL = "Decision"
KEYS = ["window_start_15min","fault_loc_key"]

def _print(title):
    print("\n" + "="*96)
    print(title)
    print("="*96)

def _subset(df):
    out = df.copy()
    out["window_start_15min"] = pd.to_datetime(out["window_start_15min"], utc=True, errors="coerce")
    out = out.dropna(subset=["window_start_15min"]).copy()
    out["fault_loc_key"] = out["fault_loc_key"].astype(str).str.strip()

    m = (
        out["fault_loc_key"].isin(LOCS) &
        (out["window_start_15min"] >= TMIN) &
        (out["window_start_15min"] <= TMAX)
    )
    out = out.loc[m].copy()

    out["E_event_any"] = pd.to_numeric(out["E_event_any"], errors="coerce").fillna(0).astype(int).clip(0,1)
    out["E_comm_fail_any"] = pd.to_numeric(out["E_comm_fail_any"], errors="coerce").fillna(0).astype(int).clip(0,1)

    return out

def _inject_comm_only(df):
    out = df.copy()
    m_inj = (out["window_start_15min"] >= INJ_START) & (out["window_start_15min"] <= INJ_END)
    out.loc[m_inj, "E_comm_fail_any"] = 1
    return out

def _compact_triplet(df):
    out = df.copy()

    out["E_kV"] = (
        out["E_kV_15min"]
        .map({"present":"normal","absent_or_invalid":"abnormal","normal":"normal","abnormal":"abnormal","unknown":"unknown"})
        .fillna("unknown")
    )

    out["E_event"] = out["E_event_any"].map({1:"present",0:"none"}).fillna("none")

    out["E_event_under_comm"] = np.where(
        (out["E_event_any"].eq(1)) & (out["E_comm_fail_any"].eq(1)),
        "present", "none"
    )

    out["LH_key3"] = list(zip(out["E_kV"], out["E_event"], out["E_event_under_comm"]))
    return out

def _run(df):
    out = posterior_update(
        df.copy(),
        likelihood_dict=likelihood,
        topo_lh=TOPO_LH,
        seq_lh=SEQUENCE_LH,
        priors=AGENT_PRIORS,
        mask=None
    )
    out = decision_update(out, utils_by_loc=UTILS_BY_LOC, mask=None)
    return out

# -----------------------------
# Build baseline + injected
# -----------------------------
BASE_IN = _subset(merged_15)
INJ_IN  = _inject_comm_only(BASE_IN)

m_inj_time = (
    (BASE_IN["window_start_15min"] >= INJ_START) &
    (BASE_IN["window_start_15min"] <= INJ_END)
)

b_view = _compact_triplet(BASE_IN.loc[m_inj_time].copy())
i_view = _compact_triplet(INJ_IN .loc[m_inj_time].copy())

# -----------------------------
# BEFORE / AFTER
# -----------------------------
_print("BEFORE injection (E_comm_fail_any)")
print(b_view["E_comm_fail_any"].value_counts(dropna=False))

_print("AFTER injection (E_comm_fail_any := 1)")
print(i_view["E_comm_fail_any"].value_counts(dropna=False))

_print("LH_key3 value_counts (inj-window) — BASELINE vs INJECTED")
print("BASELINE:")
print(b_view["LH_key3"].value_counts(dropna=False))
print("\nINJECTED:")
print(i_view["LH_key3"].value_counts(dropna=False))

# -----------------------------
# Likelihood sanity
# -----------------------------
_print("LIKELIHOOD sanity for comm change")

# Keys we expect to change when only comm_fail=1 and event=0
k1 = ("normal","none","none")
k2 = ("normal","none","present")  # will only occur if event=1 as well

for h in ["Normal","SensorFault","CommFault"]:
    v1 = float(likelihood[h].get(k1, np.nan))
    v2 = float(likelihood[h].get(k2, np.nan))
    print(h, "L", k1, "=", v1, " | L", k2, "=", v2)

# -----------------------------
# Run model
# -----------------------------
_print("AGENT run — BASELINE")
BASE_OUT = _run(BASE_IN)

_print("AGENT run — INJECTED")
INJ_OUT = _run(INJ_IN)

# -----------------------------
# Compare posterior + decision
# -----------------------------
mB = (
    (pd.to_datetime(BASE_OUT["window_start_15min"], utc=True) >= INJ_START) &
    (pd.to_datetime(BASE_OUT["window_start_15min"], utc=True) <= INJ_END) &
    (BASE_OUT["fault_loc_key"].isin(LOCS))
)

mI = (
    (pd.to_datetime(INJ_OUT["window_start_15min"], utc=True) >= INJ_START) &
    (pd.to_datetime(INJ_OUT["window_start_15min"], utc=True) <= INJ_END) &
    (INJ_OUT["fault_loc_key"].isin(LOCS))
)

B = BASE_OUT.loc[mB, KEYS + POST_COLS + [DEC_COL]]
I = INJ_OUT.loc[mI, KEYS + POST_COLS + [DEC_COL]]

MER = B.merge(I, on=KEYS, suffixes=("_base","_inj"), how="inner")

for c in POST_COLS:
    MER[f"d_{c}"] = MER[f"{c}_inj"] - MER[f"{c}_base"]

MER["d_abs_sum"] = MER[[f"d_{c}" for c in POST_COLS]].abs().sum(axis=1)

_print("POSTERIOR DIFF SUMMARY")
for c in POST_COLS:
    print(c, "max|Δ| =", MER[f"d_{c}"].abs().max())

decision_change_rate = (MER["Decision_base"] != MER["Decision_inj"]).mean()
print("\nDecisionChangeRate:", decision_change_rate)

_print("Decision distribution per location")
for loc in LOCS:
    print("\nLOC:", loc)
    print("Baseline:", B.loc[B["fault_loc_key"]==loc, DEC_COL].value_counts().to_dict())
    print("Injected:", I.loc[I["fault_loc_key"]==loc, DEC_COL].value_counts().to_dict())



BEFORE injection (E_comm_fail_any)
E_comm_fail_any
0    176
Name: count, dtype: int64

AFTER injection (E_comm_fail_any := 1)
E_comm_fail_any
1    176
Name: count, dtype: int64

LH_key3 value_counts (inj-window) — BASELINE vs INJECTED
BASELINE:
LH_key3
(normal, none, none)    176
Name: count, dtype: int64

INJECTED:
LH_key3
(normal, none, none)    176
Name: count, dtype: int64

LIKELIHOOD sanity for comm change
Normal L ('normal', 'none', 'none') = 0.3998800359892033  | L ('normal', 'none', 'present') = nan
SensorFault L ('normal', 'none', 'none') = 0.0594000594000594  | L ('normal', 'none', 'present') = nan
CommFault L ('normal', 'none', 'none') = 0.05  | L ('normal', 'none', 'present') = nan

AGENT run — BASELINE
Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst



AGENT run — INJECTED
Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst



POSTERIOR DIFF SUMMARY
P_Normal max|Δ| = 0.0
P_SensorFault max|Δ| = 0.0
P_CommFault max|Δ| = 0.0

DecisionChangeRate: 0.0

Decision distribution per location

LOC: A
Baseline: {'Ignore': 88}
Injected: {'Ignore': 88}

LOC: C
Baseline: {'Ignore': 88}
Injected: {'Ignore': 88}


In [61]:
# %% 18.kV_ONLY [EVAL] Single-evidence injection test (E_kV_15min only) — complete cell

import pandas as pd
import numpy as np

# -----------------------------
# SETTINGS
# -----------------------------
LOCS = ["A", "C"]
INJ_START = pd.to_datetime("2020-01-01 00:00:00+00:00", utc=True)
INJ_END   = pd.to_datetime("2020-01-01 23:45:00+00:00", utc=True)

# small context window (avoid huge data / kernel instability)
TMIN = INJ_START - pd.Timedelta(hours=6)
TMAX = INJ_END   + pd.Timedelta(hours=6)

INJ_VALUE = "absent_or_invalid"   # kV manipulation in this model

RUN_SEQUENTIAL = False
RUN_STATELESS  = True  

# -----------------------------
# CONTRACTS
# -----------------------------
assert "merged_15" in globals(), "Mangler merged_15"
assert "likelihood" in globals(), "Mangler likelihood"
assert "TOPO_LH" in globals(), "Mangler TOPO_LH"
assert "SEQUENCE_LH" in globals(), "Mangler SEQUENCE_LH"
assert "AGENT_PRIORS" in globals(), "Mangler AGENT_PRIORS"
assert "UTILS_BY_LOC" in globals(), "Mangler UTILS_BY_LOC"
assert "decision_update" in globals(), "Mangler decision_update"

HAS_SEQ = "posterior_update_sequential_daily" in globals()
HAS_ST  = "posterior_update" in globals()

assert RUN_SEQUENTIAL ^ RUN_STATELESS, \
    "Nøyaktig éin av RUN_SEQUENTIAL eller RUN_STATELESS må vere True"

assert (RUN_SEQUENTIAL ^ RUN_STATELESS), "Velg nøyaktig éin: RUN_SEQUENTIAL eller RUN_STATELESS"
assert (RUN_SEQUENTIAL and HAS_SEQ) or (RUN_STATELESS and HAS_ST), "Mangler posterior-funksjon for valde RUN_*"



def _print(title):
    print("\n" + "="*96)
    print(title)
    print("="*96)

def _subset(df):
    out = df.copy()
    out["window_start_15min"] = pd.to_datetime(out["window_start_15min"], utc=True, errors="coerce")
    out = out.dropna(subset=["window_start_15min"]).copy()
    out["fault_loc_key"] = out["fault_loc_key"].astype(str).str.strip()

    m = (
        out["fault_loc_key"].isin([str(x).strip() for x in LOCS]) &
        (out["window_start_15min"] >= TMIN) &
        (out["window_start_15min"] <= TMAX)
    )
    out = out.loc[m].copy()

    # normalise binary cols
    for c in ["E_event_any", "E_comm_fail_any"]:
        assert c in out.columns, f"Mangler {c} i merged_15"
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0).astype(int).clip(0, 1)

    assert "E_kV_15min" in out.columns, "Mangler E_kV_15min i merged_15"
    return out

def _inject_EkV_only(df):
    out = df.copy()
    m_inj = (out["window_start_15min"] >= INJ_START) & (out["window_start_15min"] <= INJ_END)
    out.loc[m_inj, "E_kV_15min"] = INJ_VALUE
    return out

def _compact_triplet(df):
    out = df.copy()

    out["E_kV"] = (
        out["E_kV_15min"]
        .map({"present":"normal","absent_or_invalid":"abnormal","normal":"normal","abnormal":"abnormal","unknown":"unknown"})
        .fillna("unknown")
    )
    out["E_event"] = out["E_event_any"].map({1:"present",0:"none"}).fillna("none")
    out["E_comm_fail_any"] = out["E_comm_fail_any"].fillna(0).astype(int).clip(0, 1)
    out["E_event_under_comm"] = np.where(
        (out["E_event_any"].eq(1)) & (out["E_comm_fail_any"].eq(1)),
        "present", "none"
    )
    out["LH_key3"] = list(zip(out["E_kV"], out["E_event"], out["E_event_under_comm"]))
    return out

def _run(df):
    out = df.copy()
    if RUN_SEQUENTIAL:
        out = posterior_update_sequential_daily(
            out,
            likelihood_dict=likelihood,
            topo_lh=TOPO_LH,
            seq_lh=SEQUENCE_LH,
            base_priors=AGENT_PRIORS,
            mask=None
        )
    else:
        out = posterior_update(
            out,
            likelihood_dict=likelihood,
            topo_lh=TOPO_LH,
            seq_lh=SEQUENCE_LH,
            priors=AGENT_PRIORS,
            mask=None
        )
    out = decision_update(out, utils_by_loc=UTILS_BY_LOC, mask=None)
    return out

# -----------------------------
# 0) Build input baseline vs injected (same rows)
# -----------------------------
BASE_IN = _subset(merged_15)
INJ_IN  = _inject_EkV_only(BASE_IN)

# inj-window mask (by time only; we always keep LOCS in subset)
m_inj_time = (BASE_IN["window_start_15min"] >= INJ_START) & (BASE_IN["window_start_15min"] <= INJ_END)

# -----------------------------
# 1) BEFORE/AFTER evidence diagnostics (inj-window)
# -----------------------------
b_view = _compact_triplet(BASE_IN.loc[m_inj_time].copy())
i_view = _compact_triplet(INJ_IN .loc[m_inj_time].copy())

_print(f"BEFORE injection (E_kV_15min) — {INJ_START} → {INJ_END} — locs={LOCS}")
print(b_view["E_kV_15min"].value_counts(dropna=False))

_print(f"AFTER  injection (E_kV_15min := '{INJ_VALUE}') — {INJ_START} → {INJ_END} — locs={LOCS}")
print(i_view["E_kV_15min"].value_counts(dropna=False))

_print("LH_key3 value_counts (inj-window) — BASELINE vs INJECTED")
print("BASELINE:")
print(b_view["LH_key3"].value_counts(dropna=False))
print("\nINJECTED:")
print(i_view["LH_key3"].value_counts(dropna=False))

# -----------------------------
# 2) Likelihood numeric check for the switched keys
# -----------------------------
k_normal   = ("normal",  "none", "none")
k_abnormal = ("abnormal","none", "none")

_print("LIKELIHOOD sanity (per hypothesis): compare (normal,none,none) vs (abnormal,none,none)")
for h in ["Normal","SensorFault","CommFault"]:
    lh = likelihood[h]
    v1 = float(lh.get(k_normal, np.nan))
    v2 = float(lh.get(k_abnormal, np.nan))
    ratio = (v2 / v1) if (np.isfinite(v1) and np.isfinite(v2) and v1 > 0) else np.nan
    print(f"{h:>10s}  L{str(k_normal)}={v1:.6g}   L{str(k_abnormal)}={v2:.6g}   ratio(abnormal/normal)={ratio:.6g}")

# -----------------------------
# 3) Run agent baseline vs injected
# -----------------------------
_print("AGENT run — BASELINE")
BASE_OUT = _run(BASE_IN)

_print("AGENT run — INJECTED (E_kV_15min only)")
INJ_OUT  = _run(INJ_IN)

# -----------------------------
# 4) Compare posterior + decision (inj-window) using merge on (time, loc)
# -----------------------------
POST_COLS = ["P_Normal","P_SensorFault","P_CommFault"]
DEC_COL = "Decision"
KEYS = ["window_start_15min","fault_loc_key"]

for c in POST_COLS + [DEC_COL]:
    assert c in BASE_OUT.columns, f"BASE_OUT manglar {c}"
    assert c in INJ_OUT.columns,  f"INJ_OUT manglar {c}"

# Filter by time window directly on each output (no boolean index alignment issues)
mB = (pd.to_datetime(BASE_OUT["window_start_15min"], utc=True, errors="coerce") >= INJ_START) & \
     (pd.to_datetime(BASE_OUT["window_start_15min"], utc=True, errors="coerce") <= INJ_END) & \
     (BASE_OUT["fault_loc_key"].astype(str).str.strip().isin([str(x).strip() for x in LOCS]))

mI = (pd.to_datetime(INJ_OUT["window_start_15min"], utc=True, errors="coerce") >= INJ_START) & \
     (pd.to_datetime(INJ_OUT["window_start_15min"], utc=True, errors="coerce") <= INJ_END) & \
     (INJ_OUT["fault_loc_key"].astype(str).str.strip().isin([str(x).strip() for x in LOCS]))

B = BASE_OUT.loc[mB, KEYS + POST_COLS + [DEC_COL]].copy()
I = INJ_OUT .loc[mI, KEYS + POST_COLS + [DEC_COL]].copy()

MER = B.merge(I, on=KEYS, suffixes=("_base","_inj"), how="inner")

for c in POST_COLS:
    MER[f"d_{c}"] = (MER[f"{c}_inj"] - MER[f"{c}_base"]).astype(float)

MER["d_abs_sum"] = MER[[f"d_{c}" for c in POST_COLS]].abs().sum(axis=1)

_print("POSTERIOR DIFF SUMMARY (inj-window) — max|ΔP|")
decision_change_rate = (MER["Decision_base"] != MER["Decision_inj"]).mean()
print("\nDecisionChangeRate:", decision_change_rate)

for c in POST_COLS:
    print(f"{c:12s} max|Δ| = {MER[f'd_{c}'].abs().max():.6g}")
print("\nAny posterior changed (tol=1e-12)?", bool((MER["d_abs_sum"] > 1e-12).any()))

_print("Decision distribution (inj-window) — BASELINE vs INJECTED (per loc)")
for loc in LOCS:
    b = B.loc[B["fault_loc_key"].eq(loc), DEC_COL].value_counts(dropna=False).to_dict()
    i = I.loc[I["fault_loc_key"].eq(loc), DEC_COL].value_counts(dropna=False).to_dict()
    print(f"\nLOC={loc}")
    print("  BASELINE:", b)
    print("  INJECTED:", i)

_print("Rows where Decision CHANGED (inj-window) — sample up to 20")
chg = MER[MER["Decision_base"] != MER["Decision_inj"]].copy()
if len(chg) == 0:
    print("No decision changes in injection window.")
else:
    display(chg.sort_values(KEYS).head(20)[KEYS + ["Decision_base","Decision_inj"] + [f"d_{c}" for c in POST_COLS]])

_print("TOP rows by |ΔP| sum (inj-window) — up to 20")
top = MER.sort_values("d_abs_sum", ascending=False).head(20)
display(top[KEYS + ["d_abs_sum"] + [f"d_{c}" for c in POST_COLS] + ["Decision_base","Decision_inj"]])

print("\nDONE ✅")



BEFORE injection (E_kV_15min) — 2020-01-01 00:00:00+00:00 → 2020-01-01 23:45:00+00:00 — locs=['A', 'C']
E_kV_15min
present    176
Name: count, dtype: int64

AFTER  injection (E_kV_15min := 'absent_or_invalid') — 2020-01-01 00:00:00+00:00 → 2020-01-01 23:45:00+00:00 — locs=['A', 'C']
E_kV_15min
absent_or_invalid    176
Name: count, dtype: int64

LH_key3 value_counts (inj-window) — BASELINE vs INJECTED
BASELINE:
LH_key3
(normal, none, none)    176
Name: count, dtype: int64

INJECTED:
LH_key3
(abnormal, none, none)    176
Name: count, dtype: int64

LIKELIHOOD sanity (per hypothesis): compare (normal,none,none) vs (abnormal,none,none)
    Normal  L('normal', 'none', 'none')=0.39988   L('abnormal', 'none', 'none')=0.049985   ratio(abnormal/normal)=0.125
SensorFault  L('normal', 'none', 'none')=0.0594001   L('abnormal', 'none', 'none')=0.544501   ratio(abnormal/normal)=9.16667
 CommFault  L('normal', 'none', 'none')=0.05   L('abnormal', 'none', 'none')=0.05   ratio(abnormal/normal)=1

AGENT

Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst



AGENT run — INJECTED (E_kV_15min only)
Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst



POSTERIOR DIFF SUMMARY (inj-window) — max|ΔP|

DecisionChangeRate: 0.0
P_Normal     max|Δ| = 0.0412344
P_SensorFault max|Δ| = 0.0388727
P_CommFault  max|Δ| = 0.00236172

Any posterior changed (tol=1e-12)? True

Decision distribution (inj-window) — BASELINE vs INJECTED (per loc)

LOC=A
  BASELINE: {'Ignore': 88}
  INJECTED: {'Ignore': 88}

LOC=C
  BASELINE: {'Ignore': 88}
  INJECTED: {'Ignore': 88}

Rows where Decision CHANGED (inj-window) — sample up to 20
No decision changes in injection window.

TOP rows by |ΔP| sum (inj-window) — up to 20


,window_start_15min,fault_loc_key,d_abs_sum,d_P_Normal,d_P_SensorFault,d_P_CommFault,Decision_base,Decision_inj
0,2020-01-01 02:00:00+00:00,A,0.082469,-0.041234,0.038873,0.002362,Ignore,Ignore
1,2020-01-01 02:00:00+00:00,C,0.082469,-0.041234,0.038873,0.002362,Ignore,Ignore
2,2020-01-01 02:15:00+00:00,A,0.082469,-0.041234,0.038873,0.002362,Ignore,Ignore
3,2020-01-01 02:15:00+00:00,C,0.082469,-0.041234,0.038873,0.002362,Ignore,Ignore
4,2020-01-01 02:30:00+00:00,A,0.082469,-0.041234,0.038873,0.002362,Ignore,Ignore
5,2020-01-01 02:30:00+00:00,C,0.082469,-0.041234,0.038873,0.002362,Ignore,Ignore
6,2020-01-01 02:45:00+00:00,A,0.082469,-0.041234,0.038873,0.002362,Ignore,Ignore
7,2020-01-01 02:45:00+00:00,C,0.082469,-0.041234,0.038873,0.002362,Ignore,Ignore
8,2020-01-01 03:00:00+00:00,A,0.082469,-0.041234,0.038873,0.002362,Ignore,Ignore
9,2020-01-01 03:00:00+00:00,C,0.082469,-0.041234,0.038873,0.002362,Ignore,Ignore



DONE ✅


In [62]:
# %% 17.xx [EVAL] Baseline vs Injected (stateless OR sequential) + 3 response metrics

import numpy as np
import pandas as pd

# -----------------------------
# Contracts
# -----------------------------
assert "merged_15" in globals(), "Mangler merged_15"
assert "decision_update" in globals(), "Mangler decision_update (køyr 16.16-cella først)"
assert "UTILS_BY_LOC" in globals(), "Mangler UTILS_BY_LOC"

HAS_POST_STATELESS = "posterior_update" in globals()
HAS_POST_SEQDAILY  = "posterior_update_sequential_daily" in globals()

assert HAS_POST_STATELESS or HAS_POST_SEQDAILY, (
    "Mangler posterior update-funksjon. "
    "Stateless: posterior_update. Sequential: posterior_update_sequential_daily."
)

g = globals()
likelihood_dict = g.get("likelihood") or g.get("LIKELIHOOD")
topo_lh = g.get("TOPO_LH")
seq_lh = g.get("SEQUENCE_LH")
priors = g.get("AGENT_PRIORS", {"Normal": 0.90, "SensorFault": 0.05, "CommFault": 0.05})

assert likelihood_dict is not None, "Mangler likelihood"
assert topo_lh is not None, "Mangler TOPO_LH"
assert seq_lh is not None, "Mangler SEQUENCE_LH"

# -----------------------------
# Helpers
# -----------------------------
def _get_time_col(df):
    for c in ["window_start_15min", "ts15", "timestamp_15min", "timestamp"]:
        if c in df.columns:
            return c
    raise KeyError("Fann ingen tidskolonne (15-min)")

TIME_COL = _get_time_col(merged_15)

def _ensure_dt_utc(df):
    df = df.copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], utc=True, errors="coerce")
    return df

def _norm_binary_cols(df, cols=("E_event_any","E_comm_fail_any")):
    df = df.copy()
    for col in cols:
        if col in df.columns:
            if df[col].dtype == "object":
                df[col] = df[col].map({"present": 1, "none": 0}).fillna(0)
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int).clip(0, 1)
    return df

def _run_agent(df):
    # posterior
    if HAS_POST_STATELESS:
        df2 = posterior_update(
            df,
            likelihood_dict=likelihood_dict,
            topo_lh=topo_lh,
            seq_lh=seq_lh,
            priors=priors,
            mask=None,
        )
    else:
        df2 = posterior_update_sequential_daily(
            df,
            likelihood_dict=likelihood_dict,
            topo_lh=topo_lh,
            seq_lh=seq_lh,
            base_priors=priors,
            mask=None,
        )

    # decision
    df2 = decision_update(df2, utils_by_loc=UTILS_BY_LOC, mask=None)
    return df2

def apply_injection_timeboxed(df, inject_loc, inject_kind, t0, t1):
    """
    Inject ONLY within [t0,t1] for the specified location.
    inject_kind:
      - "sensor_fault_kV"      -> E_kV_15min = "absent_or_invalid"
      - "comm_fault_event"     -> E_comm_fail_any=1 AND E_event_any=1
      - "topo_inconsistent"    -> E_topo_consistency = "inconsistent"
      - "event_spike"          -> E_event_any=1 (only)
    """
    df = df.copy()
    df = _norm_binary_cols(df, ("E_event_any","E_comm_fail_any"))

    ts = pd.to_datetime(df[TIME_COL], utc=True, errors="coerce")
    m_loc  = df["fault_loc_key"].astype(str).str.strip().eq(str(inject_loc).strip())
    m_time = (ts >= t0) & (ts <= t1)
    m = m_loc & m_time

    if not m.any():
        print("WARN: Injection mask empty (no rows matched loc+time). Returning baseline copy.")
        return df

    if inject_kind == "sensor_fault_kV":
        assert "E_kV_15min" in df.columns, "Mangler E_kV_15min"
        df.loc[m, "E_kV_15min"] = "absent_or_invalid"

    elif inject_kind == "comm_fault_event":
        assert "E_event_any" in df.columns and "E_comm_fail_any" in df.columns, "Mangler E_event_any/E_comm_fail_any"
        df.loc[m, "E_event_any"] = 1
        df.loc[m, "E_comm_fail_any"] = 1

    elif inject_kind == "event_spike":
        assert "E_event_any" in df.columns, "Mangler E_event_any"
        df.loc[m, "E_event_any"] = 1

    elif inject_kind == "topo_inconsistent":
        assert "E_topo_consistency" in df.columns, "Mangler E_topo_consistency"
        df.loc[m, "E_topo_consistency"] = "inconsistent"

    else:
        raise ValueError(f"Ukjent inject_kind: {inject_kind}")

    return df

def compute_metrics(BASE, INJ, inject_loc, t0, t1):
    keys = [TIME_COL, "fault_loc_key"]

    b = BASE[keys + ["Decision"]].rename(columns={"Decision":"D_base"})
    j = INJ[keys + ["Decision"]].rename(columns={"Decision":"D_inj"})
    m = j.merge(b, on=keys, how="left")

    # 1) DecisionChangeRate across eval slice (only where baseline exists)
    m2 = m.dropna(subset=["D_base"]).copy()
    dcr = float((m2["D_inj"] != m2["D_base"]).mean()) if len(m2) else np.nan

    # Focus on inject location only for timing metrics
    x = m[m["fault_loc_key"].astype(str).str.strip().eq(str(inject_loc).strip())].copy()
    x = x.sort_values(TIME_COL)

    # Robust start: first available row inside [t0,t1]
    x_inj = x[(x[TIME_COL] >= t0) & (x[TIME_COL] <= t1)].copy()
    if len(x_inj) == 0:
        return dcr, None, None

    t0_eff = x_inj[TIME_COL].iloc[0]

    # 2) TimeToEscalation: first Inspect/Isolate within injection interval
    esc = x_inj[x_inj["D_inj"].isin(["Inspect","Isolate"])][TIME_COL]
    if len(esc):
        first_esc = esc.iloc[0]
        tte = int(round((first_esc - t0_eff) / pd.Timedelta(minutes=15)))
    else:
        tte = None

    # 3) RecoveryTime: first post-injection time where injected decision equals baseline
    x_post = x[x[TIME_COL] > t1].copy()
    rec = x_post[x_post["D_inj"] == x_post["D_base"]][TIME_COL]
    if len(rec):
        first_rec = rec.iloc[0]
        rt = int(round((first_rec - t1) / pd.Timedelta(minutes=15)))
    else:
        rt = None

    return dcr, tte, rt

# -----------------------------
# CONFIG (edit only these)
# -----------------------------
EVAL_LOCS    = ["A","C"]          # B har ikkje målingar
INJECT_LOC   = "A"                # "A" eller "C"
INJECT_KIND  = "comm_fault_event" # sensor_fault_kV | event_spike | comm_fault_event | topo_inconsistent

TMIN         = "2020-01-01 00:00:00"
TMAX         = "2020-01-01 23:45:00"

POST_MARGIN_WINDOWS = 96          # 1 døgn etter injeksjon for recovery (juster ved behov)

# -----------------------------
# Build eval slice
# -----------------------------
df0 = _ensure_dt_utc(merged_15)
t0 = pd.to_datetime(TMIN, utc=True)
t1 = pd.to_datetime(TMAX, utc=True)
t1_post = t1 + POST_MARGIN_WINDOWS * pd.Timedelta(minutes=15)

SLICE = df0[df0["fault_loc_key"].isin(EVAL_LOCS)].copy()
SLICE = SLICE[(SLICE[TIME_COL] >= t0) & (SLICE[TIME_COL] <= t1_post)].copy()

# Baseline
BASE = _run_agent(SLICE)

# Injected (time-boxed)
SLICE_INJ = apply_injection_timeboxed(SLICE, inject_loc=INJECT_LOC, inject_kind=INJECT_KIND, t0=t0, t1=t1)
INJ = _run_agent(SLICE_INJ)

# Metrics
dcr, tte, rt = compute_metrics(BASE, INJ, inject_loc=INJECT_LOC, t0=t0, t1=t1)

agent_name = "Stateless" if HAS_POST_STATELESS else "SequentialDaily"
print(f"\n=== EVAL ({agent_name}) | inject={INJECT_KIND} at {INJECT_LOC} | locs={EVAL_LOCS} | inj={TMIN}→{TMAX} | post={POST_MARGIN_WINDOWS} windows ===")
print("DecisionChangeRate :", None if np.isnan(dcr) else round(dcr, 4))
print("TimeToEscalation   :", tte, "(15-min windows; None = not observed)")
print("RecoveryTime       :", rt,  "(15-min windows; None = not observed)")

# Quick view for injected loc inside injection interval
view_cols = [TIME_COL,"fault_loc_key","Decision","P_Normal","P_SensorFault","P_CommFault"]
disp = INJ.loc[
    INJ["fault_loc_key"].astype(str).str.strip().eq(str(INJECT_LOC).strip())
    & (INJ[TIME_COL] >= t0) & (INJ[TIME_COL] <= t1),
    view_cols
].sort_values(TIME_COL).head(16)

display(disp)


Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst


Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst



=== EVAL (Stateless) | inject=comm_fault_event at A | locs=['A', 'C'] | inj=2020-01-01 00:00:00→2020-01-01 23:45:00 | post=96 windows ===
DecisionChangeRate : 0.2391
TimeToEscalation   : 0 (15-min windows; None = not observed)
RecoveryTime       : 1 (15-min windows; None = not observed)


,window_start_15min,fault_loc_key,Decision,P_Normal,P_SensorFault,P_CommFault
0,2020-01-01 02:00:00+00:00,A,Isolate,0.26048,0.000975,0.738545
2,2020-01-01 02:15:00+00:00,A,Isolate,0.26048,0.000975,0.738545
4,2020-01-01 02:30:00+00:00,A,Isolate,0.26048,0.000975,0.738545
6,2020-01-01 02:45:00+00:00,A,Isolate,0.26048,0.000975,0.738545
8,2020-01-01 03:00:00+00:00,A,Isolate,0.26048,0.000975,0.738545
10,2020-01-01 03:15:00+00:00,A,Isolate,0.26048,0.000975,0.738545
12,2020-01-01 03:30:00+00:00,A,Isolate,0.26048,0.000975,0.738545
14,2020-01-01 03:45:00+00:00,A,Isolate,0.26048,0.000975,0.738545
16,2020-01-01 04:00:00+00:00,A,Isolate,0.26048,0.000975,0.738545
18,2020-01-01 04:15:00+00:00,A,Isolate,0.26048,0.000975,0.738545


In [63]:
# %% 17.91 [SCENARIO] Localized injection – one location manipulated, others baseline (NO HMI)

import numpy as np
import pandas as pd

assert "merged_15" in globals()
assert "posterior_update" in globals()
assert "decision_update" in globals()

# --- utilities df ---
if "utils_by_loc" not in globals():
    if "UTILS_BY_LOC" in globals():
        utils_by_loc = UTILS_BY_LOC
    elif "utils_df" in globals():
        utils_by_loc = utils_df
    else:
        raise AssertionError("Mangler utils_by_loc (DataFrame).")
assert isinstance(utils_by_loc, pd.DataFrame)

# --- time col ---
def _get_time_col(df):
    for c in ["window_start_15min", "ts15", "timestamp_15min", "timestamp"]:
        if c in df.columns:
            return c
    raise KeyError("Fann ingen 15-min tidskolonne")
TIME_COL = _get_time_col(merged_15)

# --- run helper ---
def run_localized_scenario(
    name: str,
    df_base: pd.DataFrame,
    loc_subset,
    tmin: str,
    tmax: str,
    inject_loc: str,
    inject_kind: str,  # "sensor_fault_kV" or "comm_fault_event"
):
    df = df_base.copy()

    # Slice to the evaluation set (multiple locs)
    df = df[df["fault_loc_key"].isin(loc_subset)].copy()
    df = df[df[TIME_COL] >= pd.to_datetime(tmin, utc=True)].copy()
    df = df[df[TIME_COL] <= pd.to_datetime(tmax, utc=True)].copy()

    # Apply injection ONLY for one location
    m = df["fault_loc_key"].eq(inject_loc)

    if inject_kind == "sensor_fault_kV":
        if "E_kV_15min" not in df.columns:
            raise KeyError("Mangler E_kV_15min")
        df.loc[m, "E_kV_15min"] = "absent_or_invalid"

    elif inject_kind == "comm_fault_event":
        # Normalize target cols first (robust if strings)
        for col in ["E_event_any", "E_comm_fail_any"]:
            if col in df.columns and df[col].dtype == "object":
                df[col] = df[col].map({"present": 1, "none": 0}).fillna(0)

        if "E_event_any" not in df.columns or "E_comm_fail_any" not in df.columns:
            raise KeyError("Mangler E_event_any eller E_comm_fail_any")

        df.loc[m, "E_event_any"] = 1
        df.loc[m, "E_comm_fail_any"] = 1

    else:
        raise ValueError("inject_kind må vere 'sensor_fault_kV' eller 'comm_fault_event'")

    # Normalize event/comm to 0/1 for posterior_update
    for col in ["E_event_any", "E_comm_fail_any"]:
        if col in df.columns:
            if df[col].dtype == "object":
                df[col] = df[col].map({"present": 1, "none": 0}).fillna(0)
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int).clip(0, 1)

    # Likelihood objects (rev17 contract)
    g = globals()
    likelihood_dict = g.get("likelihood") or g.get("LIKELIHOOD")
    topo_lh = g.get("TOPO_LH")
    seq_lh = g.get("SEQUENCE_LH")

    df = posterior_update(df, likelihood_dict, topo_lh, seq_lh)
    df = decision_update(df, utils_by_loc)

    # Summaries
    dec_col = "Decision" if "Decision" in df.columns else ("decision" if "decision" in df.columns else "action")

    # Overall decision rates (all locs combined)
    overall = df[dec_col].value_counts(normalize=True).round(3).to_dict()

    # Per-location decision rates (so you can see baseline loc vs injected loc)
    per_loc = (
        df.groupby("fault_loc_key")[dec_col]
          .value_counts(normalize=True)
          .rename("rate")
          .reset_index()
    )

    # Posterior means per location
    post_cols = [c for c in ["P_Normal","P_SensorFault","P_CommFault"] if c in df.columns]
    post_by_loc = df.groupby("fault_loc_key")[post_cols].mean().round(4)

    print(f"\n=== {name} | inject={inject_kind} at {inject_loc} | eval_locs={loc_subset} | {tmin}→{tmax} ===")
    print("Overall decision_rate:", overall)
    print("\nDecision_rate per loc (top):")
    print(per_loc.sort_values(["fault_loc_key","rate"], ascending=[True, False]).head(30))
    print("\nPosterior mean per loc:")
    print(post_by_loc)

    return df, overall, per_loc, post_by_loc


# ---------- RUN EXAMPLES ----------
loc_subset = ["A","B"]
tmin = "2020-01-01"
tmax = "2020-02-29"

# 1) Inject sensor fault only in A (B stays baseline)
_ = run_localized_scenario(
    name="Localized SensorFault",
    df_base=merged_15,
    loc_subset=loc_subset,
    tmin=tmin,
    tmax=tmax,
    inject_loc="A",
    inject_kind="sensor_fault_kV",
)

# 2) Inject comm fault only in A (B stays baseline)
_ = run_localized_scenario(
    name="Localized CommFault",
    df_base=merged_15,
    loc_subset=loc_subset,
    tmin=tmin,
    tmax=tmax,
    inject_loc="A",
    inject_kind="comm_fault_event",
)


Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst



=== Localized SensorFault | inject=sensor_fault_kV at A | eval_locs=['A', 'B'] | 2020-01-01→2020-02-29 ===
Overall decision_rate: {'Ignore': 0.99, 'Isolate': 0.009, 'Inspect': 0.001}

Decision_rate per loc (top):
  fault_loc_key Decision      rate
0             A   Ignore  0.998056
1             A  Inspect  0.001944
2             B   Ignore  0.982323
3             B  Isolate  0.017677

Posterior mean per loc:
               P_Normal  P_SensorFault  P_CommFault
fault_loc_key                                      
A                0.9566         0.0406       0.0028
B                0.9598         0.0033       0.0369
Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst



=== Localized CommFault | inject=comm_fault_event at A | eval_locs=['A', 'B'] | 2020-01-01→2020-02-29 ===
Overall decision_rate: {'Isolate': 0.509, 'Ignore': 0.491}

Decision_rate per loc (top):
  fault_loc_key Decision      rate
0             A  Isolate  1.000000
1             B   Ignore  0.982323
2             B  Isolate  0.017677

Posterior mean per loc:
               P_Normal  P_SensorFault  P_CommFault
fault_loc_key                                      
A                0.2600         0.0010       0.7390
B                0.9598         0.0033       0.0369


In [64]:
# %% 17.92 [SCENARIO] Partial injection-rate + cross-location (NO HMI)

import numpy as np
import pandas as pd

assert "merged_15" in globals()
assert "posterior_update" in globals()
assert "decision_update" in globals()

# --- utilities df ---
if "utils_by_loc" not in globals():
    if "UTILS_BY_LOC" in globals():
        utils_by_loc = UTILS_BY_LOC
    elif "utils_df" in globals():
        utils_by_loc = utils_df
    else:
        raise AssertionError("Mangler utils_by_loc (DataFrame).")
assert isinstance(utils_by_loc, pd.DataFrame)

# --- time col ---
def _get_time_col(df):
    for c in ["window_start_15min", "ts15", "timestamp_15min", "timestamp"]:
        if c in df.columns:
            return c
    raise KeyError("Fann ingen 15-min tidskolonne")
TIME_COL = _get_time_col(merged_15)

# --- core runner (partial injection on selected fraction of rows in one location) ---
def run_partial_injection(
    name: str,
    df_base: pd.DataFrame,
    loc_subset,
    tmin: str,
    tmax: str,
    inject_loc: str,
    inject_kind: str,     # "sensor_fault_kV" or "comm_fault_event"
    inject_rate: float,   # 0..1 fraction of rows in inject_loc to modify
    seed: int = 0,
):
    rng = np.random.default_rng(seed)
    df = df_base.copy()

    # slice evaluation set
    df = df[df["fault_loc_key"].isin(loc_subset)].copy()
    df = df[df[TIME_COL] >= pd.to_datetime(tmin, utc=True)].copy()
    df = df[df[TIME_COL] <= pd.to_datetime(tmax, utc=True)].copy()

    # choose rows inside inject_loc
    idx_loc = df.index[df["fault_loc_key"].eq(inject_loc)]
    n = len(idx_loc)
    k = int(round(inject_rate * n))

    if k > 0:
        chosen = rng.choice(idx_loc.to_numpy(), size=k, replace=False)

        if inject_kind == "sensor_fault_kV":
            if "E_kV_15min" not in df.columns:
                raise KeyError("Mangler E_kV_15min")
            df.loc[chosen, "E_kV_15min"] = "absent_or_invalid"

        elif inject_kind == "comm_fault_event":
            # normalize base if strings
            for col in ["E_event_any", "E_comm_fail_any"]:
                if col in df.columns and df[col].dtype == "object":
                    df[col] = df[col].map({"present": 1, "none": 0}).fillna(0)

            if "E_event_any" not in df.columns or "E_comm_fail_any" not in df.columns:
                raise KeyError("Mangler E_event_any eller E_comm_fail_any")

            df.loc[chosen, "E_event_any"] = 1
            df.loc[chosen, "E_comm_fail_any"] = 1

        else:
            raise ValueError("inject_kind må vere 'sensor_fault_kV' eller 'comm_fault_event'")

    # normalize event/comm to 0/1 for posterior_update
    for col in ["E_event_any", "E_comm_fail_any"]:
        if col in df.columns:
            if df[col].dtype == "object":
                df[col] = df[col].map({"present": 1, "none": 0}).fillna(0)
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int).clip(0, 1)

    # run agent (rev17 contract)
    g = globals()
    likelihood_dict = g.get("likelihood") or g.get("LIKELIHOOD")
    topo_lh = g.get("TOPO_LH")
    seq_lh = g.get("SEQUENCE_LH")

    df = posterior_update(df, likelihood_dict, topo_lh, seq_lh)
    df = decision_update(df, utils_by_loc)

    dec_col = "Decision" if "Decision" in df.columns else ("decision" if "decision" in df.columns else "action")
    post_cols = [c for c in ["P_Normal","P_SensorFault","P_CommFault"] if c in df.columns]

    # summaries
    overall = df[dec_col].value_counts(normalize=True).to_dict()
    per_loc = (
        df.groupby("fault_loc_key")[dec_col]
          .value_counts(normalize=True)
          .rename("rate")
          .reset_index()
    )
    post_by_loc = df.groupby("fault_loc_key")[post_cols].mean()

    # compact output rows
    row = {
        "name": name,
        "inject_loc": inject_loc,
        "inject_kind": inject_kind,
        "inject_rate": inject_rate,
        "rows": len(df),
        "windows": df[TIME_COL].nunique(),
        "p_ignore_overall": float(overall.get("Ignore", 0.0)),
        "p_inspect_overall": float(overall.get("Inspect", 0.0)),
        "p_isolate_overall": float(overall.get("Isolate", 0.0)),
    }

    # per-loc decision rates for injected + other locs (useful for dose–response)
    for loc in loc_subset:
        sub = per_loc[per_loc["fault_loc_key"].eq(loc)]
        row[f"{loc}_p_ignore"] = float(sub.loc[sub[dec_col].eq("Ignore"), "rate"].sum()) if len(sub) else 0.0
        row[f"{loc}_p_inspect"] = float(sub.loc[sub[dec_col].eq("Inspect"), "rate"].sum()) if len(sub) else 0.0
        row[f"{loc}_p_isolate"] = float(sub.loc[sub[dec_col].eq("Isolate"), "rate"].sum()) if len(sub) else 0.0

    return row, post_by_loc.round(4)


# ------------------- CONFIG -------------------
loc_subset = ["A", "B"]
tmin = "2020-01-01"
tmax = "2020-02-29"

rates = [0.0, 0.10, 0.25, 0.50, 1.0]  # dose–response
kinds = ["sensor_fault_kV", "comm_fault_event"]

# Cross-location: test injection in A and in B
inject_locs = ["A", "B"]

rows = []
post_snapshots = {}

for inject_loc in inject_locs:
    for kind in kinds:
        for r in rates:
            name = f"{kind} @ {inject_loc} | rate={r:.2f}"
            row, post_by_loc = run_partial_injection(
                name=name,
                df_base=merged_15,
                loc_subset=loc_subset,
                tmin=tmin,
                tmax=tmax,
                inject_loc=inject_loc,
                inject_kind=kind,
                inject_rate=r,
                seed=0,
            )
            rows.append(row)

            # store a few snapshots (optional)
            if r in (0.0, 0.5, 1.0):
                post_snapshots[(inject_loc, kind, r)] = post_by_loc

df_dose = pd.DataFrame(rows)

# --- Print: dose–response table (per injected loc) ---
print("=== Dose–response (decision rates) ===")
cols_show = [
    "inject_kind","inject_loc","inject_rate",
    "p_ignore_overall","p_inspect_overall","p_isolate_overall",
    "A_p_ignore","A_p_inspect","A_p_isolate",
    "B_p_ignore","B_p_inspect","B_p_isolate",
]
print(df_dose[cols_show].sort_values(["inject_kind","inject_loc","inject_rate"]))

# --- Optional: show a few posterior snapshots (per loc) ---
print("\n=== Posterior snapshots (selected rates) ===")
for key, post in post_snapshots.items():
    inject_loc, kind, r = key
    print(f"\n[{kind} @ {inject_loc} | rate={r:.2f}]")
    print(post)


Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst


Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

...

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst


Rows in mask with any missing utility: 0
Distinct fault_loc_key with missing utility: 0


Series([], Name: count, dtype: int64)

,fault_loc_key,SensorIntegrity_Utility_worst,StatusConsistency_Utility_worst,CyberSecurity_Utility_worst,DataIntegrity_Utility_worst,DigitalTwinImpact_Utility_worst


KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

def plot_dose_response(inject_kind, inject_loc):

    df_plot = df_dose[
        (df_dose["inject_kind"] == inject_kind) &
        (df_dose["inject_loc"] == inject_loc)
    ].sort_values("inject_rate")

    x = df_plot["inject_rate"]

    if inject_kind == "sensor_fault_kV":
        y = df_plot[f"{inject_loc}_p_inspect"]
        ylabel = "Share of time windows selecting INSPECT\n(at manipulated location)"
        title = f"Dose–response behaviour under sensor manipulation at {inject_loc}"

    elif inject_kind == "comm_fault_event":
        y = df_plot[f"{inject_loc}_p_isolate"]
        ylabel = "Share of time windows selecting ISOLATE\n(at manipulated location)"
        title = f"Dose–response behaviour under communication fault at {inject_loc}"

    else:
        return

    plt.figure()
    plt.plot(x, y, marker="o")
    plt.xlabel("Injection rate (fraction of manipulated time windows)")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.ylim(0, 1)
    plt.grid(True)
    plt.tight_layout()
    plt.show()


# Eksempel
plot_dose_response("sensor_fault_kV", "A")
plot_dose_response("comm_fault_event", "A")


In [ ]:
# 17.1 [TEST] Slice sanity (TZ-safe, agent-correct, NO MASK)
#   - avoids "no effect" by targeting event rows OR injecting event


import pandas as pd
import numpy as np

assert "merged_15" in globals(), "Mangler merged_15"
assert "posterior_update" in globals(), "Mangler posterior_update (16.15)"
assert "decision_update" in globals(), "Mangler decision_update (16.16)"
assert "UTILS_BY_LOC" in globals(), "Mangler UTILS_BY_LOC (16.16)"
assert "likelihood" in globals() and "TOPO_LH" in globals() and "SEQUENCE_LH" in globals(), "Mangler 16.14 ting"

def _ts_naive_utc(series: pd.Series) -> pd.Series:
    ts = pd.to_datetime(series, utc=True, errors="coerce")
    # make comparable to naive timestamps
    return ts.dt.tz_convert("UTC").dt.tz_localize(None)

def slice_for_test(df, start, end):
    view = df.copy()
    ts = _ts_naive_utc(view["window_start_15min"])
    idx = view.index[(ts >= start) & (ts < end)]
    return view.loc[idx].copy()

# --- slice window ---
test_slice = slice_for_test(
    merged_15,
    start=pd.Timestamp("2018-01-01 00:00"),
    end=pd.Timestamp("2018-01-02 00:00"),
)

print("Rows in slice:", len(test_slice))

# ---------- BASELINE ----------
BASE0 = posterior_update(
    test_slice,
    likelihood_dict=likelihood,
    topo_lh=TOPO_LH,
    seq_lh=SEQUENCE_LH,
    priors=AGENT_PRIORS,
    mask=None,
)
BASE0 = decision_update(BASE0, utils_by_loc=UTILS_BY_LOC, mask=None)

# ---------- INJECT (ensure EFFECT) ----------
INJ = test_slice.copy()

# Try A først
event_rows = INJ["E_event_any"].fillna(0).astype(int).eq(1)

if event_rows.any():
    INJ = INJ.loc[event_rows].copy()
    inj_mode = "A_event_rows_only"
else:
    # fallback: force event everywhere
    INJ = test_slice.copy()
    INJ["E_event_any"] = 1
    inj_mode = "B_force_event"

# Always inject comm fail
INJ["E_comm_fail_any"] = 1


INJ1 = posterior_update(
    INJ,
    likelihood_dict=likelihood,
    topo_lh=TOPO_LH,
    seq_lh=SEQUENCE_LH,
    priors=AGENT_PRIORS,
    mask=None,
)
INJ1 = decision_update(INJ1, utils_by_loc=UTILS_BY_LOC, mask=None)

# ---------- REPORT ----------
base_mean = float(BASE0["P_CommFault"].mean()) if len(BASE0) else float("nan")
inj_mean  = float(INJ1["P_CommFault"].mean()) if len(INJ1) else float("nan")

print("\nOK: 17.1 slice sanity")
print("Mode:", inj_mode)
print("Rows baseline :", len(BASE0))
print("Rows injected :", len(INJ1))
print("Mean P_CommFault baseline:", base_mean)
print("Mean P_CommFault injected :", inj_mean)
print("Delta mean P_CommFault    :", inj_mean - base_mean)

if len(INJ1) and len(BASE0):
    # decision change rate: compare INJ rows against BASE rows on same keys
    keys = ["window_start_15min", "fault_loc_key"]
    b = BASE0[keys + ["Decision", "P_CommFault"]].rename(
        columns={"Decision":"base_decision","P_CommFault":"base_P_CommFault"}
    )
    j = INJ1[keys + ["Decision", "P_CommFault"]].rename(
        columns={"Decision":"inj_decision","P_CommFault":"inj_P_CommFault"}
    )
    m = j.merge(b, on=keys, how="left")
    if "base_decision" in m.columns:
        rate = float((m["inj_decision"] != m["base_decision"]).mean())
        print("Decision changed rate     :", rate)
    else:
        print("Decision changed rate     : N/A (no baseline match on keys)")

display(INJ1.head(10))


In [ ]:
# %% 17.2 [EVAL] Utility-flat test (SAME posterior, utilities-only effect)  [REN + merge-safe]

import pandas as pd
import numpy as np

assert "merged_15" in globals(), "Mangler merged_15"
assert "posterior_update" in globals(), "Mangler posterior_update (16.15)"
assert "decision_update" in globals(), "Mangler decision_update (16.16 el.l.)"
assert "UTILS_BY_LOC" in globals(), "Mangler UTILS_BY_LOC (utilities per fault_loc_key)"
assert "likelihood" in globals(), "Mangler likelihood"
assert "TOPO_LH" in globals(), "Mangler TOPO_LH"
assert "SEQUENCE_LH" in globals(), "Mangler SEQUENCE_LH"

# ------------------------------------------------------------
# A) Eval-sett (liten og relevant, kan endre maska om du vil)
# ------------------------------------------------------------
EVAL_MASK = (
    (merged_15["E_comm_fail_any"].fillna(0).astype(int) == 1) |
    (merged_15["E_event_any"].fillna(0).astype(int) == 1)
)

BASE_SRC = merged_15.loc[EVAL_MASK].copy()
print("Eval rows:", len(BASE_SRC))

# ------------------------------------------------------------
# B) Run-agent helper (mask-free)
# ------------------------------------------------------------
def _run_agent(df: pd.DataFrame, utils_by_loc_df: pd.DataFrame) -> pd.DataFrame:
    df2 = posterior_update(
        df,
        likelihood_dict=likelihood,
        topo_lh=TOPO_LH,
        seq_lh=SEQUENCE_LH,
        priors=AGENT_PRIORS,
        mask=None,
    )
    df2 = decision_update(df2, utils_by_loc=utils_by_loc_df, mask=None)
    return df2

# ------------------------------------------------------------
# C) Make UTILS_FLAT with SAME schema as UTILS_BY_LOC
#    (decision_update merges on fault_loc_key)
# ------------------------------------------------------------
NEED_UTIL_COLS = [
    "SensorIntegrity_Utility_worst",
    "StatusConsistency_Utility_worst",
    "CyberSecurity_Utility_worst",
    "DataIntegrity_Utility_worst",
    "DigitalTwinImpact_Utility_worst",
]

def _utils_to_df(utils_by_loc):
    # already DF
    if isinstance(utils_by_loc, pd.DataFrame):
        return utils_by_loc.copy()

    # dict -> df
    if isinstance(utils_by_loc, dict):
        rows = []
        for loc, d in utils_by_loc.items():
            row = {"fault_loc_key": loc}
            if isinstance(d, dict):
                row.update(d)
            rows.append(row)
        return pd.DataFrame(rows)

    raise TypeError(f"UTILS_BY_LOC must be DataFrame or dict, got: {type(utils_by_loc)}")

UTILS_DF = _utils_to_df(UTILS_BY_LOC)

assert "fault_loc_key" in UTILS_DF.columns, "UTILS_BY_LOC manglar 'fault_loc_key' kolonne (etter konvertering)."

# ensure required util cols exist (create if missing)
for c in NEED_UTIL_COLS:
    if c not in UTILS_DF.columns:
        UTILS_DF[c] = np.nan

UTILS_FLAT = UTILS_DF.copy()
for c in NEED_UTIL_COLS:
    # "flate" = 0.0 (same across locations)
    UTILS_FLAT[c] = 0.0

# ------------------------------------------------------------
# D) Baseline vs flat-utilities (SAME evidence => SAME posterior)
# ------------------------------------------------------------
BASE = _run_agent(BASE_SRC, utils_by_loc_df=UTILS_DF)
FLAT = _run_agent(BASE_SRC, utils_by_loc_df=UTILS_FLAT)

# sanity: posterior identical (numerical tolerance)
for c in ["P_Normal", "P_SensorFault", "P_CommFault"]:
    if c in BASE.columns and c in FLAT.columns:
        max_abs = float(np.max(np.abs(BASE[c].values - FLAT[c].values)))
        print(f"Posterior diff max |{c}|:", max_abs)

# keep compact outputs
base_keep = BASE[[
    "window_start_15min", "fault_loc_key",
    "Decision", "P_CommFault", "EU_Ignore", "EU_Inspect", "EU_Isolate"
]].rename(columns={
    "Decision": "base_Decision",
    "P_CommFault": "base_P_CommFault",
    "EU_Ignore": "base_EU_Ignore",
    "EU_Inspect": "base_EU_Inspect",
    "EU_Isolate": "base_EU_Isolate",
})

flat_keep = FLAT[[
    "window_start_15min", "fault_loc_key",
    "Decision", "P_CommFault", "EU_Ignore", "EU_Inspect", "EU_Isolate"
]].rename(columns={
    "Decision": "Decision",
    "P_CommFault": "P_CommFault",
    "EU_Ignore": "EU_Ignore",
    "EU_Inspect": "EU_Inspect",
    "EU_Isolate": "EU_Isolate",
})

batch_results_17_2_utilflat = flat_keep.merge(
    base_keep,
    on=["window_start_15min", "fault_loc_key"],
    how="left",
    validate="1:1",
)

batch_results_17_2_utilflat["scenario"] = "utility_flat_same_posterior"
batch_results_17_2_utilflat["decision_changed"] = (
    batch_results_17_2_utilflat["Decision"] != batch_results_17_2_utilflat["base_Decision"]
).astype(int)

batch_results_17_2_utilflat["delta_P_CommFault"] = (
    batch_results_17_2_utilflat["P_CommFault"] - batch_results_17_2_utilflat["base_P_CommFault"]
)

print("OK: 17.2 util-flat ferdig.")
print("Decision changed rate:", float(batch_results_17_2_utilflat["decision_changed"].mean()))
print("Mean delta_P_CommFault (should be ~0):", float(batch_results_17_2_utilflat["delta_P_CommFault"].mean()))

display(batch_results_17_2_utilflat.head(20))


In [ ]:
# 17.3 [EVAL] Robustness report (NO pass/fail)  -- for batch_results_17_2 schema

import numpy as np
import pandas as pd

df = batch_results_17_2_utilflat.copy()


# decision rates + flip-rate
rates = (
    df.groupby("scenario")
      .agg(
          n=("Decision", "size"),
          Ignore_rate=("Decision", lambda s: (s=="Ignore").mean()),
          Inspect_rate=("Decision", lambda s: (s=="Inspect").mean()),
          Isolate_rate=("Decision", lambda s: (s=="Isolate").mean()),
          flip_rate=("decision_changed","mean"),
          avg_P_CommFault=("P_CommFault","mean"),
          mean_delta_P_CommFault=("delta_P_CommFault","mean"),
          median_delta_P_CommFault=("delta_P_CommFault","median"),
      )
      .reset_index()
)

# --- Combined robustness summary ---

# Decision rates + flip-rate
rates = (
    df.groupby("scenario")
      .agg(
          n=("Decision", "size"),
          Ignore_rate=("Decision", lambda s: (s=="Ignore").mean()),
          Inspect_rate=("Decision", lambda s: (s=="Inspect").mean()),
          Isolate_rate=("Decision", lambda s: (s=="Isolate").mean()),
          flip_rate=("decision_changed","mean"),
          avg_P_CommFault=("P_CommFault","mean"),
          mean_delta_P_CommFault=("delta_P_CommFault","mean"),
          median_delta_P_CommFault=("delta_P_CommFault","median"),
      )
)

# Posterior shift (CommFault)
posterior_shift = (
    df.groupby("scenario")
      .agg(
          median_abs_dP_CommFault=("delta_P_CommFault", 
                                   lambda s: float(np.median(np.abs(s))))
      )
)

# ΔEU vs baseline
df["dEU_best"] = df[["EU_Inspect","EU_Isolate"]].max(axis=1) - df["base_EU_Ignore"]

delta_eu = (
    df.groupby("scenario")
      .agg(mean_dEU_best=("dEU_best","mean"))
)

# --- Merge everything ---
robustness_summary = (
    rates
    .join(posterior_shift)
    .join(delta_eu)
    .reset_index()
)

print("=== Robustness summary (stateless v17.2) ===")
display(robustness_summary)



In [ ]:
# %% 17.95 NOTEBOOK VERIFY (Stateless / Sequential / +/- etbr)  [PASTE BEFORE HMI]
import pandas as pd
import numpy as np

def _pp(title):
    print("\n" + "="*90)
    print(title)
    print("="*90)

def _exists(name: str) -> bool:
    return name in globals() and globals()[name] is not None

def _to_ts(s):
    return pd.to_datetime(s, utc=True, errors="coerce")

def _safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

_pp("VERIFY START")

# ------------------------------------------------------------
# 0) REQUIRED: merged_15
# ------------------------------------------------------------
if not _exists("merged_15"):
    print("ERROR: merged_15 is missing. Run pipeline cells up to merged_15 before verification.")
else:
    m = merged_15.copy()

    _pp("A) merged_15 basic sanity")
    print("merged_15 shape:", m.shape)
    print("merged_15 columns (sample):", list(m.columns)[:25], "...")

    req = ["window_start_15min","fault_loc_key","Decision","P_Normal","P_SensorFault","P_CommFault"]
    print("required present:", [c for c in req if c in m.columns])
    print("required missing:", [c for c in req if c not in m.columns])

    for c in ["window_start_15min","fault_loc_key","Decision"]:
        if c in m.columns:
            print(f"dtype[{c}]:", m[c].dtype)

    if "window_start_15min" in m.columns:
        m["_ts"] = _to_ts(m["window_start_15min"])
        print("bad timestamps:", int(m["_ts"].isna().sum()), "/", len(m))
    else:
        m["_ts"] = pd.NaT

    # ------------------------------------------------------------
    # 1) Decision distribution
    # ------------------------------------------------------------
    _pp("B) Decisions distribution")
    vc = m["Decision"].value_counts(dropna=False)
    print(vc)
    n = len(m)
    rate = (vc.get("Inspect",0) + vc.get("Isolate",0)) / n if n else np.nan
    print("inspect+isolate rate:", _safe_float(rate))

    # ------------------------------------------------------------
    # 2) Posterior sanity
    # ------------------------------------------------------------
    _pp("C) Posterior sanity")
    post_cols = ["P_Normal","P_SensorFault","P_CommFault"]
    s = m[post_cols].sum(axis=1)
    print("posterior sum min/mean/max:",
          _safe_float(s.min()), _safe_float(s.mean()), _safe_float(s.max()))
    print("posterior NaN rows:",
          int(m[post_cols].isna().any(axis=1).sum()))
    print("posterior out-of-range rows:",
          int(((m[post_cols] < -1e-9) | (m[post_cols] > 1+1e-9)).any(axis=1).sum()))

    # ------------------------------------------------------------
    # 3) Evidence overview
    # ------------------------------------------------------------
    _pp("D) Evidence overview")
    evid_cols = ["E_kV_15min","E_event_any","E_comm_fail_any",
                 "E_topo_consistency","E_interlock","E_interlock_agg"]
    found = [c for c in evid_cols if c in m.columns]
    print("evidence cols found:", found)
    for c in found:
        print(f"\n{c} value_counts:")
        print(m[c].value_counts(dropna=False).head(10))

    # ------------------------------------------------------------
    # 4) etbr / Event_Triggered_Belief_Reset verification (ETBR) (AUTO)
    # ------------------------------------------------------------
    _pp("E) etbr / Event_Triggered_Belief_Reset verification")

    has_etbr = all(c in m.columns for c in ["etbr_time","etbr_strength"])
    if not has_etbr:
        print("No etbr_time / etbr_strength columns → etbr not active in this agent.")
    else:
        print("etbr columns detected: etbr_time, etbr_strength")

        print("\netbr_strength.describe():")
        print(m["etbr_strength"].describe())

        hi = m[m["etbr_strength"] >= 0.9].copy()
        print("\nrows with etbr_strength >= 0.9:", len(hi))

        if len(hi) == 0:
            print("WARNING: No strong etbr events found.")
        else:
            show_cols = [
                "window_start_15min","fault_loc_key",
                "etbr_time","etbr_strength",
                "E_kV_15min","E_topo_consistency",
                "P_Normal","P_SensorFault","P_CommFault",
                "Decision"
            ]
            show_cols = [c for c in show_cols if c in hi.columns]

            print("\nTop 10 rows around strong etbr (sorted by etbr_strength desc):")
            print(
                hi.sort_values("etbr_strength", ascending=False)
                  .head(10)[show_cols]
            )

    # ------------------------------------------------------------
    # 5) Timeline slice (generic, eyeball sequential vs stateless)
    # ------------------------------------------------------------
    _pp("F) Timeline slice (1 loc / 1 day)")
    if m["_ts"].notna().any():
        loc = m["fault_loc_key"].dropna().astype(str).iloc[0]
        day = m.loc[m["_ts"].notna(), "_ts"].dt.floor("D").iloc[0]

        one = m[(m["fault_loc_key"].astype(str)==loc) &
                (m["_ts"].dt.floor("D")==day)].sort_values("_ts")

        print("picked loc/day:", loc, str(day.date()), "rows:", len(one))

        cols = ["_ts","E_kV_15min","E_event_any","E_comm_fail_any",
                "E_topo_consistency",
                "P_Normal","P_SensorFault","P_CommFault","Decision"]
        cols = [c for c in cols if c in one.columns]

        print(one[cols].head(20))
    else:
        print("SKIP: no valid timestamps")

    # ------------------------------------------------------------
    # 6) HMI cache prereqs (non-fatal)
    # ------------------------------------------------------------
    _pp("G) HMI cache prerequisites")
    for g in ["G_topology","df_day","day_labels","events_all","status_mapping"]:
        print(f"{g}:", "OK" if _exists(g) else "MISSING (ok if not built yet)")

    _pp("VERIFY END ✅")


In [ ]:
merged_15["P_SensorFault"].quantile([0.5,0.9,0.99,0.999])
merged_15.groupby("fault_loc_key")["Decision"].value_counts(normalize=True).head(60)


In [ ]:
# %% 17.99 CACHE EXPORT (ONE-TIME, HMI-CORRECT)
import pickle
import pandas as pd
from pathlib import Path

assert "G_topology" in globals(), "Mangler G_topology. Køyr topologi-del først."
assert "merged_15" in globals(), "Mangler merged_15. Køyr pipeline til merged_15."

CACHE_DIR = Path("hmi_cache")
CACHE_DIR.mkdir(exist_ok=True)

# -------------------------------------------------
# 1) Lagre topologi (uendra)
# -------------------------------------------------
with open(CACHE_DIR / "G_topology.pkl", "wb") as f:
    pickle.dump(G_topology, f)

# -------------------------------------------------
# 2) Bygg df_day: éi rad per (day, Location)
# -------------------------------------------------
TIME_COL = "window_start_15min"
LOC_COL  = "fault_loc_key"
DEC_COL  = "Decision"

POST_COLS = ["P_Normal", "P_SensorFault", "P_CommFault"]

EVID_COLS = [
    "E_kV_15min",
    "E_event_any",
    "E_comm_fail_any",
    "E_topo_consistency",
]

# Ta berre kolonnar som faktisk finst
use_cols = [TIME_COL, LOC_COL, DEC_COL, *POST_COLS]
use_cols += [c for c in EVID_COLS if c in merged_15.columns]

tmp = merged_15[use_cols].copy()

# Tidsreinsing
tmp[TIME_COL] = pd.to_datetime(tmp[TIME_COL], utc=True, errors="coerce")
tmp = tmp.dropna(subset=[TIME_COL, LOC_COL])

# Dag-nivå
tmp["day"] = tmp[TIME_COL].dt.floor("D")

# -------------------------------------------------
# Aggregeringsreglar
# -------------------------------------------------
DEC_ORDER = {"Ignore": 0, "Inspect": 1, "Isolate": 2}

def agg_decision(s):
    # strengaste vedtak den dagen
    return max(s, key=lambda x: DEC_ORDER.get(x, -1))

def agg_e_kv(s):
    # abnormal > unknown > normal
    if "abnormal" in s.values:
        return "abnormal"
    if "absent_or_invalid" in s.values:
        return "absent_or_invalid"
    if "unknown" in s.values:
        return "unknown"
    return "normal"

agg_map = {
    DEC_COL: agg_decision,
    "P_Normal": "mean",
    "P_SensorFault": "mean",
    "P_CommFault": "mean",
}

if "E_event_any" in tmp.columns:
    agg_map["E_event_any"] = "max"

if "E_comm_fail_any" in tmp.columns:
    agg_map["E_comm_fail_any"] = "max"

if "E_kV_15min" in tmp.columns:
    agg_map["E_kV_15min"] = agg_e_kv

if "E_topo_consistency" in tmp.columns:
    agg_map["E_topo_consistency"] = lambda s: (
        "inconsistent" if "inconsistent" in s.values
        else "consistent" if "consistent" in s.values
        else "unknown"
    )

# -------------------------------------------------
# Groupby: KRITISK DEL
# -------------------------------------------------
df_day = (
    tmp
    .groupby(["day", LOC_COL], as_index=False)
    .agg(agg_map)
    .sort_values(["day", LOC_COL])
)

# 3) df_15: fin logg (kun Inspect/Isolate)
#    Inneheld 15-min rader, brukt av /api/log
LOG_COLS = [TIME_COL, LOC_COL, DEC_COL, *POST_COLS]
LOG_COLS += [c for c in ["E_kV_15min","E_event_any","E_comm_fail_any","E_topo_consistency"] if c in merged_15.columns]

df_15 = merged_15[LOG_COLS].copy()
df_15[TIME_COL] = pd.to_datetime(df_15[TIME_COL], utc=True, errors="coerce")
df_15 = df_15.dropna(subset=[TIME_COL, LOC_COL])

# berre beslutningar vi bryr oss om i logg
df_15 = df_15[df_15[DEC_COL].isin(["Inspect","Isolate"])].copy()

df_15.to_parquet(CACHE_DIR / "df_15.parquet", index=False)
print("  - saved:", (CACHE_DIR / "df_15.parquet").name, "rows:", len(df_15))


# -------------------------------------------------
# Lagre
# -------------------------------------------------
df_day.to_parquet(CACHE_DIR / "df_day.parquet", index=False)

print("CACHE OK:", CACHE_DIR.resolve())
print("  - saved:", (CACHE_DIR / "G_topology.pkl").name)
print("  - saved:", (CACHE_DIR / "df_day.parquet").name)
print("  - df_day rows:", len(df_day))
print("  - days:", df_day['day'].nunique(),
      "locations/day (min/mean/max):",
      df_day.groupby("day")[LOC_COL].nunique().agg(["min","mean","max"]).to_dict())


In [ ]:
# %% 17.99b [CACHE EXPORT] comp_state_15.parquet (ALL YEARS, deltas-only; uses events_all + status_mapping)
import pandas as pd
from pathlib import Path

assert "events_all" in globals(), "Mangler events_all. Kjør 7.1B først."
assert "status_mapping" in globals(), "Mangler status_mapping. Kjør 7.1A først."

CACHE_DIR = Path("hmi_cache")
CACHE_DIR.mkdir(exist_ok=True)
OUT_PATH = CACHE_DIR / "comp_state_15.parquet"

ev = events_all.copy()
ev.columns = ev.columns.str.strip()

need = {"timestamp", "Component ID", "Status"}
missing = need - set(ev.columns)
assert not missing, f"events_all mangler {missing}. Har: {list(ev.columns)}"

ev["timestamp"] = pd.to_datetime(ev["timestamp"], utc=True, errors="coerce")
ev = ev.dropna(subset=["timestamp"])

# map Status -> category
m = status_mapping.copy()
m["Status"] = m["Status"].astype(str).str.strip()
ev["Status"] = ev["Status"].astype(str).str.strip()

ev = ev.merge(m, on="Status", how="left")
ev["category"] = ev["category"].fillna("UNKNOWN")

def category_to_state(cat: str) -> str | None:
    c = (cat or "").strip().upper()

    # brytar/operasjon
    if c == "CLOSED":
        return "IN"
    if c in ("OPEN", "TRIPPED"):
        return "OUT"

    # feilindikasjonar (sensor/IDK) -> “PRESENT” (blink)
    if c in ("FAULT_INDICATION", "EARTH_FAULT_IND", "ALARM", "WARNING"):
        return "PRESENT"

    # NORMAL/UNKNOWN gir ingen delta (default handterast i avspelaren)
    return None

ev["component_id"] = ev["Component ID"].astype(str).str.strip()
ev["state"] = ev["category"].map(category_to_state)
ev = ev.dropna(subset=["state"])

ev["ts15"] = ev["timestamp"].dt.floor("15min")

# delta-only: siste state i slot + berre endring
ev = ev.sort_values(["component_id", "ts15", "timestamp"])
d = ev.groupby(["component_id", "ts15"], as_index=False).tail(1)[["ts15","component_id","state"]]

d = d.sort_values(["component_id","ts15"])
d["prev"] = d.groupby("component_id")["state"].shift(1)
deltas = d[d["state"].ne(d["prev"])].drop(columns=["prev"]).reset_index(drop=True)

deltas.to_parquet(OUT_PATH, index=False)

print("OK:", OUT_PATH.resolve())
print("rows:", len(deltas), "components:", deltas["component_id"].nunique())
print("ts15 range:", deltas["ts15"].min(), "->", deltas["ts15"].max())
print(deltas.head(10))


In [ ]:
# %% 17.99b.1 [DIAG] finn dag med state-events (IN/OUT/PRESENT)
import pandas as pd

assert "events_all" in globals()
assert "status_mapping" in globals()

ev = events_all.copy()
ev.columns = ev.columns.str.strip()
ev["timestamp"] = pd.to_datetime(ev["timestamp"], utc=True, errors="coerce")
ev = ev.dropna(subset=["timestamp"])

m = status_mapping.copy()
m["Status"] = m["Status"].astype(str).str.strip()

ev["Status"] = ev["Status"].astype(str).str.strip()
ev = ev.merge(m, on="Status", how="left")
ev["category"] = ev["category"].fillna("UNKNOWN")

interesting = {"CLOSED","OPEN","TRIPPED","FAULT_INDICATION","EARTH_FAULT_IND","ALARM","WARNING"}

ev["day"] = ev["timestamp"].dt.floor("D")
hits = ev[ev["category"].isin(interesting)].groupby("day").size().sort_values(ascending=False)

print("Antall dagar med minst 1 relevant event:", int((hits > 0).sum()))
if len(hits):
    print("Topp 10 dagar (flest relevante events):")
    print(hits.head(10))
    print("Første dag med relevante events:", hits.index[0].date().isoformat())
else:
    print("Ingen relevante events funnet i det heile (sjå mapping/status_mapping).")


In [ ]:
# %% 18.0 HMI CACHE LOAD (AUTO)
import pickle
import pandas as pd
from pathlib import Path

CACHE_DIR = Path("hmi_cache")
G_PATH = CACHE_DIR / "G_topology.pkl"
D_PATH = CACHE_DIR / "df_day.parquet"

if not (G_PATH.exists() and D_PATH.exists()):
    raise FileNotFoundError(
        "Mangler HMI-cache.\n"
        "Køyr først celle 17.99 CACHE EXPORT éin gong (etter at 0–17 er køyrt ferdig).\n"
        f"Forventa filer:\n  - {G_PATH}\n  - {D_PATH}"
    )

with open(G_PATH, "rb") as f:
    G_topology = pickle.load(f)

df_day = pd.read_parquet(D_PATH)
day_labels = sorted(pd.to_datetime(df_day["day"]).dt.strftime("%Y-%m-%d").unique().tolist())

print("HMI cache loaded:")
print("  - G_topology nodes:", G_topology.number_of_nodes(), "edges:", G_topology.number_of_edges())
print("  - df_day rows:", len(df_day), "days:", len(day_labels), "range:", (day_labels[0], day_labels[-1]))


In [ ]:
# %% 18.1 HMI RENDER (REV15 – CACHE-BASED, ROBUST)
%pip -q install pyvis

import re
import math
import pandas as pd
import networkx as nx
from pyvis.network import Network
from pathlib import Path
from collections import deque

assert "G_topology" in globals(), "Mangler G_topology (køyr 18.0)"
assert "df_day" in globals(), "Mangler df_day (køyr 18.0)"
assert "day_labels" in globals(), "Mangler day_labels (køyr 18.0)"

G = G_topology

# -----------------------------
# 1) loc_to_nodes: ROBUST prefix-map (A/B/C/IDK8/...)
# -----------------------------
loc_to_nodes = {}
for n in G.nodes:
    s = str(n)
    prefix = s.split("-", 1)[0]   # A / IDK8
    loc_to_nodes.setdefault(prefix, []).append(n)
    if prefix != s:
        loc_to_nodes.setdefault(s, []).append(n)


# -----------------------------
# 2) STRUCT-subgraf for layout (main/branch)
# -----------------------------
STRUCT_REL = {"main", "branch"}
ATTACH_REL = {"attach"}
DROP_REL   = {"serves"}

def rel(u, v):
    return G[u][v].get("relation")

S = nx.DiGraph()
S.add_nodes_from((n, dict(G.nodes[n])) for n in G.nodes)
for u, v, d in G.edges(data=True):
    if d.get("relation") in STRUCT_REL:
        S.add_edge(u, v, relation=d.get("relation"))

preferred = "A-B1"
if preferred in S.nodes and S.in_degree(preferred) == 0:
    root = preferred
else:
    roots = [n for n in S.nodes if S.in_degree(n) == 0] or [list(S.nodes)[0]]
    root = roots[0]

depth = {root: 0}
q = deque([root])
while q:
    u = q.popleft()
    for v in S.successors(u):
        if v not in depth:
            depth[v] = depth[u] + 1
            q.append(v)
for n in S.nodes:
    depth.setdefault(n, 0)

# -----------------------------
# 3) POS params (tweak spacing)
# -----------------------------
Y_STEP = 150          # meir luft vertikalt
X_STEP = 90           # meir luft horisontalt for parallellar (A-B1 / L1 / etc)

MAIN_X   = 0
BRANCH_X = 360        # flytt branch-lane lenger ut
COMM_X   = -520       # comm-kolonne litt meir venstre
COMM_R_FACTOR = 0.5   # 50 % kortare linje


# fan params (mer spredning)
FAN_R_MIN  = 90
FAN_R_STEP = 14
FAN_ANGLE  = 140
FAN_BASE_X = 520
FAN_KIND_X = {"sensor": 0, "indicator": 40, "earthing": 0}
FORCE_LEFT_FAN_HOSTS = {"C-B4", "L6"}
FAN_BASE_X_LEFT = -FAN_BASE_X   # juster ved behov (negativ = venstre)
kind_map = FAN_KIND_X   # same som høgre

EARTH_R_FACTOR = 0.8   # linje
EARTH_ANGLE    = 60
EARTH_BASE_X   = 180
EARTH_DIAG_ANGLE = -135   # 45° ned til venstre (i grader)


def succ_by(u, relation):
    return sorted([v for v in G.successors(u) if rel(u, v) == relation])

# IDK8-COM skal ikkje i comm-kolonna
def is_special_right_comm(comm_id: str) -> bool:
    cid = str(comm_id)
    return (cid.startswith("IDK8-") or cid.startswith("IDK5-")) and cid.endswith("-COM")


def anchor_for_comm(comm_id: str):
    preds = [p for p in G.predecessors(comm_id) if rel(p, comm_id) == "attach"]
    if preds:
        return preds[0]
    return None

# -----------------------------
# 4) pos: main placement per depth
# -----------------------------
layers = {}
for n in S.nodes:
    layers.setdefault(depth[n], []).append(n)
for d_ in layers:
    layers[d_] = sorted(layers[d_])

pos = {}
for d_, nodes_ in layers.items():
    phys = [n for n in nodes_ if S.nodes[n].get("kind") in ("line", "breaker")]
    phys = sorted(phys)
    k = len(phys) if phys else 1
    for i, n in enumerate(phys):
        offset = (i - (k - 1) / 2) * X_STEP
        pos[n] = {"x": MAIN_X + offset, "y": d_ * Y_STEP}

for n in S.nodes:
    if n not in pos:
        pos[n] = {"x": MAIN_X, "y": depth[n] * Y_STEP}

EXTRA_MAIN_GAPS = {
    ("L3", "D-B1"): 120,   # juster
    ("L6", "G-B1"): 120,   # juster
}

for (u, v), dy in EXTRA_MAIN_GAPS.items():
    if u in pos and v in pos:
        # flytt v og alt nedstrøms v nedover (bevarer rekkefølge)
        base_y = pos[v]["y"]
        for n in pos:
            if pos[n]["y"] >= base_y:
                pos[n]["y"] += dy


# 4.2 T-branches: flytt til høgre, meir y-spread (NESTING-SAFE)
branch_edges = [(u, v) for u, v in S.edges if S[u][v].get("relation") == "branch"]

BRANCH_DX = 360    # kor langt ut kvar branch-hop går
BRANCH_DY = 70     # spreiing

for u, v in branch_edges:
    uy = pos[u]["y"]
    ux = pos[u]["x"]

    siblings = sorted([vv for uu, vv in branch_edges if uu == u])
    idx = siblings.index(v)
    off = (idx - (len(siblings)-1)/2) * BRANCH_DY

    pos[v] = {"x": ux + BRANCH_DX, "y": uy + off}




# 4.3 attachments: comm venstre, resten fan høgre
for host in sorted(G.nodes()):
    host_pos = pos.get(host, {"x": MAIN_X, "y": depth.get(host,0) * Y_STEP})
    hx, hy = host_pos["x"], host_pos["y"]

    atts = succ_by(host, "attach")
    if not atts:
        continue

    comms = [a for a in atts if G.nodes[a].get("kind") == "comm" and not is_special_right_comm(a)]
    oth   = [a for a in atts if a not in comms]

    # comms: venstre kolonne
    for j, n in enumerate(sorted(comms)):
        a = anchor_for_comm(n) or host
        ax = pos.get(a, {"x": hx})["x"]
        ay = pos.get(a, {"y": hy})["y"]

        x = ax + COMM_R_FACTOR * (COMM_X - ax)
        y = ay + j * 22

        pos[n] = {"x": x, "y": y}


    # fan: splitt earthing (nær) og resten (vanleg)
    oth = sorted(oth)
    earth = [n for n in oth if G.nodes[n].get("kind") == "earthing"]
    rest  = [n for n in oth if n not in earth]

    # 4.3a EARTHING: kort "strek" (nær host)
    mE = len(earth)
    if mE:
        R0 = FAN_R_MIN + (max(len(rest), 1)-1) * FAN_R_STEP   # referanse-lengde (sensor-ish)
        RE = max(25, EARTH_R_FACTOR * R0)                     # minst 25 for synlighet
        a0 = -EARTH_ANGLE/2
        a1 = +EARTH_ANGLE/2
        for i, n in enumerate(sorted(earth)):
            rad = math.radians(EARTH_DIAG_ANGLE)
            x = hx + RE * math.cos(rad)
            y = hy + RE * math.sin(rad)
            pos[n] = {"x": x, "y": y}

    # 4.3b RESTEN: vanleg vifte (sensor/indicator/etc)
    m = len(rest)
    if m:
        R = FAN_R_MIN + (m-1) * FAN_R_STEP
        a0 = -FAN_ANGLE/2
        a1 = +FAN_ANGLE/2

        # vel fan-side per host (MÅ stå her)
        use_left = (str(host) in FORCE_LEFT_FAN_HOSTS)
        base_x = FAN_BASE_X_LEFT if use_left else FAN_BASE_X
        kind_map = FAN_KIND_X
        R = R * (1.35 if use_left else 1.0)   # lengre streker for C-sensorene (juster 1.2–1.8)


        for i, n in enumerate(sorted(rest)):
            kind = G.nodes[n].get("kind", "unknown")
            kind_dx = kind_map.get(kind, 0)
            ang = 0.0 if m == 1 else (a0 + (a1 - a0) * (i/(m-1)))
            rad = math.radians(ang)
            dx = R * math.cos(rad)
            if use_left:
                dx = -dx

            x = base_x + 0.25*hx + kind_dx + dx
            y = hy + R*math.sin(rad)
            
            pos[n] = {"x": x, "y": y}




# failsafe
for n in G.nodes:
    if n not in pos:
        kind = G.nodes[n].get("kind", "unknown")
        if kind == "comm":
            pos[n] = {"x": COMM_X, "y": depth.get(n,0)*Y_STEP}
        else:
            pos[n] = {"x": MAIN_X, "y": depth.get(n,0)*Y_STEP}

# -----------------------------
# 5) PyVis: nodes + edges
# -----------------------------
net = Network(height="950px", width="100%", directed=True, notebook=True, cdn_resources="remote")
net.toggle_physics(False)

net.set_options(r"""
{
  "nodes": {
    "font": {"size": 16},
    "shapeProperties": { "useBorderWithImage": true },
    "borderWidthSelected": 6
  },
  "edges": { "smooth": false, "arrows": { "to": { "enabled": true, "scaleFactor": 0.6 } } },
  "interaction": { "hover": true }
}
""")





KIND_BORDER = {
    "breaker":  "#273c75",
    "line":     "#353b48",
    "sensor":   "#0097e6",
    "indicator":"#8c7ae6",
    "earthing": "#44bd32",
    "comm":     "#c23616",
    "unknown":  "#718093",
    "scope":    "#718093",
}

DEFAULT_BG = "#dcdde1"

# Earthing icon (SVG) – “jordingssymbol”
EARTH_SVG = (
    "data:image/svg+xml;utf8,"
    "<svg xmlns='http://www.w3.org/2000/svg' width='80' height='80' viewBox='0 0 80 80'>"
    "<line x1='40' y1='8' x2='40' y2='36' stroke='%232c3e50' stroke-width='6'/>"
    "<line x1='18' y1='42' x2='62' y2='42' stroke='%232c3e50' stroke-width='6'/>"
    "<line x1='24' y1='54' x2='56' y2='54' stroke='%232c3e50' stroke-width='6'/>"
    "<line x1='30' y1='66' x2='50' y2='66' stroke='%232c3e50' stroke-width='6'/>"
    "</svg>"
)

KIND_SHAPE = {
    "breaker": "box",
    "line": "ellipse",
    "sensor": "dot",
    "indicator": "diamond",
    "comm": "triangle",
    # earthing blir image
    "unknown": "ellipse",
    "scope": "ellipse",
}

node_border = {n: KIND_BORDER.get(G.nodes[n].get("kind","unknown"), "#718093") for n in G.nodes}

for n in G.nodes:
    kind = G.nodes[n].get("kind", "unknown")
    if kind == "earthing":
        net.add_node(
            n, label=str(n),
            x=float(pos[n]["x"]), y=float(pos[n]["y"]),
            fixed=True,
            shape="image",
            image=EARTH_SVG,
            size=28,
            borderWidth=2,
            shapeProperties={"useBorderWithImage": True},
            color={"background": DEFAULT_BG, "border": node_border[n]},
            title=f"{n} ({kind})",
        )
    else:
        net.add_node(
            n, label=str(n),
            x=float(pos[n]["x"]), y=float(pos[n]["y"]),
            fixed=True,
            shape=KIND_SHAPE.get(kind, "ellipse"),
            color={"background": DEFAULT_BG, "border": node_border[n]},
            title=f"{n} ({kind})",
        )

# Edge-fargar:
# - STRUCT (main/branch) = svart
# - ATTACH (til sensor/indicator/comm/earthing) = grøn (kun når target er "attachment-kind")
EDGE_BLACK = "#2f3640"
EDGE_GREEN = "#44bd32"

ATTACH_KINDS = {"sensor", "indicator", "comm", "earthing"}

for u, v, d in G.edges(data=True):
    r = d.get("relation")
    if r in DROP_REL:
        continue
    if r not in (STRUCT_REL | ATTACH_REL):
        continue

    v_kind = G.nodes[v].get("kind", "unknown")

    if r in STRUCT_REL:
        ecolor = EDGE_BLACK
    else:
        # attach: berre grøn dersom vi faktisk går til "attachment-kind"
        ecolor = EDGE_GREEN if v_kind in ATTACH_KINDS else EDGE_BLACK

    net.add_edge(str(u), str(v), title=str(r), color={"color": ecolor}, width=2)


# -----------------------------
# 6) Sanitize (JSON-safe)
# -----------------------------
def _json_safe(x):
    if isinstance(x, set): return list(x)
    if isinstance(x, tuple): return list(x)
    return x

net.edges = [
    {k: (_json_safe(v) if not isinstance(v, (dict, list)) else v) for k, v in e.items() if k != "smooth"}
    for e in net.edges
]
net.nodes = [
    {k: (_json_safe(v) if not isinstance(v, (dict, list)) else v) for k, v in n.items()}
    for n in net.nodes
]

# -----------------------------
# 7) UI + JS (regex inject)
# -----------------------------
DEC_BG = {"Ignore": "#44bd32", "Inspect": "#fbc531", "Isolate": "#e84118"}

UI = f"""
<div id="topBar" style="padding:10px; display:flex; flex-wrap:wrap; gap:18px; align-items:flex-end; border-bottom:1px solid #ddd;">
  <div style="display:flex; gap:10px; align-items:center;">
    <button id="prevBtn">◀</button>
    <button id="playBtn">Play</button>
    <button id="nextBtn">▶</button>
    <input id="slider" type="range"
       min="0" max="{max(len(day_labels)-1,0)}"
       value="0" style="width:420px;">
    <span id="label" style="font-family:monospace"></span>

    <div style="padding-left:12px; border-left:1px solid #eee;">
      <div style="font-size:12px; color:#666;">View</div>
      <input id="speedMs" type="number" value="2000" min="200" step="100" style="width:90px;">
      <select id="viewMode">
        <option value="decision" selected>Decision</option>
        <option value="evidence">Evidence</option>
        <option value="posterior">Posterior-view</option>
      </select>
    </div>
  </div>

  <div style="display:flex; gap:10px; align-items:flex-end; padding-left:10px; border-left:1px solid #eee;">
    <div>
      <div style="font-size:12px; color:#666;">Location</div>
      <input id="inj_loc" style="width:120px;" placeholder="A / B / C / IDK2 ...">
    </div>
    <div>
      <div style="font-size:12px; color:#666;">Component</div>
      <select id="comp_id" style="width:160px;">
        <option value="">(select)</option>
      </select>
    </div>
    <div>
      <div style="font-size:12px; color:#666;">State</div>
      <select id="comp_state">
        <option value="IN">IN</option>
        <option value="OUT">OUT</option>
        <option value="PRESENT">PRESENT</option>
        <option value="NONE">NONE</option>
        <option value="UP">UP</option>
        <option value="DOWN">DOWN</option>
      </select>
    </div>
    <button id="setCompBtn">Set component</button>
    <div id="setOut" style="font-family:monospace; font-size:12px; color:#444; padding-left:8px;"></div>
  </div>
</div>

<div style="padding:10px;">
  <div style="display:flex; align-items:center; justify-content:space-between; gap:10px;">
    <div style="font-family:monospace;">Decisions this day</div>
    <button id="togglePanelBtn">Hide</button>
  </div>

  <div id="decisionPanel" class="decisionPanel">

    <div class="decisionTableHeadWrap">
      <table id="decisionTableHead" class="decisionTable">
        <colgroup>
          <col style="width:12%">
          <col style="width:12%">
          <col style="width:40%">
          <col style="width:12%">
          <col style="width:12%">
          <col style="width:12%">
        </colgroup>
        <thead>
          <tr>
            <th class="th-left">Location</th>
            <th class="th-left">Decision</th>
            <th class="th-left">Reason</th>
            <th class="th-right">P(N)</th>
            <th class="th-right">P(S)</th>
            <th class="th-right">P(C)</th>
          </tr>
        </thead>
      </table>
    </div>

    <div class="decisionTableBodyWrap">
      <table id="decisionTableBody" class="decisionTable">
        <colgroup>
          <col style="width:12%">
          <col style="width:12%">
          <col style="width:40%">
          <col style="width:12%">
          <col style="width:12%">
          <col style="width:12%">
        </colgroup>
        <tbody></tbody>
      </table>
    </div>

  </div>
</div>
"""


JS = r"""
<script>
const DEFAULT_BG = "#dcdde1";

let timer = null;
let playing = false;

let frameK = 0;        // legacy (ikkje brukt til event-playback)

// --- event-driven timeline ---
let timelineKs = [0];  // unike k-verdier (0..95) for vald dag
let stepIdx = 0;       // indeks inn i timelineKs
let timelineWhy = {};  // optional debug: why[k] = [...]



function setLabel(txt){
  const el = document.getElementById("label");
  if (el) el.textContent = txt || "";
}

function getPlaybackDayLabel(){
  const lbl = (document.getElementById("label")?.textContent || "").trim();
  const day = lbl.split(" ")[0]; // "2018-03-03"
  return (/^\d{4}-\d{2}-\d{2}$/.test(day)) ? day : null;
}



function applyUpdates(updates){
  if (!updates) return;

  const patched = updates.map(u => {
    const st = (u.state ? String(u.state).toUpperCase() : "");

    let out = { ...u };
    if (!out.color) out.color = {};
    if (typeof out.color !== "object") out.color = {};

    const bg0 =
      (out.color && typeof out.color === "object" && out.color.background)
        ? out.color.background
        : null;

    out.color = { ...out.color, background: (bg0 || DEFAULT_BG) };

    // --- kind lookup (to treat earthing/comm specially) ---
    const cur = (typeof nodes !== "undefined" && u.id != null) ? nodes.get(u.id) : null;

    let kind = "";
    if (cur && cur.kind) kind = String(cur.kind).toLowerCase();
    if (!kind && cur && typeof cur.title === "string"){
      if (cur.title.includes("(earthing)")) kind = "earthing";
      else if (cur.title.includes("(breaker)")) kind = "breaker";
      else if (cur.title.includes("(comm)")) kind = "comm";
      else if (cur.title.includes("(sensor)")) kind = "sensor";
      else if (cur.title.includes("(indicator)")) kind = "indicator";
    }

    // COMM-objekter
    if (kind === "comm"){
      if (st === "PRESENT"){
        out.color.border = "#fbc531";   // gul
        out.borderWidth = 4;
        out.className = "blink-yellow";
      } else {
        out.color.border = "#718093";   // nøytral grå
        out.borderWidth = 1;
        out.className = "";
      }
      return out;
    }


    // --- EARTHING rules ---
    if (kind === "earthing" && st === "OUT"){
      const existingBorder =
        (cur && cur.color && typeof cur.color === "object" && cur.color.border)
          ? cur.color.border
          : (out.color.border || "#718093");

      out.color.border = existingBorder;
      out.borderWidth = 1;
      out.className = "";
      return out;
    }

    if (kind === "earthing" && st === "IN"){
      out.color.border = "#e84118";
      out.borderWidth = 4;
      out.className = "";
      return out;
    }

    // --- default for alle andre komponentar ---
    if (st === "OUT"){
      out.color.border = "#e84118";
      out.borderWidth = 8;
      out.className = "blink-red";
      return out;
    }

    if (st === "IN"){
      out.color.border = "#44bd32";
      out.borderWidth = 3;
      out.className = "";
      return out;
    }

    // isolate-lokasjon: rød ramme
    const isolateLocs = window.__isolateLocs || new Set();
    const idStr = String(u.id || "");
    const letbrrefix = idStr.includes("-") ? idStr.split("-",1)[0] : idStr;

    if (isolateLocs.has(letbrrefix)){
      out.color.border = "#e84118";
      out.borderWidth = 6;
    }

    // blink for PRESENT (typisk comm)
    if (st === "PRESENT"){
      out.className = "blink-yellow";
    } else {
      out.className = "";
    }

    return out;
  });

  nodes.update(patched);
}


function setComponentOptions(components){
  const sel = document.getElementById("comp_id");
  if (!sel) return;

  const prev = sel.value;

  sel.innerHTML = "";
  const placeholder = document.createElement("option");
  placeholder.value = "";
  placeholder.textContent = "(select)";
  sel.appendChild(placeholder);

  (components || []).forEach(c => {
    const opt = document.createElement("option");
    opt.value = c.id;
    opt.textContent = `${c.id} (${c.kind})`;
    sel.appendChild(opt);
  });

  // restore hvis mulig
  if (prev && [...sel.options].some(o => o.value === prev)){
    sel.value = prev;
  }
}




async function refreshComponentsForLocation(){
  const loc = (document.getElementById("inj_loc").value || "").trim();
  if (!loc){
    setComponentOptions([]);
    return;
  }


  try{
    const resp = await fetch(`/api/components?location=${encodeURIComponent(loc)}`);
    if (!resp.ok) return;
    const data = await resp.json();
    if (data && data.components) setComponentOptions(data.components);
  }catch(e){ /* silent */ }
}


function resetAll(){
  const all = nodes.get();
  const upd = all.map(n => {
    const b = (n.color && typeof n.color === "object" && n.color.border) ? n.color.border : "#718093";
    return {
      id: n.id,
      color: { background: DEFAULT_BG, border: b },
      borderWidth: 1,
      className: ""   // viktig: fjerner blink
    };
  });
  nodes.update(upd);
}




let seenKeys = new Set();
let lastDayLabel = null;
let historyMode = true; // sett false om du vil ha "kun dagens"

function clearHistory(){
  const tb = document.querySelector("#decisionTableBody tbody");
  if (tb) tb.innerHTML = "";
  seenKeys = new Set();
  lastDayLabel = null;

  // (valfritt) scroll til topp når du “clearar”
  const wrap = document.querySelector(".decisionTableBodyWrap");
  if (wrap) wrap.scrollTop = 0;
}

function renderDecisionTable(rows, dayLabel){
  const tb = document.querySelector("#decisionTableBody tbody");
  if (!tb) return;

  // normal (ikkje-history): same som før
  if (!historyMode){
    tb.innerHTML = "";
    seenKeys = new Set();
    lastDayLabel = null;
  }

  if (historyMode && dayLabel && dayLabel !== lastDayLabel){
    const hdr = document.createElement("tr");
    hdr.className = "day-row";   // ← DETTE er linja eg meinte
    hdr.innerHTML = "<td colspan='6'>" + dayLabel + "</td>";


   tb.appendChild(hdr);

   const spacer = document.createElement("tr");
   spacer.className = "spacer-row";
   spacer.innerHTML = `<td colspan="6"></td>`;
   tb.appendChild(spacer);
    lastDayLabel = dayLabel;
  }


  if (!rows || rows.length === 0) return;

  // berre Inspect/Isolate
  const filt = rows.filter(r =>
    (r.Decision === "Inspect" || r.Decision === "Isolate") ||
    String(r.Reason || "").startsWith("MANUAL_OVERRIDE")
  );
  if (filt.length === 0) return;


  // sort for stabilitet

  filt.sort((a,b) =>
  (a.Reason || "").localeCompare(b.Reason || "") ||
  (a.Location || "").localeCompare(b.Location || "")
  );


  for (const r of filt){
    const loc = r.Location || "";
    const dec = r.Decision || "";
    const reason = r.Reason || "";
    const key = `${dayLabel}__${loc}__${dec}__${reason}`;

    if (historyMode && seenKeys.has(key)) continue;
    seenKeys.add(key);

    const pN = (r.P_Normal ?? 0).toFixed(3);
    const pS = (r.P_SensorFault ?? 0).toFixed(3);
    const pC = (r.P_CommFault ?? 0).toFixed(3);

    const tr = document.createElement("tr");
    tr.style.cursor = "pointer";
    tr.style.background = (dec === "Isolate") ? "#fdecea" : "#fff8e1";

    const decBadge =
      dec === "Isolate" ? "badge badge-isolate" :
      dec === "Inspect" ? "badge badge-inspect" :
      "badge badge-ignore";

    const reasonBadge =
      reason.startsWith("MANUAL_OVERRIDE") ? "badge badge-post" :
      reason === "COMM_FAIL" ? "badge badge-comm" :
      reason === "EVENT_PRESENT" ? "badge badge-event" :
      reason === "KV_ABNORMAL" ? "badge badge-kv" :
    "badge badge-post";


    tr.innerHTML = `
      <td style="padding:4px; border-bottom:1px solid #f3f3f3;">${loc}</td>
      <td style="padding:4px; border-bottom:1px solid #f3f3f3;">
        <span class="${decBadge}">${dec}</span>
      </td>
      <td style="padding:4px; border-bottom:1px solid #f3f3f3;">
        <span class="${reasonBadge}">${reason}</span>
      </td>
      <td style="padding:4px; border-bottom:1px solid #f3f3f3; text-align:right;">${pN}</td>
      <td style="padding:4px; border-bottom:1px solid #f3f3f3; text-align:right;">${pS}</td>
      <td style="padding:4px; border-bottom:1px solid #f3f3f3; text-align:right;">${pC}</td>
    `;

    tr.onclick = async () => {
      // toggle detail row
      if (tr.nextSibling && tr.nextSibling.classList.contains("detail-row")) {
        tr.nextSibling.remove();
        return;
      }

      const detail = document.createElement("tr");
      detail.className = "detail-row";

      const e_kv = (r.E_kV_15min != null) ? r.E_kV_15min : "unknown";
      const e_ev = (r.E_event_any != null) ? r.E_event_any : 0;
      const e_cm = (r.E_comm_fail_any != null) ? r.E_comm_fail_any : 0;
      const e_tp = (r.E_topo_consistency != null) ? r.E_topo_consistency : "unknown";

      // optional: hent 15-min log frå /api/log
      let logHtml = "";

      // dropp /api/log for MANUAL_OVERRIDE (dei er ikkje i df_15 på same måte)
      const reasonStr = String(r.Reason ?? r.reason ?? "");
      const isManual = reasonStr.startsWith("MANUAL_OVERRIDE");


      if (!isManual){

        const day = (dayLabel && /^\d{4}-\d{2}-\d{2}$/.test(dayLabel))
          ? dayLabel
          : (getPlaybackDayLabel() || "");

        if (!day){
          logHtml = "<br><i>No day context for /api/log (dayLabel missing)</i>";
        } else {
          try{
            const resp = await fetch(`/api/log?location=${encodeURIComponent(loc)}&day=${encodeURIComponent(day)}`);
            if (resp.ok){
              const data = await resp.json();
              if (data.rows && data.rows.length){
                logHtml = "<br><b>15-min log (Inspect/Isolate):</b><br>" +
                  data.rows.slice(0, 12)
                    .map(x => `${x.t} — ${x.Decision} (E_comm=${x.E_comm_fail_any}, E_event=${x.E_event_any}, E_kV=${x.E_kV_15min})`)
                    .join("<br>");
                if (data.rows.length > 12) logHtml += `<br><i>... (${data.rows.length} rows)</i>`;
              } else {
                logHtml = "<br><i>No 15-min Inspect/Isolate entries</i>";
              }
            }
          }catch(e){ /* silent */ }
        }

      } else {
        logHtml = "<br><i>Manual override – no /api/log lookup.</i>";
      }



      const hyp = (r.hypothesis != null && String(r.hypothesis).trim() !== "") ? r.hypothesis : "unknown";
      const target = Array.isArray(r.target_components) ? r.target_components : [];
      const allComps = Array.isArray(r.components) ? r.components : [];

      const targetHtml = target.length
        ? target.map(x => `${x.id} (${x.kind})`).join(", ")
        : "<i>Heile lokasjon (ingen spesifikk komponent peika ut)</i>";

      const allHtml = allComps.length
        ? allComps.map(x => `${x.id} (${x.kind})`).join(", ")
        : "<i>Ingen komponentliste</i>";


      detail.innerHTML = `
        <td colspan="6" style="padding:6px 10px; background:#fafafa;
                               border-left:4px solid ${dec === "Isolate" ? "#e84118" : "#fbc531"};">
          <b>What to inspect:</b> ${targetHtml}<br>                     
          <b>All components in location:</b> ${allHtml}<br><br>                     
          <b>Interpretation:</b> hypothesis=<b>${hyp}</b><br>                     
          <b>Evidence:</b>
          E_kV_15min=<b>${e_kv}</b>,
          E_event_any=<b>${e_ev}</b>,
          E_comm_fail_any=<b>${e_cm}</b>,
          E_topo_consistency=<b>${e_tp}</b>
          ${logHtml}
        </td>
      `;
      tr.after(detail);
    };

    tb.appendChild(tr);
  }

  // auto-scroll til botn i history mode (etter at nye rader er lagt til)
  if (historyMode){
    const wrap = document.querySelector(".decisionTableBodyWrap");
    if (wrap) wrap.scrollTop = wrap.scrollHeight;
  }
}


async function loadFrame(dayIdx, k){
  const resp = await fetch(`/api/frame?day_idx=${dayIdx}&k=${k}`);
  if (!resp.ok){
    setLabel("frame API error");
    return;
  }
  const data = await resp.json();
  window.__current_ts15 = data.ts15;
  setLabel(`${data.label_day}  k=${data.k}  ${data.ts15}  [state]`);
  const w = (timelineWhy && timelineWhy[String(k)]) ? timelineWhy[String(k)].join(", ") : "";
  if (w) setLabel(`${data.label_day}  k=${data.k}  ${data.ts15}  [state]  :: ${w}`);
  applyUpdates(data.updates);
}

async function loadTimeline(i){
  try{
    const resp = await fetch(`/api/timeline?day_idx=${i}`);
    if (!resp.ok) return { ks:[0], why:{} };
    const data = await resp.json();
    return {
      ks: (data.ks && data.ks.length) ? data.ks : [0],
      why: data.why || {}
    };
  }catch(e){
    return { ks:[0], why:{} };
  }
}


async function loadDay(i){

  frameK = 0;
  const s = document.getElementById("slider");
  if (s) s.value = i;

  const tl = await loadTimeline(i);
  timelineKs = tl.ks;
  timelineWhy = tl.why || {};
  stepIdx = 0;

  const modeSel = document.getElementById("viewMode");
  const mode = modeSel ? modeSel.value : "decision";

  const resp = await fetch(`/api/day/${i}?mode=${encodeURIComponent(mode)}`);
  if (!resp.ok){
    setLabel("API error");
    return;
  }

  const data = await resp.json();
  setLabel(`${data.label}  [${data.mode}]`);


  window.__isolateLocs = new Set(data.isolate_locations || []);

  resetAll();
  applyUpdates(data.updates);
  renderDecisionTable(data.rows, data.label);

  const sel = document.getElementById("comp_id");
  if (sel && sel.options.length <= 1){
    await refreshComponentsForLocation();
  }

  // start på første faktiske hendelse for dagen
  const k0 = timelineKs[Math.max(0, Math.min(stepIdx, timelineKs.length-1))] || 0;
  await loadFrame(i, k0);


  
}


async function step(){
  const s = document.getElementById("slider");
  let dayIdx = parseInt(s.value, 10);
  const maxDay = parseInt(s.max, 10);

  // 1) Har vi fleire events igjen i SAME dag?
  if (timelineKs && stepIdx < timelineKs.length){
    const k = timelineKs[stepIdx];
    await loadFrame(dayIdx, k);
    stepIdx += 1;
    return;
  }

  // 2) Finn NESTE DAG MED EVENTS
  for (let i = dayIdx + 1; i <= maxDay; i++){
    try{
      const r = await fetch(`/api/timeline?day_idx=${i}`);
      if (!r.ok) continue;
      const d = await r.json();

      // ekte events = meir enn berre [0]
      if (d.ks && d.ks.length > 1){
        await loadDay(i);      // resetter stepIdx=0 internt
        const k0 = d.ks[0];
        await loadFrame(i, k0);
        stepIdx = 1;
        return;
      }
    }catch(e){ /* ignore */ }
  }

  // 3) Ingen fleire events i heile datasettet
  stopPlaying();
}






function stopPlaying(){
  playing = false;
  if (timer) clearInterval(timer);
  timer = null;
  const btn = document.getElementById("playBtn");
  if (btn) btn.textContent = "Play";
}






function clampDay(i){
  const s = document.getElementById("slider");
  const max = parseInt(s.max);
  return Math.max(0, Math.min(i, max));
}

async function gotoDay(i){
  await loadDay(clampDay(i));
}

async function prevDay(){
  stopPlaying();
  const s = document.getElementById("slider");
  await gotoDay(parseInt(s.value) - 1);
}

async function nextDay(){
  stopPlaying();
  const s = document.getElementById("slider");
  await gotoDay(parseInt(s.value) + 1);
}

async function skipBackEvent(){
  stopPlaying();
  if (!timelineKs || timelineKs.length === 0) return;

  stepIdx = Math.max(0, stepIdx - 1);

  const s = document.getElementById("slider");
  const dayIdx = parseInt(s.value);

  const k = timelineKs[stepIdx] || 0;
  await loadFrame(dayIdx, k);
}

async function skipFwdEvent(){
  stopPlaying();
  if (!timelineKs || timelineKs.length === 0) return;

  stepIdx = Math.min(timelineKs.length - 1, stepIdx + 1);

  const s = document.getElementById("slider");
  const dayIdx = parseInt(s.value);

  const k = timelineKs[stepIdx] || 0;
  await loadFrame(dayIdx, k);
}



document.getElementById("playBtn").onclick = function () {
  playing = !playing;
  this.textContent = playing ? "Pause" : "Play";

  if (playing){
    const ms = parseInt(document.getElementById("speedMs")?.value || "2500", 10);
    timer = setInterval(step, Math.max(200, ms));
  } else {
    if (timer) clearInterval(timer);
    timer = null;
  }
};



document.getElementById("prevBtn").onclick = prevDay;
document.getElementById("nextBtn").onclick = nextDay;

document.getElementById("slider").oninput = async (e) => {
  await loadDay(parseInt(e.target.value));
};

document.getElementById("viewMode").onchange = async () => {
  const s = document.getElementById("slider");
  await loadDay(parseInt(s.value));
};

document.getElementById("togglePanelBtn").onclick = function(){
  const p = document.getElementById("decisionPanel");
  if (!p) return;
  const hidden = (p.style.display === "none");
  p.style.display = hidden ? "block" : "none";
  this.textContent = hidden ? "Hide" : "Show";
};

document.getElementById("inj_loc").addEventListener("change", refreshComponentsForLocation);
document.getElementById("inj_loc").addEventListener("keyup", (e) => {
  if (e.key === "Enter") refreshComponentsForLocation();
});

document.getElementById("inj_loc").addEventListener("input", () => {
  // liten debounce så du ikke spammer API
  clearTimeout(window.__locTimer);
  window.__locTimer = setTimeout(refreshComponentsForLocation, 150);
});



document.getElementById("setCompBtn").onclick = async () => {
  const out = document.getElementById("setOut");

  const loc = (document.getElementById("inj_loc").value || "").trim();
  if (!loc){
    out.textContent = "ERROR: set Location first";
    return;
  }

  const cid = document.getElementById("comp_id").value;
  if (!cid){
    out.textContent = "ERROR: choose component first (set Location and wait dropdown)";
    return;
  }

  const st = document.getElementById("comp_state").value;

  // ✅ Bruk eksakt ts15 frå playback (sett i loadFrame)
  const time = window.__current_ts15 || new Date().toISOString();

  out.textContent = "Setting component...";

  const payload = { fault_loc_key: loc, timestamp: time, component_id: cid, state: st };

  let resp;
  try {
    resp = await fetch(`/api/set_component_state`, {
      method: "POST",
      headers: {"Content-Type":"application/json"},
      body: JSON.stringify(payload)
    });
  } catch (e) {
    out.textContent = "ERROR: network / server not reachable";
    return;
  }

  const raw = await resp.text();
  let data = null;
  try { data = JSON.parse(raw); } catch(e) {}

  if (!resp.ok){
    out.textContent = "ERROR: " + (data?.detail || raw);
    return;
  }

  // 1) oppdater graf (override-farge)
  if (data.updates) applyUpdates(data.updates);

  // 2) vis agentresultat + interlock i setOut
  const dec = (data && data.Decision != null) ? data.Decision : "(no decision)";
  const pn = (data && typeof data.P_Normal === "number") ? data.P_Normal.toFixed(3) : "NA";
  const ps = (data && typeof data.P_SensorFault === "number") ? data.P_SensorFault.toFixed(3) : "NA";
  const pc = (data && typeof data.P_CommFault === "number") ? data.P_CommFault.toFixed(3) : "NA";
  const ae = (data && data.agent_error) ? `  agent_error=${data.agent_error}` : "";

  const il = (data && data.topo_illegal_interlock != null) ? data.topo_illegal_interlock : "NA";
  const ei = (data && data.E_interlock != null) ? data.E_interlock : "NA";

  out.textContent =
    `OK: ${data.component_id}=${data.state} @ ${data.ts15}  ` +
    `Decision=${dec}  Pn=${pn}  Ps=${ps}  Pc=${pc}  ` +
    `interlock=${il} E_interlock=${ei}` + ae;

  // 3) legg i tabellen som ei “live”-rad som kan klikkast (bruk renderDecisionTable)
  //    Krev at backend returnerer `row` (data.row)
  if (data && data.row){
    const day = getPlaybackDayLabel() || (data.ts15 ? data.ts15.slice(0,10) : "");
    renderDecisionTable([data.row], day);

    // auto-scroll
    const wrap = document.querySelector(".decisionTableBodyWrap");
    if (wrap) wrap.scrollTop = wrap.scrollHeight;
  }

  await refreshComponentsForLocation();
};



(async () => {
  const s = document.getElementById("slider");
  if (!s || parseInt(s.max) < 0){
    setLabel("No days");
    return;
  }

  // start på første dag som faktisk har event (fallback = 0)
  let startIdx = 0;
  try{
    const r = await fetch("/api/timeline_first");
    if (r.ok){
      const d = await r.json();
      if (typeof d.day_idx === "number") startIdx = d.day_idx;
    }
  }catch(e){ /* ignore */ }

  await loadDay(startIdx);

})();

</script>
"""


html = net.generate_html()

CSS = """
<style>

/* --- PAGE / CANVAS BACKGROUND --- */
html, body{
  background:#f2f3f5;
}

/* sjølve nettverksflata */
#mynetwork{
  background:#f2f3f5;
}

/* panel/kort */
.decisionPanel,
.decisionTableHeadWrap,
.decisionTableBodyWrap{
  background:#ffffff;
}

/* --- DECISION PANEL --- */
.decisionPanel{
  border:1px solid #e5e5e5;
  border-radius:8px;
  padding:0;
  box-shadow:0 1px 2px rgba(0,0,0,0.06);
}

/* --- TOP BAR (sticky, stabil layout) --- */
#topBar{
  position: sticky;
  top: 0;
  z-index: 9999;

  display: flex;
  flex-wrap: wrap;
  align-items: flex-end;
  gap: 18px;

  background:#ffffff;
  border-radius:8px;
  box-shadow:0 1px 2px rgba(0,0,0,0.06);
  padding: 8px 10px;
  margin-bottom:10px;
}

/* Første blokk: knappar + slider + label */
#topBar > div:first-child{
  display: flex;
  align-items: center;
  gap: 10px;

  flex: 1 1 600px;     /* kan krympe, men har god basis */
  min-width: 520px;
}

/* Label: fyller resten av linja, kuttar med ... */
#label{
  flex: 1 1 auto;
  min-width: 0;        /* KRITISK for ellipsis i flex */
  white-space: nowrap;
  overflow: hidden;
  text-overflow: ellipsis;
  font-family: monospace;
}

/* Injeksjonsfelt: alltid heilt til høgre */
#topBar > div:last-child{
  margin-left: auto;
  display: flex;
  gap: 10px;
  align-items: flex-end;
}

/* --- DECISION TABLE --- */
.decisionTableHeadWrap{
  border-bottom:1px solid #eee;
  padding:8px;
}

.decisionTableBodyWrap{
  max-height:220px;
  overflow:auto;
  padding:0 8px 8px 8px;
  resize: vertical;
}

.decisionTable{
  width:100%;
  table-layout:fixed;
  border-collapse:separate;
  border-spacing:0;
  font-family:monospace;
  font-size:12px;
}

.decisionTable th,
.decisionTable td{
  padding:4px;
  border-bottom:1px solid #f3f3f3;
}

.th-left{ text-align:left; }
.th-right{ text-align:right; }

.decisionTable tr.day-row td{
  background:#f7f7f7;
  font-weight:bold;
  font-size:11px;
  color:#444;
  border-top:1px solid #e0e0e0;
}

.decisionTable tr.spacer-row td{
  height:6px;
  padding:0;
  border:0;
  background:#fff;
}

/* --- BADGES --- */
.badge{
  padding:2px 6px;
  border-radius:4px;
  font-size:11px;
  font-weight:bold;
  color:#222;
}
.badge-isolate{ background:#e84118; color:white; }
.badge-inspect{ background:#fbc531; }
.badge-ignore{ background:#44bd32; color:white; }

.badge-comm{ background:#e84118; color:white; }
.badge-event{ background:#fbc531; }
.badge-kv{ background:#00a8ff; color:white; }
.badge-post{ background:#7f8fa6; color:white; }

/* --- BLINK EFFECTS --- */
@keyframes blinkRed {
  0%   { box-shadow: 0 0 0 0 rgba(232,65,24,0.0); }
  50%  { box-shadow: 0 0 0 6px rgba(232,65,24,0.35); }
  100% { box-shadow: 0 0 0 0 rgba(232,65,24,0.0); }
}
.blink-red{
  animation: blinkRed 0.9s infinite;
  border-radius: 10px;
}

@keyframes blinkYellow {
  0%   { box-shadow: 0 0 0 0 rgba(251,197,49,0.0); }
  50%  { box-shadow: 0 0 0 6px rgba(251,197,49,0.45); }
  100% { box-shadow: 0 0 0 0 rgba(251,197,49,0.0); }
}
.blink-yellow{
  animation: blinkYellow 0.9s infinite;
  border-radius: 10px;
}

</style>
"""


html = html.replace("</head>", CSS + "</head>")



# INJECT UI robustly before mynnetwork div (handles class="card-body")
html = re.sub(r'(<div[^>]*id="mynetwork"[^>]*></div>)', UI + r"\1", html, count=1, flags=re.I)
html = html.replace("</body>", JS + "</body>")

out = Path("HMI_REV15_api.html")
out.write_text(html, encoding="utf-8")

print("FERDIG:", out.resolve())
print("Days:", len(day_labels), " example:", day_labels[:3])

# Export globals for 18.2
HMI__df_day = df_day.copy()
HMI__day_labels = day_labels
HMI__loc_to_nodes = loc_to_nodes
HMI__node_border = node_border
HMI__html_path = out.resolve()


In [ ]:
# %% 18.2 HMI API SERVER (REV15 – CACHE-BASED, MODE + PANEL + COMPONENTS)

from pathlib import Path
import pandas as pd
from fastapi import FastAPI, Query
from fastapi.responses import HTMLResponse, JSONResponse, PlainTextResponse
from pydantic import BaseModel
import uvicorn

# --- REQUIRES (from 18.1) ---
assert "HMI__df_day" in globals(), "Mangler HMI__df_day. Køyr 18.1."
assert "HMI__day_labels" in globals(), "Mangler HMI__day_labels. Køyr 18.1."
assert "HMI__loc_to_nodes" in globals(), "Mangler HMI__loc_to_nodes. Køyr 18.1."
assert "HMI__node_border" in globals(), "Mangler HMI__node_border. Køyr 18.1."
assert "HMI__html_path" in globals(), "Mangler HMI__html_path. Køyr 18.1."
assert "G_topology" in globals(), "Mangler G_topology. Køyr 18.0/18.1."

HMI_HTML_PATH = Path(HMI__html_path)
assert HMI_HTML_PATH.exists(), f"Mangler {HMI_HTML_PATH}. Køyr 18.1 og sjekk filnamn."

# --- Color maps ---
DEC_BG = {"Ignore": "#44bd32", "Inspect": "#fbc531", "Isolate": "#e84118"}

POST_BG = {
    "Normal":      "#00a8ff",
    "SensorFault": "#9c88ff",
    "CommFault":   "#f368e0",
}

EVID_BG = {
    "comm":  "#e84118",
    "event": "#fbc531",
    "kv":    "#00a8ff",
    "none":  "#7f8fa6",
}

STATE_BG = {
    "IN":      "#44bd32",
    "OUT":     "#dcdde1",
    "PRESENT": "#fbc531",
    "NONE":    "#dcdde1",
    "UP":      "#44bd32",
    "DOWN":    "#e84118",
}



app = FastAPI()

# --- HMI synthetic component-state overrides (in-memory) ---
# key: (ts15_iso, fault_loc_key, component_id) -> {"kind":..., "state":...}
HMI_OVERRIDES = {}

# ------------------------------------------------------------
# HELPER: returner noder for eksakt komponent-id eller prefix
#  - "C-B4" -> kun noder under "C-B4" (eksakt)
#  - "C"    -> alle noder under "C" (prefix)
# ------------------------------------------------------------
def _nodes_for_key(key_raw: str):
    key_raw = (key_raw or "").strip()
    if not key_raw:
        return []

    # eksakt komponent-id: C-B4 => berre den noden (viss den finst)
    if "-" in key_raw:
        return HMI__loc_to_nodes.get(key_raw, [])

    # prefix: C => alle noder under C (som 18.1 har bygd)
    return HMI__loc_to_nodes.get(key_raw, [])

def _component_override_updates(ts15: str, loc_prefix: str, cid: str, kind: str, state: str):
    state = (state or "").strip().upper()
    kind = (kind or "unknown").strip().lower()

    bg = STATE_BG.get(state, "#fbc531")
    border = HMI__node_border.get(cid, "#718093")

    # sterkare border når IN/OUT (tydeleg i HMI)
    bw = 2
    if state == "IN":
        bw = 6
        border = "#44bd32"
    elif state == "OUT":
        bw = 6
        border = "#e84118"

    title = (
        f"<b>Component override</b><br>"
        f"ts15: {ts15}<br>"
        f"loc: {loc_prefix}<br>"
        f"id: <b>{cid}</b> ({kind})<br>"
        f"state: <b>{state}</b>"
    )

    # NOTE: vis.js image-shape treng dette for å teikne border “rundt”
    extra = {}
    if kind == "earthing":
        extra["shapeProperties"] = {"useBorderWithImage": True}
        # (valfritt) litt større node når den er IN
        if state == "IN":
            extra["size"] = 34

    if cid in G_topology.nodes:
        return [{
            "id": cid,
            "color": {"background": bg, "border": border},
            "borderWidth": bw,
            "state": state,
            "title": title,
            **extra
        }]

    # Fallback: farg loc-noder
    nodes = _nodes_for_key(loc_prefix)
    return [{
        "id": nid,
        "color": {"background": bg, "border": HMI__node_border.get(nid, "#718093")},
        "borderWidth": bw,
        "title": title,
        "state": state,
    } for nid in nodes]





class SetComponentPayload(BaseModel):
    fault_loc_key: str
    timestamp: str
    component_id: str
    state: str  # e.g. "IN", "OUT", "PRESENT", "NONE", "UP", "DOWN"


class InjectPayload(BaseModel):
    fault_loc_key: str
    timestamp: str

    # Defaults => ikkje required i request body
    E_kV: str = "unknown"      # normal/abnormal/unknown
    E_event: str = "none"      # none/present
    E_comm: str = "none"       # none/present
    E_topo: str = "unknown"    # unknown/consistent/inconsistent



def _components_for_loc(key_raw: str):
    ids = _nodes_for_key(key_raw)
    return [{"id": nid, "kind": G_topology.nodes[nid].get("kind", "unknown")} for nid in ids]



def _day_updates(i: int, mode: str = "decision"):
    labels = HMI__day_labels
    if i < 0 or i >= len(labels):
        return {"label": "out-of-range", "mode": mode, "updates": [], "rows": []}

    label = labels[i]
    day_ts = pd.Timestamp(label, tz="UTC")

    df_d = HMI__df_day[HMI__df_day["day"] == day_ts]
    updates = []
    rows = []

    isolate_locs = sorted(set(str(x.get("fault_loc_key","")) for _, x in df_d.iterrows()
        if str(x.get("Decision","")) == "Isolate"))
    



    mode = (mode or "decision").strip().lower()
    if mode not in ("decision", "posterior", "evidence"):
        mode = "decision"

    day_start = pd.Timestamp(label, tz="UTC")
    day_end = day_start + pd.Timedelta(days=1)

    for _, r in df_d.iterrows():
        # Lokasjon i df_day er fortsatt fault_loc_key internt
        loc = str(r.get("fault_loc_key", ""))
        dec = str(r.get("Decision", "unknown"))

        pN = float(r.get("P_Normal", 0.0))
        pS = float(r.get("P_SensorFault", 0.0))
        pC = float(r.get("P_CommFault", 0.0))

        # --- vel bakgrunnsfarge ---
        if mode == "decision":
            bg = DEC_BG.get(dec, "#7f8fa6")

        elif mode == "posterior":
            hyp = max(
                [("Normal", pN), ("SensorFault", pS), ("CommFault", pC)],
                key=lambda x: x[1]
            )[0]
            bg = POST_BG.get(hyp, "#7f8fa6")

        else:  # evidence
            # Krever at desse kolonnane finst i df_day for å gi meining
            comm = int(r.get("E_comm_fail_any", 0)) if "E_comm_fail_any" in df_d.columns else 0
            evnt = int(r.get("E_event_any", 0)) if "E_event_any" in df_d.columns else 0

            kv_raw = r.get("E_kV_15min", None) if "E_kV_15min" in df_d.columns else None
            kv_abn = 1 if (str(kv_raw).lower() in ("abnormal", "absent_or_invalid")) else 0

            if comm == 1:
                bg = EVID_BG["comm"]
            elif evnt == 1:
                bg = EVID_BG["event"]
            elif kv_abn == 1:
                bg = EVID_BG["kv"]
            else:
                bg = EVID_BG["none"]

        # Robust node lookup: exact, else prefix
        nodes = _nodes_for_key(loc)

        for node_id in nodes:
            kind_n = G_topology.nodes[node_id].get("kind", "unknown")
            tooltip_n = (
                f"<b>Node: {node_id}</b> ({kind_n})<br>"
                f"<b>Location: {loc}</b><br>"
                f"Day: {label}<br>"
                f"View: <b>{mode}</b><br>"
                f"Decision: <b>{dec}</b><br><br>"
                f"P(Normal): {pN:.3f}<br>"
                f"P(SensorFault): {pS:.3f}<br>"
                f"P(CommFault): {pC:.3f}"
            )

            updates.append({
                "id": node_id,
                "color": {"background": bg, "border": HMI__node_border.get(node_id, "#718093")},
                "title": tooltip_n,
            })


        # components til tabell/panel
        components = _components_for_loc(loc)

        
        # --- evidens (robust: tolererer manglande kolonnar) ---
        E_kV = r.get("E_kV_15min", None) if "E_kV_15min" in df_d.columns else None
        E_event = int(r.get("E_event_any", 0)) if "E_event_any" in df_d.columns else 0
        E_comm  = int(r.get("E_comm_fail_any", 0)) if "E_comm_fail_any" in df_d.columns else 0
        E_topo  = r.get("E_topo_consistency", None) if "E_topo_consistency" in df_d.columns else None

        E_kV_s   = "unknown" if E_kV is None else str(E_kV)
        E_topo_s = "unknown" if E_topo is None else str(E_topo)

        # --- enkel forklaring (prioritet = faktisk agent-logikk) ---
        reason = "POSTERIOR/UTILITY"
        if E_comm == 1:
            reason = "COMM_FAIL"
        elif E_event == 1:
            reason = "EVENT_PRESENT"
        elif str(E_topo_s).lower() == "inconsistent":
            reason = "TOPO_INCONSISTENT"
        elif str(E_kV_s).lower() in ("abnormal", "absent_or_invalid"):
            reason = "KV_ABNORMAL"

        # --- dominant hypothesis (til forklaring) ---
        hypothesis = max(
            [("Normal", pN), ("SensorFault", pS), ("CommFault", pC)],
            key=lambda x: x[1]
        )[0]

        # --- target components (kva bør inspiserast) ---
        # Enkle heuristikkar basert på "Reason"
        if reason == "COMM_FAIL":
            target = [c for c in components if c.get("kind") == "comm"]
        elif reason == "EVENT_PRESENT":
            # typisk status-/indikatorspesifikt
            target = [c for c in components if c.get("kind") in ("indicator", "breaker")]
        elif reason == "KV_ABNORMAL":
            target = [c for c in components if c.get("kind") in ("sensor",)]
        elif reason == "TOPO_INCONSISTENT":
            target = [c for c in components if c.get("kind") in ("breaker", "earthing")]
        else:
            target = []  # fallback (vis heller "heile lokasjon")


        rows.append({
            "Location": loc,
            "Decision": dec,
            "Reason": reason,
            "hypothesis": hypothesis,
            "target_components": target,

            # evidensgrunnlag
            "E_kV_15min": E_kV_s,
            "E_event_any": int(E_event),
            "E_comm_fail_any": int(E_comm),
            "E_topo_consistency": E_topo_s,

            # posterior
            "P_Normal": pN,
            "P_SensorFault": pS,
            "P_CommFault": pC,

            # topologisk kontekst (ikkje årsak)
            "components": components,
        })
    # ------------------------------------------------------------
    # OVERRIDES overlay (latch framover innan dagen)
    # ------------------------------------------------------------
    latest_by_key = {}
    for (t_iso, loc_ov, cid), ov in list(HMI_OVERRIDES.items()):
        t = pd.to_datetime(t_iso, utc=True, errors="coerce")
        if t is pd.NaT:
            continue
        if not (day_start <= t < day_end):
            continue
        key = (str(loc_ov).split("-", 1)[0], str(cid))
        prev = latest_by_key.get(key)
        if (prev is None) or (t > prev["t"]):
            latest_by_key[key] = {"t": t, "t_iso": t_iso, "ov": ov}

    for (loc_prefix, cid), pack in latest_by_key.items():
        ov = pack["ov"]
        kind = ov.get("kind", "unknown")
        state = ov.get("state", "UNKNOWN")

        updates += _component_override_updates(pack["t_iso"], loc_prefix, cid, kind, state)

        rows.append({
            "Location": loc_prefix,
            "Decision": "Inspect",
            "Reason": f"MANUAL_OVERRIDE {cid}={state} @ {pack['t_iso']}",
            "hypothesis": "manual",
            "target_components": [{"id": cid, "kind": kind}],
            "E_kV_15min": "unknown",
            "E_event_any": 0,
            "E_comm_fail_any": 0,
            "E_topo_consistency": "unknown",
            "P_Normal": 0.0,
            "P_SensorFault": 0.0,
            "P_CommFault": 0.0,
            "components": _components_for_loc(loc_prefix),
        })




    return {"label": label, "mode": mode, "updates": updates, "rows": rows,
            "isolate_locations": isolate_locs}

DF15_PATH = Path("hmi_cache") / "df_15.parquet"
DF15 = None
if DF15_PATH.exists():
    DF15 = pd.read_parquet(DF15_PATH)

# --- 15-min component-state deltas (events -> IN/OUT/PRESENT), deltas-only ---
STATE_PATH = Path("hmi_cache") / "comp_state_15.parquet"
DF_STATE = None
if STATE_PATH.exists():
    DF_STATE = pd.read_parquet(STATE_PATH)
    DF_STATE["ts15"] = pd.to_datetime(DF_STATE["ts15"], utc=True, errors="coerce")
    DF_STATE = DF_STATE.dropna(subset=["ts15", "component_id", "state"]).copy()
    DF_STATE["component_id"] = DF_STATE["component_id"].astype(str).str.strip()
    DF_STATE["state"] = DF_STATE["state"].astype(str).str.strip().str.upper()
    DF_STATE = DF_STATE.sort_values(["component_id", "ts15"])
    print("OK: loaded DF_STATE:", len(DF_STATE), "rows,", DF_STATE["component_id"].nunique(), "components")
else:
    print("WARN: missing", STATE_PATH, "(run 17.99b)")

@app.get("/api/timeline")
def api_timeline(day_idx: int = Query(...)):
    labels = HMI__day_labels
    if day_idx < 0 or day_idx >= len(labels):
        return JSONResponse({"error": "day_idx out of range"}, status_code=400)

    day_label = labels[day_idx]
    day0 = pd.Timestamp(day_label, tz="UTC")
    day1 = day0 + pd.Timedelta(days=1)

    # Fallback: viss DF_STATE manglar, returner berre k=0
    if DF_STATE is None or DF_STATE.empty:
        return JSONResponse({
            "label_day": day_label,
            "ks": [0],
            "why": {"0": ["(no DF_STATE)"]}
        })

    # Hent alle delta-hendelser for dagen
    df = DF_STATE[(DF_STATE["ts15"] >= day0) & (DF_STATE["ts15"] < day1)].copy()
    if df.empty:
        return JSONResponse({
            "label_day": day_label,
            "ks": [0],
            "why": {"0": ["(no events)"]}
        })

    # k = 0..95 per dag
    df["k"] = ((df["ts15"] - day0) / pd.Timedelta(minutes=15)).astype(int)
    df = df[(df["k"] >= 0) & (df["k"] <= 95)]

    # Unike ks i kronologisk rekkefølge
    ks = sorted(set([0] + df["k"].unique().tolist()))
    if not ks:
        ks = [0]

    # "why": kort forklaring per k (maks 12 linjer)
    why = {}
    # sort for stabilitet
    df = df.sort_values(["k", "component_id", "state"])
    for k, g in df.groupby("k"):
        msgs = []
        for _, r in g.head(12).iterrows():
            cid = str(r.get("component_id", "")).strip()
            st  = str(r.get("state", "")).strip().upper()
            if cid:
                msgs.append(f"{cid} {st}")
        why[str(int(k))] = msgs

    return JSONResponse({
        "label_day": day_label,
        "ks": ks,
        "why": why
    })

@app.get("/api/timeline_first")
def api_timeline_first():
    labels = HMI__day_labels
    if DF_STATE is None or DF_STATE.empty:
        return JSONResponse({"day_idx": 0, "label_day": labels[0], "note": "no DF_STATE"})

    days_with = set(
        DF_STATE["ts15"].dt.strftime("%Y-%m-%d").unique().tolist()
    )

    for i, lab in enumerate(labels):
        if lab in days_with:
            return JSONResponse({"day_idx": i, "label_day": lab})

    return JSONResponse({"day_idx": 0, "label_day": labels[0], "note": "no events in label range"})


@app.get("/api/log")
def api_log(location: str = Query(...), day: str = Query(...)):
    if DF15 is None:
        return JSONResponse({"error": "df_15.parquet manglar – køyr 17.99 igjen"}, status_code=400)

    # parse day til UTC-day range
    d0 = pd.Timestamp(day, tz="UTC")
    d1 = d0 + pd.Timedelta(days=1)

    loc = location.strip()
    df = DF15.copy()
    s = df["fault_loc_key"].astype(str)

    if "-" in loc:
        # eksakt komponent-id (C-B4)
        df_loc = df[s.eq(loc)].copy()
    else:
        # prefix (C) => C og C-*
        df_loc = df[s.eq(loc) | s.str.startswith(loc + "-")].copy()

    if df_loc.empty:
        return JSONResponse({"day": day, "location": loc, "n": 0, "rows": []})


    # filter day range
    t = pd.to_datetime(df_loc["window_start_15min"], utc=True, errors="coerce")
    df_loc = df_loc[(t >= d0) & (t < d1)].copy()
    df_loc["window_start_15min"] = t[(t >= d0) & (t < d1)]

    # sort + return
    df_loc = df_loc.sort_values("window_start_15min")

    rows = []
    for _, r in df_loc.iterrows():
        rows.append({
            "t": r["window_start_15min"].isoformat(),
            "Location": str(r.get("fault_loc_key","")),
            "Decision": str(r.get("Decision","")),
            "P_Normal": float(r.get("P_Normal",0.0)),
            "P_SensorFault": float(r.get("P_SensorFault",0.0)),
            "P_CommFault": float(r.get("P_CommFault",0.0)),
            "E_kV_15min": str(r.get("E_kV_15min","unknown")),
            "E_event_any": int(r.get("E_event_any",0)) if "E_event_any" in df_loc.columns else 0,
            "E_comm_fail_any": int(r.get("E_comm_fail_any",0)) if "E_comm_fail_any" in df_loc.columns else 0,
            "E_topo_consistency": str(r.get("E_topo_consistency","unknown")),
        })

    return JSONResponse({"day": day, "location": loc, "n": len(rows), "rows": rows})


def _default_state_for_kind(kind: str) -> str | None:
    k = (kind or "unknown").strip().lower()
    if k == "breaker":
        return "IN"
    if k == "earthing":
        return "OUT"
    # sensor/indicator/comm: default = None (ingen highlight)
    return None


def _current_state_map(ts15: pd.Timestamp, loc_prefix: str):
    """
    component_id -> {"kind": kind, "state": state}
    Gjeldande state ved ts15 for lokasjon (prefix), prioritet:
      defaults -> DF_STATE latch -> HMI_OVERRIDES latch
    """
    loc_prefix = str(loc_prefix).split("-", 1)[0].strip()

    # noder i lokasjonen (best-effort) + fallback prefix-match
    loc_nodes = _nodes_for_key(loc_prefix) or []
    loc_set = set(str(x).strip() for x in loc_nodes)

    def _in_loc(cid: str) -> bool:
        cid = str(cid).strip()
        return (cid in loc_set) or (cid == loc_prefix) or cid.startswith(loc_prefix + "-")

    cur = {}

    # 1) defaults
    for cid in G_topology.nodes:
        if not _in_loc(cid):
            continue
        kind = (G_topology.nodes[cid].get("kind", "unknown") or "").strip().lower()
        st0 = _default_state_for_kind(kind)
        if st0:
            cur[cid] = {"kind": kind, "state": st0}

    # 2) DF_STATE latch (men ikkje comm)
    if DF_STATE is not None and not DF_STATE.empty:
        df_before = DF_STATE[DF_STATE["ts15"].le(ts15)]
        if not df_before.empty:
            last = (
                df_before.sort_values(["component_id", "ts15"])
                        .groupby("component_id", as_index=False)
                        .tail(1)
            )
            for _, r in last.iterrows():
                cid = str(r["component_id"]).strip()
                if cid not in G_topology.nodes or (not _in_loc(cid)):
                    continue

                kind = (G_topology.nodes[cid].get("kind", "unknown") or "").strip().lower()
                if kind == "comm":
                    continue  # same regel som /api/frame

                st = str(r["state"]).strip().upper()
                cur[cid] = {"kind": kind, "state": st}

    # 3) overrides latch (overstyr alt)
    latest = {}
    for (t_iso, l, cid), ov in HMI_OVERRIDES.items():
        if str(l).split("-", 1)[0].strip() != loc_prefix:
            continue
        cid = str(cid).strip()
        if cid not in G_topology.nodes or (not _in_loc(cid)):
            continue

        t = pd.to_datetime(t_iso, utc=True, errors="coerce")
        if t is pd.NaT or t > ts15:
            continue

        prev = latest.get(cid)
        if prev is None or t > prev["t"]:
            latest[cid] = {"t": t, "ov": ov}

    for cid, pack in latest.items():
        ov = pack["ov"]
        kind = (ov.get("kind", "unknown") or "").strip().lower()
        st = str(ov.get("state", "UNKNOWN")).strip().upper()
        cur[cid] = {"kind": kind, "state": st}

    return cur


def _derive_topo_illegal_interlock(ts15_iso: str, loc: str):
    """
    Returnerer 1 dersom (earthing IN) og (breaker IN) i same lokasjon ved ts15
    basert på gjeldande state (defaults + DF_STATE latch + overrides).
    Returnerer None ellers (one-sided).
    """
    ts15 = pd.to_datetime(ts15_iso, utc=True, errors="coerce")
    if ts15 is pd.NaT:
        return None

    loc_prefix = str(loc).split("-", 1)[0].strip()
    if not loc_prefix:
        return None

    cur = _current_state_map(ts15, loc_prefix) or {}

    earth_in = False
    breaker_in = False

    for v in cur.values():
        kind = str(v.get("kind", "")).strip().lower()
        st   = str(v.get("state", "")).strip().upper()

        if kind == "earthing" and st == "IN":
            earth_in = True
        elif kind == "breaker" and st == "IN":
            breaker_in = True

        if earth_in and breaker_in:
            return 1

    return None



def _derive_interlock_agg(ts15_iso: str):
    """
    Tel kor mange lokasjonar (prefix) som har illegal interlock ved ts15.
    Returnerer: (count:int, bucket:str in {"none","few","many"})
    """
    ts15 = pd.to_datetime(ts15_iso, utc=True, errors="coerce")
    if ts15 is pd.NaT:
        return 0, "none"

    # alle "lokasjon-prefix" som faktisk finst i topologien
    prefixes = sorted(set(str(n).split("-", 1)[0] for n in G_topology.nodes))

    cnt = 0
    for loc in prefixes:
        til = _derive_topo_illegal_interlock(ts15_iso, loc)
        if til == 1:
            cnt += 1

    if cnt == 0:
        bucket = "none"
    elif cnt <= 2:
        bucket = "few"
    else:
        bucket = "many"

    return cnt, bucket




def _run_agent_once(payload: InjectPayload):
    """
    Brukar agentfunksjonane om dei finst. Elles feilar vi pent.
    """
    required = ["posterior_update", "decision_update", "likelihood", "TOPO_LH", "AGENT_PRIORS", "UTILS_BY_LOC"]
    missing = [x for x in required if x not in globals()]
    if missing:
        raise RuntimeError(
            "Inject krev agentceller (16.14–16.16). Manglar: " + ", ".join(missing)
        )

    ts = pd.to_datetime(payload.timestamp, utc=True, errors="coerce")
    if ts is pd.NaT:
        raise ValueError(f"Ugyldig timestamp: {payload.timestamp}")

    E_kV_15min = (payload.E_kV or "unknown").strip().lower()
    if E_kV_15min not in ("normal", "abnormal", "unknown", "present", "absent_or_invalid"):
        E_kV_15min = "unknown"

    E_event_any = 1 if (str(payload.E_event).strip().lower() == "present") else 0
    E_comm_fail_any = 1 if (str(payload.E_comm).strip().lower() == "present") else 0
    E_topo = (payload.E_topo or "unknown").strip().lower()
    if E_topo not in ("unknown", "consistent", "inconsistent"):
        E_topo = "unknown"

    loc_raw = str(payload.fault_loc_key).strip()
    loc_agent = loc_raw.split("-", 1)[0]   # agent kjøyrer på prefix (C / IDK8 / ...)

    one = pd.DataFrame([{
        "window_start_15min": ts.floor("15min"),
        "fault_loc_key": loc_agent,
        "E_kV_15min": E_kV_15min,
        "E_event_any": int(E_event_any),
        "E_comm_fail_any": int(E_comm_fail_any),
        "E_topo_consistency": E_topo,
    }])

    # --- derived interlock evidence (local + aggregated) ---
    ts15_iso = one.loc[0, "window_start_15min"].isoformat()
    loc = loc_agent  # overrides lagres på prefix

    # local (per lokasjon)
    one.loc[0, "E_interlock"] = "none"
    one.loc[0, "topo_illegal_interlock"] = 0

    til = _derive_topo_illegal_interlock(ts15_iso, loc)
    if til == 1:
        one.loc[0, "topo_illegal_interlock"] = 1
        one.loc[0, "E_interlock"] = "present"

    # aggregated (på tvers av alle lokasjonar)
    cnt, bucket = _derive_interlock_agg(ts15_iso)
    one.loc[0, "interlock_count"] = int(cnt)
    one.loc[0, "E_interlock_agg"] = str(bucket)   # "none" / "few" / "many"



    one = posterior_update(one, likelihood_dict=likelihood, topo_lh=TOPO_LH, priors=AGENT_PRIORS, mask=None)
    one = decision_update(one, utils_by_loc=UTILS_BY_LOC, mask=None)

    r = one.iloc[0]
    return {
        "Decision": str(r["Decision"]),
        "P_Normal": float(r["P_Normal"]),
        "P_SensorFault": float(r["P_SensorFault"]),
        "P_CommFault": float(r["P_CommFault"]),
        "topo_illegal_interlock": int(one.get("topo_illegal_interlock", pd.Series([0])).iloc[0] or 0),
        "E_interlock": str(r.get("E_interlock", "none")),


    }




# --- ROUTES ---

@app.get("/", response_class=HTMLResponse)
def root():
    return HTMLResponse(HMI_HTML_PATH.read_text(encoding="utf-8"))


@app.get("/api/day/{i}")
def api_day(i: int, mode: str = Query("decision")):
    return JSONResponse(_day_updates(i, mode=mode))

@app.get("/api/frame")
def api_frame(day_idx: int = Query(...), k: int = Query(...), mode: str = Query("state")):
    labels = HMI__day_labels
    if day_idx < 0 or day_idx >= len(labels):
        return JSONResponse({"error": "day_idx out of range"}, status_code=400)

    k = int(k)
    if k < 0 or k > 95:
        return JSONResponse({"error": "k must be 0..95"}, status_code=400)

    day_label = labels[day_idx]
    day0 = pd.Timestamp(day_label, tz="UTC")
    ts15 = day0 + pd.Timedelta(minutes=15 * k)

    # 1) start med default for breakers/earthing (kun lokasjonar som faktisk finst i topologien)
    # (vi sender berre updates for desse typane, resten er default "ingen highlight")
    updates = []
    if k == 0:
        for nid in G_topology.nodes:
            kind = G_topology.nodes[nid].get("kind", "unknown")
            st0 = _default_state_for_kind(kind)
            if st0:
                updates.append({"id": nid, "state": st0})


    # 2) appliser deltas for ts15 (override defaults)
    #    - breakers/earthing: LATCHE (siste state før/lik ts15)
    #    - comm: IKKE latche (vis berre PRESENT på eksakt ts15)
    if DF_STATE is not None:

        # A) eksakte eventer i denne slot'en (brukes for COM blink)
        hit_exact = DF_STATE[DF_STATE["ts15"].eq(ts15)]
        hit_exact = hit_exact[["component_id", "state"]] if not hit_exact.empty else None

        # B) latch for alt UNNTATT comm
        df_before = DF_STATE[DF_STATE["ts15"].le(ts15)]
        if not df_before.empty:
            last = (
                df_before.sort_values(["component_id", "ts15"])
                        .groupby("component_id", as_index=False)
                        .tail(1)
            )

            for _, r in last.iterrows():
                cid = str(r["component_id"]).strip()
                st  = str(r["state"]).strip().upper()
                if cid not in G_topology.nodes:
                    continue

                kind = (G_topology.nodes[cid].get("kind", "unknown") or "").strip().lower()
                if kind == "comm":
                    continue   # COM skal ikkje latche

                updates.append({"id": cid, "state": st})

        # C) COM blink berre på eksakt ts15 (typisk PRESENT ved FEIL)
        if hit_exact is not None:
            for _, r in hit_exact.iterrows():
                cid = str(r["component_id"]).strip()
                st  = str(r["state"]).strip().upper()
                if cid not in G_topology.nodes:
                    continue

                kind = (G_topology.nodes[cid].get("kind", "unknown") or "").strip().lower()
                if kind == "comm" and st == "PRESENT":
                    updates.append({"id": cid, "state": "PRESENT"})



    # 3) overlay manuelle overrides (latch framover)
    ts15_iso = ts15.isoformat()

    # finn siste override per (loc,cid) med t <= ts15
    latest_by_key = {}
    for (t_iso, loc_ov, cid), ov in list(HMI_OVERRIDES.items()):
        t = pd.to_datetime(t_iso, utc=True, errors="coerce")
        if t is pd.NaT or t > ts15:
            continue
        key = (str(loc_ov).split("-", 1)[0], str(cid))
        prev = latest_by_key.get(key)
        if (prev is None) or (t > prev["t"]):
            latest_by_key[key] = {"t": t, "t_iso": t_iso, "ov": ov}

    for (loc_prefix, cid), pack in latest_by_key.items():
        ov = pack["ov"]
        kind = ov.get("kind", "unknown")
        state = ov.get("state", "UNKNOWN")
        updates += _component_override_updates(pack["t_iso"], loc_prefix, cid, kind, state)


    seen = set()
    dedup = []
    for u in reversed(updates):
        uid = str(u.get("id", "")).strip()
        if not uid:
            continue
        if uid in seen:
            continue
        seen.add(uid)
        dedup.append(u)
    updates = list(reversed(dedup))



    # 4) konverter state-only updates til full update (vi bruker JS applyUpdates til å sette rammer)
    # minimum: id + state
    # (JS din fargar IN/OUT med ramme uansett)
    return JSONResponse({
        "label_day": day_label,
        "k": k,
        "ts15": ts15.isoformat(),
        "updates": updates,
        "rows": []  # kan fylle agent rows seinare
    })



@app.get("/api/decisions")
def api_decisions(i: int = 0, j: int | None = None):
    labels = HMI__day_labels
    if not labels:
        return JSONResponse({"range": [0, 0], "days": []})

    if j is None:
        j = i
    i = max(0, int(i))
    j = min(len(labels) - 1, int(j))
    if j < i:
        i, j = j, i

    out = []
    for k in range(i, j + 1):
        d = _day_updates(k, mode="decision")
        out.append({"label": d["label"], "rows": d["rows"]})
    return JSONResponse({"range": [i, j], "days": out})

@app.post("/api/set_component_state")
def api_set_component_state(p: SetComponentPayload):
    ts = pd.to_datetime(p.timestamp, utc=True, errors="coerce")
    if ts is pd.NaT:
        return PlainTextResponse(f"Ugyldig timestamp: {p.timestamp}", status_code=400)

    ts15_dt = ts.floor("15min")
    ts15 = ts15_dt.isoformat()

    loc = str(p.fault_loc_key).strip()
    loc = loc.split("-", 1)[0]  # alltid prefix for overrides

    cid = str(p.component_id).strip()
    state = str(p.state).strip().upper()

    # kind frå topologi (hvis finst)
    kind = "unknown"
    if cid in G_topology.nodes:
        kind = (G_topology.nodes[cid].get("kind", "unknown") or "unknown")

    # lagre override
    HMI_OVERRIDES[(ts15, loc, cid)] = {"kind": kind, "state": state}

    # graf-oppdatering for denne komponenten
    updates = _component_override_updates(ts15, loc, cid, kind, state)

    # -----------------------------
    # Agent: bruk HISTORISK evidens
    # -----------------------------
    agent = {"Decision": None, "P_Normal": None, "P_SensorFault": None, "P_CommFault": None}
    agent_err = None
    row_for_table = None

    try:
        if "merged_15" not in globals():
            raise RuntimeError("merged_15 finst ikkje i kernel (køyr agent/merge-celler før 18.2)")

        df = merged_15

        # loc-prefix frå historikk
        loc_series = (
            df["fault_loc_key"]
            .astype(str)
            .str.split("-", n=1)
            .str[0]
        )

        row = df.loc[
            (pd.to_datetime(df["window_start_15min"], utc=True, errors="coerce") == ts15_dt) &
            (loc_series == loc)
        ].head(1)

        if row.empty:
            raise RuntimeError(f"Fann ingen historikk-rad for ts15={ts15} loc={loc}")

        r0 = row.iloc[0]

        inject_payload = InjectPayload(
            fault_loc_key=loc,
            timestamp=ts15,
            E_kV=str(r0.get("E_kV_15min", "unknown")),
            E_event="present" if int(r0.get("E_event_any", 0) or 0) == 1 else "none",
            E_comm="present" if int(r0.get("E_comm_fail_any", 0) or 0) == 1 else "none",
            E_topo=str(r0.get("E_topo_consistency", "unknown")),
        )

        agent = _run_agent_once(inject_payload)

        # bygg ei rad som ser ut som dei andre i decision-tabellen (for klikk + detaljer)
        pN = float(agent.get("P_Normal", 0.0) or 0.0)
        pS = float(agent.get("P_SensorFault", 0.0) or 0.0)
        pC = float(agent.get("P_CommFault", 0.0) or 0.0)

        E_kV_s = str(r0.get("E_kV_15min", "unknown"))
        E_event = int(r0.get("E_event_any", 0) or 0)
        E_comm  = int(r0.get("E_comm_fail_any", 0) or 0)
        E_topo_s = str(r0.get("E_topo_consistency", "unknown"))

        reason = "POSTERIOR/UTILITY"
        if E_comm == 1:
            reason = "COMM_FAIL"
        elif E_event == 1:
            reason = "EVENT_PRESENT"
        elif str(E_topo_s).lower() == "inconsistent":
            reason = "TOPO_INCONSISTENT"
        elif str(E_kV_s).lower() in ("abnormal", "absent_or_invalid"):
            reason = "KV_ABNORMAL"

        # same target-heuristikk som i _day_updates
        comps = _components_for_loc(loc)
        if reason == "COMM_FAIL":
            target = [c for c in comps if c.get("kind") == "comm"]
        elif reason == "EVENT_PRESENT":
            target = [c for c in comps if c.get("kind") in ("indicator", "breaker")]
        elif reason == "KV_ABNORMAL":
            target = [c for c in comps if c.get("kind") in ("sensor",)]
        elif reason == "TOPO_INCONSISTENT":
            target = [c for c in comps if c.get("kind") in ("breaker", "earthing")]
        else:
            target = []

        hyp = max([("Normal", pN), ("SensorFault", pS), ("CommFault", pC)], key=lambda x: x[1])[0]

        row_for_table = {
            "Location": loc,
            "Decision": str(agent.get("Decision", "")),
            "Reason": f"MANUAL_OVERRIDE {cid}={state} @ {ts15} :: {reason}",
            "hypothesis": hyp,
            "target_components": target,
            "components": comps,
            "E_kV_15min": E_kV_s,
            "E_event_any": E_event,
            "E_comm_fail_any": E_comm,
            "E_topo_consistency": E_topo_s,
            "P_Normal": pN,
            "P_SensorFault": pS,
            "P_CommFault": pC,
            "ts15": ts15,
        }

    except Exception as e:
        agent_err = f"{type(e).__name__}: {e}"

    return JSONResponse({
        "ok": True,
        "ts15": ts15,
        "loc": loc,
        "component_id": cid,
        "kind": kind,
        "state": state,
        "updates": updates,
        **agent,
        "agent_error": agent_err,
        "row": row_for_table,   # <-- viktig: UI kan legge denne i tabellen
    })









@app.post("/api/inject")
def api_inject(payload: InjectPayload):
    try:
        res = _run_agent_once(payload)
    except Exception as e:
        return PlainTextResponse(f"{type(e).__name__}: {e}", status_code=400)

    dec = res["Decision"]
    bg = DEC_BG.get(dec, "#7f8fa6")

    loc = str(payload.fault_loc_key).strip()
    nodes = _nodes_for_key(loc)


    updates = [{
        "id": nid,
        "color": {"background": bg, "border": HMI__node_border.get(nid, "#718093")},
        "title": f"<b>Inject</b><br>Key: {loc}<br>Decision: <b>{dec}</b><br>"
    } for nid in nodes]

    return JSONResponse({**res, "updates": updates})

@app.get("/api/components")
def api_components(location: str = Query(...)):
    loc = (location or "").strip()
    comps = _components_for_loc(loc)
    # valfritt: filtrer vekk scope/unknown om du vil
    # comps = [c for c in comps if c["kind"] not in ("scope",)]
    return JSONResponse({"location": loc, "components": comps})



# --- START SERVER (bytt port om 10048) ---
PORT = 8001
print("SERVER STARTA:")
print(f"  Opne i nettlesar: http://127.0.0.1:{PORT}")
print("  Slider: hentar /api/day/{i}?mode=...")
print("  Inject: POST /api/inject")

config = uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="warning")
server = uvicorn.Server(config)
await server.serve()


In [ ]:
# ============================================================
# KODESJEKK: List alle kodecelle-IDar (# X.Y[a] [TAG])
# ============================================================

import nbformat
import re
from pathlib import Path

# Finn notebook-fila (standard i Jupyter)
nb_path = Path.cwd()
nb_files = list(nb_path.glob("*.ipynb"))
assert len(nb_files) == 1, f"Forventa 1 .ipynb i mappa, fann {len(nb_files)}"

nb_file = nb_files[0]
nb = nbformat.read(nb_file, as_version=4)

# Regex for ID-linja di:
#   # 12.3a [SETUP]
#   # 1.e [AGENT]
pattern = re.compile(
    r"^\s*#\s*([0-9]+(?:\.[0-9]+)?(?:\.[a-z])?)\s*\[([A-Z_]+)\]",
    re.IGNORECASE
)

found = []

for i, cell in enumerate(nb.cells):
    if cell.cell_type != "code":
        continue
    for line in cell.source.splitlines():
        m = pattern.match(line)
        if m:
            found.append({
                "cell_index": i,
                "id": m.group(1),
                "tag": m.group(2),
                "line": line.strip()
            })
            break  # berre første ID per celle

# Print pent
print(f"Notebook: {nb_file.name}")
print(f"Fant {len(found)} kodecelle-IDar:\n")

for f in found:
    print(f"Cell {f['cell_index']:>3}: {f['id']:<6} [{f['tag']}]")

